"""
Bubble Evaluation Tool (Results / Validation Post-Processing)
=============================================================

Purpose
-------
This notebook is used after the main Bubble Evaluation Tool has completed detection,
tracking, measurement, and CSV/video export. It loads the tracking outputs from one
run or a batch of runs and generates thesis-ready validation figures, summary tables,
and diagnostic plots.

The post-processing workflow supports two analysis cases:

1. No-Ground-Truth Experimental Runs
   These are real experimental videos where no known ground-truth bubble positions,
   masks, or velocities are available. The notebook summarizes measured bubble size,
   velocity, track behavior, trajectory quality, bubble density, and representative
   visual outputs.

2. Ground-Truth Validation Runs
   These are synthetic or labeled validation cases where known bubble locations,
   diameters, speeds, IDs, or mask videos are available. The notebook compares
   measured outputs to ground truth and computes accuracy, segmentation, and tracking
   metrics.

Major Capabilities
------------------
This notebook can generate:

- run-level summary tables
- per-track and per-bubble statistics
- bubble diameter distributions
- bubble velocity distributions
- velocity-versus-time plots
- smoothed representative velocity histories
- velocity-versus-diameter plots
- mean velocity versus mean diameter per track
- 2D bubble trajectory plots colored by time
- track length and track duration distributions
- frame-to-frame displacement statistics
- centroid uncertainty estimates
- diameter uncertainty estimates
- bubble density per frame and per image area
- qualitative image panels from raw, mask, and filtered videos
- Grace diagram placement using Reynolds and Eötvös/Bond numbers
- ground-truth measured-versus-true plots
- ground-truth error distributions
- relative diameter error plots
- IoU time histories and IoU distributions
- mask difference panels showing over- and under-prediction
- tracking metrics such as MOTA, MOTP, and IDF1
- batch-level summary CSV files
- batch overlay plots comparing multiple runs

Inputs
------
The notebook expects outputs generated by the main Bubble Evaluation Tool.

Required input for each run:
    *_tracked.csv
        Frame-resolved bubble tracking output. Contains bubble ID, frame number,
        centroid coordinates, diameter, and velocity information.

Optional inputs:
    *_averages.csv
        Run settings and averaged quantities exported by the Bubble Evaluation Tool.
        Used to recover scale, FPS, fluid settings, gas settings, and average
        dimensionless quantities.

    *_pred_mask.mp4
        Binary predicted semantic mask video. Used for segmentation visualization
        and IoU calculations when ground-truth masks are available.

    *_filtered.mp4
        Annotated tracking video. Used for qualitative viewing and figure panels.
        This file is NOT used for IoU calculations because it contains overlays.

    *_gt.csv
        Ground-truth CSV used for synthetic validation. Expected to contain known
        bubble position, diameter, velocity, and ID information.

    *_gt_mask.mp4
        Ground-truth binary semantic mask video. Used with *_pred_mask.mp4 for
        frame-level IoU validation.

    *_manual.csv
        Optional hand-labeled comparison file. If present, it can be used for
        manual-versus-automated comparison.

Expected Per-Run Folder Pattern
-------------------------------
A typical processed run folder may contain:

    sample_run_tracked.csv
    sample_run_averages.csv
    sample_run_filtered.mp4
    sample_run_pred_mask.mp4

A ground-truth validation run may also contain:

    sample_run_gt.csv
    sample_run_gt_mask.mp4

A manual-comparison run may also contain:

    sample_run_manual.csv

Outputs
-------
The notebook writes results to a selected output directory.

For each run, outputs are typically organized as:

    out_dir/
    ├── figs/
    │   ├── diameter and velocity histograms
    │   ├── trajectory plots
    │   ├── velocity plots
    │   ├── qualitative panels
    │   ├── Grace diagram plots
    │   ├── IoU plots, if ground truth is available
    │   └── measured-versus-true plots, if ground truth is available
    │
    └── tables/
        ├── run_summary.csv
        ├── per_bubble_stats.csv
        ├── per_track_mean_diam_vel.csv
        ├── track_stats.csv
        ├── bubble_density_summary.csv
        ├── bubble_density_per_frame.csv
        ├── centroid_uncertainty_summary.csv
        ├── diameter_uncertainty_summary.csv
        ├── matched_pairs.csv, if ground truth is available
        ├── tracking_metrics.csv, if ground truth is available
        ├── iou_per_frame.csv, if masks are available
        └── ks2_results.csv, if ground truth is available

For batch processing, the notebook may also generate:

    master_summary.csv
    batch_overlays/
    batch_overlays/accuracy/

Ground-Truth Validation Metrics
-------------------------------
When ground-truth data are available, the notebook can compute:

- matched measured-versus-true bubble diameters
- matched measured-versus-true bubble velocities
- diameter MAE, RMSE, bias, and standard deviation
- velocity MAE, RMSE, bias, and standard deviation
- relative diameter error
- segmentation IoU
- MOTA
- MOTP
- IDF1
- false positives
- false negatives
- ID switches
- Kolmogorov-Smirnov two-sample comparisons between true and measured distributions

No-Ground-Truth Experimental Metrics
------------------------------------
When ground truth is not available, the notebook computes descriptive and diagnostic
statistics, including:

- bubble diameter statistics
- bubble velocity statistics
- track length statistics
- frame-to-frame displacement statistics
- centroid repeatability/uncertainty estimates
- diameter repeatability/uncertainty estimates
- bubble density per frame
- bubble density per square inch when scale is available
- representative trajectory and velocity figures

Important Notes
---------------
- IoU is computed only from binary mask videos:
      *_gt_mask.mp4 and *_pred_mask.mp4

- IoU is not computed from *_filtered.mp4 because the filtered video contains
  colored overlays, labels, and visual annotations.

- ROI filtering removes non-imaging regions before selected validation operations.
  This includes the top ignored text region and the bottom timestamp bar.

- Scale and FPS are inferred from *_averages.csv when available.

- If scale is available, pixel-based measurements are converted to inches and
  pixel/s velocities are converted to in/s.

- If required optional files are missing, the notebook skips the unavailable
  validation step and continues producing the outputs that can still be generated.

- Some plotting functions use percentile clipping for visualization only so that
  extreme outliers do not dominate the figure axes. Summary statistics are generally
  computed from the full available data unless otherwise noted.

- The notebook is intended for thesis validation, figure generation, and diagnostic
  review. It is a post-processing tool, not the primary detection/tracking pipeline.

Software Context
----------------
Primary detection/tracking is performed in:

    Bubbly_Flow_Analysis.ipynb

This notebook should be run after the main tool has produced:

    *_tracked.csv
    *_averages.csv
    *_filtered.mp4
    *_pred_mask.mp4

Training and calibration notebooks are stored separately in:

    Software/Training/

"""

In [7]:
# ============================================================
# Bubble Evaluation Tool - Results / Validation Post-Processing
# ============================================================
#
# Purpose
# -------
# Reads the output files generated by the Bubble Tracking GUI and produces
# validation figures, summary tables, per-track statistics, uncertainty
# estimates, and thesis-ready plots.
#
# Expected Tracking Outputs
# -------------------------
# Required:
#   *_tracked.csv
#   *_averages.csv
#
# Optional:
#   annotated/filtered video outputs
#   predicted mask videos
#   ground-truth mask videos
#   hand-labeled or synthetic ground-truth CSVs
#
# Main Outputs
# ------------
#   tables/run_summary.csv
#   tables/track_stats.csv
#   tables/per_bubble_stats.csv
#   tables/bubble_density_summary.csv
#   figs/*.png
#   errors.log, if any plots fail
#
# Notes
# -----
# This script is designed to work with both older thesis outputs and the
# current cleaned tracking-code outputs. When available, exported time_s
# values are used directly instead of reconstructing time from frame/fps.
# ============================================================

import os, glob, traceback, random, re
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.collections import LineCollection

from scipy.optimize import linear_sum_assignment
from scipy.interpolate import UnivariateSpline
from scipy.stats import ks_2samp

import tkinter as tk
from tkinter import ttk, filedialog, messagebox

try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

BOTTOM_BAR_PX = 30  # timestamp bar height
# Plot layout constants
FIG_WIDE = (10.8, 5.6)
FIG_STANDARD = (9.8, 5.4)
FIG_TALL = (8.6, 6.3)

RIGHT_MARGIN = 0.76   # leaves room for outside stat boxes / legends

# ============================================================
# Helpers / style / IO
# ============================================================
# ---------------- Fluids / physics ----------------
# Note: Dimensionless numbers computed in English units with gc (G_C).
IQ_PROPS = {
    "Water": {"mu": 0.0000001412232, "sigma": 0.000416, "rho": 0.03605},   # mu: lbf*s/in^2, sigma: lbf/in, rho: lbm/in^3
    "SPT3" : {"mu": 0.000005076321, "sigma": 0.000406, "rho": 0.10838},
}
LIQ_PROPS = IQ_PROPS
GAS_RHO_FT3 = {"Air": 0.07487, "Nitrogen": 0.07210}  # lbm/ft^3

G_C       = 386.4   # (lbm*in)/(lbf*s^2)
G0_IN_S2  = 386.4   # in/s^2

def infer_liquid_gas(settings):
    """
    Pull LIQUID and GAS from the settings block in averages.csv.
    Returns (liq_name, gas_name) or (None, None).
    """
    if not settings:
        return None, None

    # Common direct keys
    liq = settings.get("LIQUID", None) or settings.get("Liquid", None) or settings.get("liquid", None)
    gas = settings.get("GAS", None)    or settings.get("Gas", None)    or settings.get("gas", None)

    # Fuzzy fallback
    if liq is None:
        for k, v in settings.items():
            if str(k).strip().upper() == "LIQUID":
                liq = v
                break
    if gas is None:
        for k, v in settings.items():
            if str(k).strip().upper() == "GAS":
                gas = v
                break

    liq = str(liq).strip() if liq not in (None, "", "None") else None
    gas = str(gas).strip() if gas not in (None, "", "None") else None
    return liq, gas

def gas_rho_in3(gas_name):
    """Convert gas density from lbm/ft^3 to lbm/in^3."""
    if gas_name is None:
        return np.nan
    if gas_name not in GAS_RHO_FT3:
        return np.nan
    return float(GAS_RHO_FT3[gas_name]) / 1728.0

def compute_re_bo_from_medians(D_in, V_in_s, liquid_name, gas_name):
    if not (np.isfinite(D_in) and np.isfinite(V_in_s) and D_in > 0 and V_in_s > 0):
        return np.nan, np.nan

    if liquid_name not in LIQ_PROPS:
        return np.nan, np.nan

    props = LIQ_PROPS[liquid_name]
    rho_l = float(props["rho"])     # lbm/in^3
    mu    = float(props["mu"])      # lbf*s/in^2
    sigma = float(props["sigma"])   # lbf/in

    rho_g = gas_rho_in3(gas_name)   # lbm/in^3

    Re = (rho_l * V_in_s * D_in) / (mu * G_C)

    if not np.isfinite(rho_g):
        rho_g = 0.0
    Bo = ((rho_l - rho_g) * G0_IN_S2 * (D_in ** 2)) / (sigma * G_C)

    return float(Re), float(Bo)

def _binarize_from_bgr_or_gray(frame, thresh=128):
    if frame is None:
        return None
    if frame.ndim == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame
    return (gray >= thresh)

def _apply_vertical_crop(mask_bool, crop_top=0, crop_bottom=0):
    if mask_bool is None:
        return None
    H = mask_bool.shape[0]
    y0 = int(max(0, crop_top))
    y1 = int(H - crop_bottom) if crop_bottom else H
    y1 = int(min(H, y1))
    if y1 <= y0:
        return mask_bool  
    return mask_bool[y0:y1, :]

def _mask_diff_rgb(gt_bool, pr_bool):
    if gt_bool is None or pr_bool is None:
        return None

    if pr_bool.shape != gt_bool.shape:
        pr_u8 = pr_bool.astype(np.uint8) * 255
        pr_u8 = cv2.resize(pr_u8, (gt_bool.shape[1], gt_bool.shape[0]), interpolation=cv2.INTER_NEAREST)
        pr_bool = (pr_u8 >= 128)

    gt = gt_bool.astype(bool)
    pr = pr_bool.astype(bool)

    over  = pr & (~gt)  
    under = gt & (~pr)   
    both  = gt & pr     

    rgb = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)

    # Start with GT white (includes overlap)
    rgb[gt] = (255, 255, 255)

    # Under-prediction: blue (override GT-white where GT-only)
    rgb[under] = (0, 0, 255)

    # Over-prediction: red
    rgb[over] = (255, 0, 0)

    # Overlap remains white (because gt set to white and we didn't override with blue/red)
    return rgb



def plot_velocity_vs_time_smoothed(df, out_png, units_vel,
                                   max_tracks=6,
                                   min_points=8,
                                   upward_frac_min=0.70,
                                   jump_max_in=0.12,
                                   jump_max_px=25.0,
                                   gap_max_s=0.03,
                                   gap_max_frames=2,
                                   clip_high_quantile=0.990,
                                   linewidth=2.2,
                                   alpha=0.9,
                                   median_window=11,
                                   mean_window=9,
                                   plot_every=3):

    set_thesis_style()

    if df is None or df.empty or (("vel_px_s" not in df.columns) and ("vel_in_s" not in df.columns)):
        fig, ax = plt.subplots(figsize=FIG_STANDARD)
        ax.text(0.5, 0.5, "No velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # ----- time column -----
    has_time = ("t_s" in df.columns) and df["t_s"].notna().any()
    tcol = "t_s" if has_time else ("time_s" if "time_s" in df.columns else "frame")
    xlabel = "Time (s)" if (tcol in ("t_s", "time_s")) else "Frame"

    # ----- velocity column -----
    vel_col = "vel_px_s" if units_vel == "px/s" else "vel_in_s"
    if vel_col not in df.columns:
        vel_col = "vel_in_s" if "vel_in_s" in df.columns else "vel_px_s"

    # ----- position columns for jump/gap detection -----
    xcol = "x_in" if "x_in" in df.columns else ("cx" if "cx" in df.columns else None)
    ycol = "y_in" if "y_in" in df.columns else ("cy" if "cy" in df.columns else None)
    can_jump = (xcol is not None) and (ycol is not None)

    d0 = df.copy()
    for c in ("id", tcol, vel_col, xcol, ycol):
        if c and c in d0.columns:
            d0[c] = pd.to_numeric(d0[c], errors="coerce")

    req = np.isfinite(d0["id"]) & np.isfinite(d0[tcol]) & np.isfinite(d0[vel_col])
    if can_jump:
        req &= np.isfinite(d0[xcol]) & np.isfinite(d0[ycol])

    d0 = d0[req].copy()

    if d0.empty:
        fig, ax = plt.subplots(figsize=FIG_STANDARD)
        ax.text(0.5, 0.5, "No valid numeric velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    jump_max = float(jump_max_in if (can_jump and xcol.endswith("_in")) else jump_max_px)
    gap_max = float(gap_max_s if has_time else gap_max_frames)

    def _segments_by_gaps(x, y, t):
        n = len(t)
        if n < 2:
            return [(0, n)]

        dx = np.diff(x)
        dy = np.diff(y)
        dt = np.diff(t)
        dr = np.hypot(dx, dy)

        bad = (dr > jump_max) | (dt > gap_max) | (dt <= 0)

        cuts = [0]
        for i, b in enumerate(bad):
            if b:
                cuts.append(i + 1)
        cuts.append(n)

        segs = []
        for a, b in zip(cuts[:-1], cuts[1:]):
            if (b - a) >= min_points:
                segs.append((a, b))

        return segs

    def _stability_score(g):
        g = g.sort_values(tcol)
        t = g[tcol].to_numpy(float)
        v = g[vel_col].to_numpy(float)

        if len(t) < min_points:
            return None

        v_med = np.nanmedian(v)
        v_mad = np.nanmedian(np.abs(v - v_med)) + 1e-9
        spike_frac = float(np.mean(np.abs(v - v_med) > 6.0 * v_mad))

        up_frac = 1.0
        tele_frac = 0.0
        gap_frac = 0.0
        longest_seg = len(t)

        if can_jump:
            x = g[xcol].to_numpy(float)
            y = g[ycol].to_numpy(float)

            dx = np.diff(x)
            dy = np.diff(y)
            dt = np.diff(t)
            dr = np.hypot(dx, dy)

            good = np.isfinite(dr) & np.isfinite(dt) & (dt > 0)
            if good.sum() < (min_points - 1):
                return None

            up_frac = float((dy[good] < 0).mean())
            if up_frac < upward_frac_min:
                return None

            tele_frac = float((dr[good] > jump_max).mean())
            gap_frac = float((dt[good] > gap_max).mean())

            segs = _segments_by_gaps(x, y, t)
            longest_seg = max((b - a) for a, b in segs) if segs else 0

        score = (
            3.0 * up_frac
            + 0.03 * longest_seg
            - 3.5 * spike_frac
            - 3.0 * tele_frac
            - 2.0 * gap_frac
        )
        return score

    # ----- choose representative tracks -----
    scored = []
    for tid, g in d0.groupby("id", sort=False):
        sc = _stability_score(g)
        if sc is not None:
            scored.append((int(float(tid)), float(sc)))

    if scored:
        scored.sort(key=lambda x: x[1], reverse=True)
        ids = [tid for tid, _ in scored[:max_tracks]]
    else:
        lens = d0.groupby("id").size().sort_values(ascending=False)
        ids = list(lens.head(max_tracks).index.astype(int))

    # ----- plot clipping threshold -----
    clip_thr = None
    if clip_high_quantile is not None:
        vv_all = d0[vel_col].to_numpy(float)
        vv_all = vv_all[np.isfinite(vv_all)]
        if vv_all.size:
            clip_thr = float(np.quantile(vv_all, clip_high_quantile))

    def _despike_and_smooth(tt, vv):
        tt = np.asarray(tt, float)
        vv = np.asarray(vv, float)

        m = np.isfinite(tt) & np.isfinite(vv)
        tt = tt[m]
        vv = vv[m]

        if len(vv) < 5:
            return tt, vv

        # Sort just in case
        order = np.argsort(tt)
        tt = tt[order]
        vv = vv[order]

        s = pd.Series(vv)

        # Step 1: hard local spike replacement using rolling median
        med = s.rolling(window=median_window, center=True, min_periods=1).median()
        resid = np.abs(s - med)
        mad = resid.rolling(window=median_window, center=True, min_periods=1).median()
        robust_sigma = 1.4826 * mad.replace(0, np.nan)

        spike_mask = resid > (2.5 * robust_sigma)
        vv_clean = vv.copy()
        vv_clean[spike_mask.fillna(False).to_numpy()] = med[spike_mask.fillna(False)].to_numpy()

        # Step 2: rolling median smooth
        vv_med = (
            pd.Series(vv_clean)
            .rolling(window=median_window, center=True, min_periods=1)
            .median()
            .to_numpy(float)
        )

        # Step 3: rolling mean smooth
        vv_smooth = (
            pd.Series(vv_med)
            .rolling(window=mean_window, center=True, min_periods=1)
            .mean()
            .to_numpy(float)
        )

        return tt, vv_smooth

    # ----- plot -----
    fig, ax = plt.subplots(figsize=FIG_STANDARD)

    for tid in ids:
        g = d0[d0["id"] == tid].sort_values(tcol)

        t = g[tcol].to_numpy(float)
        v = g[vel_col].to_numpy(float)

        if can_jump:
            x = g[xcol].to_numpy(float)
            y = g[ycol].to_numpy(float)
            segs = _segments_by_gaps(x, y, t)
        else:
            segs = [(0, len(t))]

        for a, b in segs:
            tt = t[a:b]
            vv = v[a:b]

            if len(tt) < min_points:
                continue

            if clip_thr is not None:
                vv = np.minimum(vv, clip_thr)

            tt_s, vv_s = _despike_and_smooth(tt, vv)

            if plot_every > 1 and len(tt_s) > plot_every:
                tt_s = tt_s[::plot_every]
                vv_s = vv_s[::plot_every]

            ax.plot(tt_s, vv_s, linewidth=linewidth, alpha=alpha)

    ax.grid(True)
    ax.set_title("Bubble Rise Velocity vs Time Smoothed")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"Rise velocity ({units_vel})")

    if clip_thr is not None:
        ax.text(
            0.99, 0.01,
            f"Plot clipped at {clip_high_quantile*100:.1f}th pct ({clip_thr:.2f} {units_vel})",
            ha="right",
            va="bottom",
            transform=ax.transAxes,
            fontsize=9
        )

    ax.text(
        0.99, 0.04,
        "Rolling median despike + rolling mean smoothing",
        ha="right",
        va="bottom",
        transform=ax.transAxes,
        fontsize=9
    )

    savefig(out_png)

def ks2_test(x, y, alpha=0.05):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]

    if x.size < 5 or y.size < 5:
        return np.nan, np.nan, np.nan, x.size, y.size  # D, p, reject?, N1, N2

    res = ks_2samp(x, y, alternative="two-sided", method="auto")
    D = float(res.statistic)
    p = float(res.pvalue)
    reject = float(p < alpha)
    return D, p, reject, x.size, y.size

def make_mask_diff_panel_from_videos(gt_mask_mp4, pred_mask_mp4, out_png,
                                    start_frame=0, n_frames=5,
                                    thresh=128, crop_top=0, crop_bottom=0):
    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read mask videos.")
    if not gt_mask_mp4 or not os.path.exists(gt_mask_mp4):
        raise FileNotFoundError(f"GT mask video not found: {gt_mask_mp4}")
    if not pred_mask_mp4 or not os.path.exists(pred_mask_mp4):
        raise FileNotFoundError(f"Pred mask video not found: {pred_mask_mp4}")

    set_thesis_style()

    cap_gt = cv2.VideoCapture(gt_mask_mp4)
    cap_pr = cv2.VideoCapture(pred_mask_mp4)

    frs = list(range(int(start_frame), int(start_frame) + int(n_frames)))

    fig, axes = plt.subplots(1, len(frs), figsize=(4.0 * len(frs), 4.0))
    if len(frs) == 1:
        axes = np.array([axes])

    for j, fr in enumerate(frs):
        gt_bgr = _read_frame_at(cap_gt, fr)
        pr_bgr = _read_frame_at(cap_pr, fr)

        gt_bool = _binarize_from_bgr_or_gray(gt_bgr, thresh=thresh)
        pr_bool = _binarize_from_bgr_or_gray(pr_bgr, thresh=thresh)

        gt_bool = _apply_vertical_crop(gt_bool, crop_top=crop_top, crop_bottom=crop_bottom)
        pr_bool = _apply_vertical_crop(pr_bool, crop_top=crop_top, crop_bottom=crop_bottom)

        rgb = _mask_diff_rgb(gt_bool, pr_bool)

        if rgb is not None:
            axes[j].imshow(rgb)  # rgb already
        axes[j].set_title(f"Frame {fr}")
        axes[j].axis("off")

    # One legend block (text) to avoid adding a matplotlib legend for image pixels
    fig.suptitle("Mask Difference Overlay (GT vs Pred)", y=1.02, fontsize=18)
    fig.text(
        0.5, 0.02,
        "White: GT (incl overlap)    Red: Over-prediction (FP)    Blue: Under-prediction (FN)",
        ha="center", va="bottom",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", alpha=0.95)
    )

    cap_gt.release()
    cap_pr.release()

    savefig(out_png)

def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

def infer_ignore_top_px(settings):
    if not settings:
        return 0
    # common variants you might have
    for k in ["IGNORE_TOP_PX", "IGNORE_TOP", "IGNORE_T", "IGNORE_TEXT_PX", "IGNORE_TEXT_TOP_PX"]:
        if k in settings:
            v = _safe_float(settings[k])
            if v is not None:
                return int(round(v))
    # fallback: try fuzzy match
    for k, v in settings.items():
        kk = str(k).upper()
        if "IGNORE" in kk and "TOP" in kk:
            vv = _safe_float(v)
            if vv is not None:
                return int(round(vv))
    return 0

def apply_roi_filter(df, cy_col, ignore_top_px=0, bottom_bar_px=0, frame_h=None):
    """
    Keep only rows whose centroid y is inside the detectable region:
      y >= ignore_top_px  AND  y < (frame_h - bottom_bar_px)
    If frame_h is None, only applies the top filter.
    """
    if df is None or df.empty:
        return df
    df = df.copy()
    y = pd.to_numeric(df[cy_col], errors="coerce")
    keep = (y >= float(ignore_top_px))
    if frame_h is not None:
        keep &= (y < float(frame_h - bottom_bar_px))
    return df[keep].copy()

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def set_thesis_style():
    plt.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "legend.fontsize": 11,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
        "grid.alpha": 0.25,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

def clean_plot_title(title):
    """
    Removes parenthetical text from plot titles.
    Example:
      'Bubble Diameter Distribution (Measured)'
      -> 'Bubble Diameter Distribution'
    """
    if title is None:
        return ""
    return re.sub(r"\s*\([^)]*\)", "", str(title)).strip()

def infer_frame_wh_from_video(video_path):
    """
    Returns (frame_w, frame_h) from the first readable frame of a video.
    """
    if (not _HAS_CV2) or (video_path is None) or (not os.path.exists(video_path)):
        return (None, None)
    cap = cv2.VideoCapture(video_path)
    ok, fr = cap.read()
    cap.release()
    if (not ok) or (fr is None):
        return (None, None)
    h, w = fr.shape[:2]
    return (int(w), int(h))


def compute_bubble_density_per_run(df_det,
                                  frame_col="frame",
                                  cy_col="cy",
                                  ignore_top_px=0,
                                  bottom_bar_px=0,
                                  frame_h=None,
                                  frame_w=None,scale_in_per_px=0,):
  
    if df_det is None or df_det.empty:
        return (
            {
                "roi_area_in2": np.nan,
                "mean_bubbles_per_frame": np.nan,
                "median_bubbles_per_frame": np.nan,
                "mean_density_bubbles_in2": np.nan,
                "median_density_bubbles_in2": np.nan,
                "N_frames_used": 0
            },
            pd.DataFrame())

    d = df_det.copy()
    d[frame_col] = pd.to_numeric(d[frame_col], errors="coerce").astype("Int64")
    d[cy_col] = pd.to_numeric(d[cy_col], errors="coerce")

    # Apply ROI filter (top ignore + bottom timestamp bar)
    d = apply_roi_filter(d, cy_col=cy_col, ignore_top_px=ignore_top_px,
                         bottom_bar_px=bottom_bar_px, frame_h=frame_h)

    d = d.dropna(subset=[frame_col]).copy()
    if d.empty:
        return (
            {
                "roi_area_in2": np.nan,
                "mean_bubbles_per_frame": np.nan,
                "median_bubbles_per_frame": np.nan,
                "mean_density_bubbles_in2": np.nan,
                "median_density_bubbles_in2": np.nan,
                "N_frames_used": 0
            },
            pd.DataFrame()
        )

    # Count detections per frame (this is "number of bubbles in observation field" per frame)
    counts = d.groupby(frame_col).size().sort_index()
    df_pf = pd.DataFrame({
        "frame": counts.index.astype(int),
        "bubble_count": counts.values.astype(int),
    })

    # Need scale + frame size to compute in^2
    roi_area_in2 = np.nan
    if (scale_in_per_px is not None) and np.isfinite(scale_in_per_px) and (scale_in_per_px > 0) \
       and (frame_w is not None) and (frame_h is not None):

        roi_h_px = float(frame_h) - float(ignore_top_px) - float(bottom_bar_px)
        roi_w_px = float(frame_w)

        if roi_h_px > 0 and roi_w_px > 0:
            roi_w_in = roi_w_px * float(scale_in_per_px)
            roi_h_in = roi_h_px * float(scale_in_per_px)
            roi_area_in2 = roi_w_in * roi_h_in

    if np.isfinite(roi_area_in2) and roi_area_in2 > 0:
        df_pf["density_bubbles_in2"] = df_pf["bubble_count"] / roi_area_in2
        mean_density = float(df_pf["density_bubbles_in2"].mean())
        median_density = float(df_pf["density_bubbles_in2"].median())
    else:
        df_pf["density_bubbles_in2"] = np.nan
        mean_density = np.nan
        median_density = np.nan

    summary = {
        "roi_area_in2": float(roi_area_in2) if np.isfinite(roi_area_in2) else np.nan,
        "mean_bubbles_per_frame": float(df_pf["bubble_count"].mean()),
        "median_bubbles_per_frame": float(df_pf["bubble_count"].median()),
        "mean_density_bubbles_in2": mean_density,
        "median_density_bubbles_in2": median_density,
        "N_frames_used": int(df_pf.shape[0]),
        "ignore_top_px": int(ignore_top_px),
        "bottom_bar_px": int(bottom_bar_px),
        "frame_w_px": int(frame_w) if frame_w is not None else "",
        "frame_h_px": int(frame_h) if frame_h is not None else "",
        "note": "Density computed from detections-per-frame in ROI divided by ROI area (in^2)."
    }
    return summary, df_pf


def log_error(out_dir, msg):
    try:
        with open(os.path.join(out_dir, "errors.log"), "a", encoding="utf-8") as f:
            f.write(msg.rstrip() + "\n")
    except Exception:
        pass

def savefig(path):
    fig = plt.gcf()

    # Only reserve right-side space when an axes has outside text/legend.
    # Wider figures prevent the plot itself from becoming tall and skinny.
    try:
        fig.subplots_adjust(right=RIGHT_MARGIN)
    except Exception:
        pass

    try:
        fig.tight_layout()
    except Exception:
        pass

    ensure_dir(os.path.dirname(path))
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def safe_plot(out_dir, name, out_png, fn):
    """
    Always writes a PNG:
      - success -> saves out_png
      - failure -> placeholder PNG + logs errors.log
    """
    try:
        fn()
    except Exception as e:
        log_error(out_dir, f"[PLOT FAIL] {name}: {type(e).__name__}: {e}")
        log_error(out_dir, traceback.format_exc())
        try:
            set_thesis_style()
            fig, ax = plt.subplots(figsize=(7.8, 5.4))
            ax.grid(True)
            ax.set_title(name)
            ax.text(
                0.5, 0.5,
                f"Plot failed:\n{type(e).__name__}\n{e}",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black")
            )
            savefig(out_png)
        except Exception:
            pass

def fname_units(units: str) -> str:
    """Windows-safe units string for filenames (no slashes)."""
    if units is None:
        return "unitless"
    u = str(units).replace("/", "_per_").replace("\\", "_").replace(" ", "")
    u = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", u)
    return u

def clip_by_percentile(x, lo=1, hi=99):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x, None, None
    a, b = np.percentile(x, [lo, hi])
    return x[(x >= a) & (x <= b)], a, b

def clip_for_hist(x, lo=1, hi=99):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n_total = int(x.size)
    if n_total == 0:
        return x, np.nan, np.nan, 0, 0
    a, b = np.percentile(x, [lo, hi])
    keep = (x >= a) & (x <= b)
    x2 = x[keep]
    return x2, float(a), float(b), n_total, int(x2.size)

def compute_per_bubble_stats(df, id_col="id", cols=None, min_points=2):
    """
    Per-bubble (per-track) stats over time.
    Returns a dataframe with N/mean/std/median/min/max for each requested column.
    """
    if cols is None:
        cols = []

    if df is None or df.empty or (id_col not in df.columns):
        return pd.DataFrame()

    d = df.copy()
    d[id_col] = d[id_col].astype(str)

    # ensure numeric columns
    for c in cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")

    # keep only ids that have at least 1 valid value in any col
    keep_any = np.zeros(len(d), dtype=bool)
    for c in cols:
        if c in d.columns:
            keep_any |= np.isfinite(d[c].to_numpy(float))
    d = d[keep_any].copy()
    if d.empty:
        return pd.DataFrame()

    rows = []
    for tid, g in d.groupby(id_col, sort=False):
        row = {id_col: tid}

        for c in cols:
            if c not in g.columns:
                continue
            x = pd.to_numeric(g[c], errors="coerce").to_numpy(float)
            x = x[np.isfinite(x)]
            N = int(x.size)

            row[f"N_{c}"] = N
            if N == 0:
                row[f"mean_{c}"] = np.nan
                row[f"std_{c}"]  = np.nan
                row[f"median_{c}"] = np.nan
                row[f"min_{c}"] = np.nan
                row[f"max_{c}"] = np.nan
            else:
                row[f"mean_{c}"] = float(np.mean(x))
                row[f"median_{c}"] = float(np.median(x))
                row[f"min_{c}"] = float(np.min(x))
                row[f"max_{c}"] = float(np.max(x))
                row[f"std_{c}"]  = float(np.std(x, ddof=1)) if N >= min_points else 0.0

        rows.append(row)

    out = pd.DataFrame(rows)

    if cols:
        c0 = cols[0]
        n0 = f"N_{c0}"
        if n0 in out.columns:
            out = out[out[n0].fillna(0).astype(int) >= min_points].copy()

    return out


def compute_delta_r_stats(df_tracks, require_consecutive=True, max_step_px=None):
    """
    Compute frame-to-frame displacement Δr for tracked centroids.

    df_tracks must have columns: id, frame, cx, cy (numeric-able).
    - require_consecutive: only use steps where frame_{i+1} = frame_i + 1
    - max_step_px: optional jump filter to remove ID-swap leaps (e.g., 100 px)

    Returns:
      (delta_r_median, delta_r_mean, N_steps, steps_array)
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, np.nan, 0, np.array([])

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d["cx"] = pd.to_numeric(d["cx"], errors="coerce")
    d["cy"] = pd.to_numeric(d["cy"], errors="coerce")
    d = d.dropna(subset=["id", "frame", "cx", "cy"]).copy()

    steps = []

    for tid, g in d.groupby("id"):
        g = g.sort_values("frame")
        fr = g["frame"].to_numpy(float)
        x  = g["cx"].to_numpy(float)
        y  = g["cy"].to_numpy(float)

        if len(fr) < 2:
            continue

        dfr = np.diff(fr)
        dx  = np.diff(x)
        dy  = np.diff(y)
        dr  = np.hypot(dx, dy)

        # keep only consecutive frame steps
        if require_consecutive:
            keep = (dfr == 1)
            dr = dr[keep]

        if max_step_px is not None and dr.size:
            dr = dr[dr <= float(max_step_px)]

        if dr.size:
            steps.append(dr)

    if not steps:
        return np.nan, np.nan, 0, np.array([])

    steps = np.concatenate(steps).astype(float)
    steps = steps[np.isfinite(steps)]
    if steps.size == 0:
        return np.nan, np.nan, 0, np.array([])

    return float(np.median(steps)), float(np.mean(steps)), int(steps.size), steps
# ============================================================
# Stats
# ============================================================
def _clean(x):
    x = np.asarray(x, float)
    return x[np.isfinite(x)]

def stats_dict(x):
    x = _clean(x)
    if x.size == 0:
        return {"N": 0, "mean": np.nan, "median": np.nan, "std": np.nan, "min": np.nan, "max": np.nan}
    return {
        "N": int(x.size),
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "std": float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

def mae(x):
    x = _clean(x)
    return float(np.mean(np.abs(x))) if x.size else np.nan

def rmse(x):
    x = _clean(x)
    return float(np.sqrt(np.mean(x**2))) if x.size else np.nan

def bias(x):
    x = _clean(x)
    return float(np.mean(x)) if x.size else np.nan

def add_statbox(ax, lines, loc="outside right"):
    """
    Adds a statistics box.

    Default behavior places the box outside the plotting area on the right
    so it does not cover the data.
    """
    text = "\n".join(lines)
    bbox = dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=1.0)

    # Outside-right default
    if loc in ["outside right", "right", "outside"]:
        ax.text(
            1.05, 0.98, text,
            transform=ax.transAxes,
            va="top", ha="left",
            bbox=bbox,
            fontsize=10,
            clip_on=False
        )
        return

    # Outside-bottom option
    if loc in ["outside bottom", "bottom"]:
        ax.text(
            0.5, -0.22, text,
            transform=ax.transAxes,
            va="top", ha="center",
            bbox=bbox,
            fontsize=10,
            clip_on=False
        )
        return

    # Fallback inside locations if you intentionally want them
    locs = {
        "upper left":  (0.02, 0.98, "top",    "left"),
        "upper right": (0.98, 0.98, "top",    "right"),
        "lower left":  (0.02, 0.02, "bottom", "left"),
        "lower right": (0.98, 0.02, "bottom", "right"),
    }

    x, y, va, ha = locs.get(loc, locs["upper left"])

    ax.text(
        x, y, text,
        transform=ax.transAxes,
        va=va, ha=ha,
        bbox=bbox,
        fontsize=10
    )

def robust_limits(x, p_lo=1, p_hi=99, pad_frac=0.08):
    x = _clean(x)
    if x.size == 0:
        return (-1, 1)
    a = np.percentile(x, p_lo)
    b = np.percentile(x, p_hi)
    if a == b:
        pad = 1.0 if a == 0 else abs(a) * 0.2
        return (a - pad, b + pad)
    pad = (b - a) * pad_frac
    return (a - pad, b + pad)

def diameter_uncertainty_from_fit(df_tracks, fps=None,
                                  fit_mode="constant", poly_deg=2,
                                  spline_s=None,
                                  min_points=20,
                                  max_step_px=None,
                                  units_col="diam_px"):
    """
    Per-track diameter precision uncertainty from residuals around a smooth fit:
        u_D = sqrt( 1/(N-1) * sum_i (D_i - Dfit_i)^2 )

    df_tracks must have columns: id, frame, and units_col (diam_px or diam_in).
    Returns:
        u_D_global, per_track_df
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, pd.DataFrame()

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d[units_col] = pd.to_numeric(d[units_col], errors="coerce")

    # time base
    if "t_s" in d.columns and d["t_s"].notna().any():
        d["t"] = pd.to_numeric(d["t_s"], errors="coerce")
    else:
        if fps and fps > 0:
            d["t"] = d["frame"] / float(fps)
        else:
            d["t"] = d["frame"]

    d = d.dropna(subset=["id", "t", units_col]).copy()

    rows = []
    all_r2 = []

    for tid, g in d.groupby("id"):
        g = g.sort_values("t")
        t = g["t"].to_numpy(float)
        D = g[units_col].to_numpy(float)

        m = np.isfinite(t) & np.isfinite(D)
        t, D = t[m], D[m]
        if t.size < min_points:
            continue

        if max_step_px is not None and {"cx","cy"}.issubset(g.columns) and t.size >= 3:
            x = pd.to_numeric(g["cx"], errors="coerce").to_numpy(float)[m]
            y = pd.to_numeric(g["cy"], errors="coerce").to_numpy(float)[m]
            step = np.hypot(np.diff(x), np.diff(y))
            ok = np.ones_like(D, dtype=bool)
            ok[1:] = step <= float(max_step_px)
            t, D = t[ok], D[ok]
            if t.size < min_points:
                continue

        # ---- Fit D(t) ----
        if fit_mode == "constant":
            Dfit = np.full_like(D, np.mean(D))
        elif fit_mode == "linear":
            p = np.polyfit(t, D, 1)
            Dfit = np.polyval(p, t)
        elif fit_mode == "poly":
            deg = int(max(1, min(poly_deg, 5)))
            p = np.polyfit(t, D, deg)
            Dfit = np.polyval(p, t)
        elif fit_mode == "spline":
            s = float(spline_s) if spline_s is not None else (t.size * (0.5 ** 2))
            sp = UnivariateSpline(t, D, s=s)
            Dfit = sp(t)
        else:
            raise ValueError(f"Unknown fit_mode: {fit_mode}")

        r = D - Dfit
        r2 = r**2
        N = r2.size
        uD = np.sqrt(np.sum(r2) / (N - 1)) if N > 1 else np.nan

        rows.append({
            "id": tid,
            "N": int(N),
            f"u_D_{units_col}": float(uD) if np.isfinite(uD) else np.nan,
            f"D_mean_{units_col}": float(np.mean(D)) if np.isfinite(np.mean(D)) else np.nan,
            f"D_std_{units_col}": float(np.std(D, ddof=1)) if N > 1 else np.nan,
        })
        all_r2.extend(list(r2))

    per_track = pd.DataFrame(rows)

    all_r2 = np.asarray(all_r2, float)
    all_r2 = all_r2[np.isfinite(all_r2)]
    uD_global = float(np.sqrt(np.sum(all_r2) / (all_r2.size - 1))) if all_r2.size > 1 else np.nan

    return uD_global, per_track


def run_diameter_uncertainty(df_tracks, tabs_dir, out_dir, fps, scale_in_per_px):
    """
    Computes segmentation diameter precision uncertainty (no GT).
    Saves:
      - diameter_uncertainty_per_track.csv
      - diameter_uncertainty_summary.csv
    Returns: (uD_px, uD_in)
    """
    try:
        if "id" not in df_tracks.columns or "frame" not in df_tracks.columns or "diam_px" not in df_tracks.columns:
            raise ValueError("Diameter uncertainty needs columns: id, frame, diam_px")

        uD_px, per_track_px = diameter_uncertainty_from_fit(
            df_tracks[["id","frame","diam_px","cx","cy","t_s"]].copy() if "t_s" in df_tracks.columns else
            df_tracks[["id","frame","diam_px","cx","cy"]].copy(),
            fps=fps,
            fit_mode="constant",   # bubble diameter should be ~constant over a short track
            min_points=20,
            max_step_px=25
        )
        per_track_px.to_csv(os.path.join(tabs_dir, "diameter_uncertainty_per_track_px.csv"), index=False)

        uD_in = np.nan
        if scale_in_per_px:
            df2 = df_tracks.copy()
            df2["diam_in"] = pd.to_numeric(df2["diam_px"], errors="coerce") * float(scale_in_per_px)

            uD_in, per_track_in = diameter_uncertainty_from_fit(
                df2[["id","frame","diam_in","cx","cy","t_s"]].copy() if "t_s" in df2.columns else
                df2[["id","frame","diam_in","cx","cy"]].copy(),
                fps=fps,
                fit_mode="constant",
                min_points=20,
                max_step_px=25,
                units_col="diam_in"
            )
            per_track_in.to_csv(os.path.join(tabs_dir, "diameter_uncertainty_per_track_in.csv"), index=False)

        pd.DataFrame([{
            "diameter_u_D_px": uD_px,
            "diameter_u_D_in": uD_in if scale_in_per_px else "",
            "fit_mode": "constant",
            "min_points": 20,
            "max_step_px": 25,
            "note": "Precision (repeatability) of diameter along each track; not GT accuracy."
        }]).to_csv(os.path.join(tabs_dir, "diameter_uncertainty_summary.csv"), index=False)

        return uD_px, (uD_in if scale_in_per_px else np.nan)

    except Exception as e:
        log_error(out_dir, f"[DIAM UNC FAIL] {type(e).__name__}: {e}")
        return np.nan, np.nan

def centroid_uncertainty_from_fit(df_tracks, fps=None,
                                  fit_mode="poly", poly_deg=3,
                                  spline_s=None,
                                  min_points=20,
                                  max_step_px=None):
    """
    Implements Eq. 3.17:
        u_xy = sqrt( 1/(N-1) * sum_i [ (x_i-xfit_i)^2 + (y_i-yfit_i)^2 ] )

    df_tracks must have columns: id, frame, cx, cy (and optionally t_s).
    - fit_mode:
        "constant" : xfit=mean(x), yfit=mean(y)  (quiescent)
        "linear"   : degree-1 poly fit           (slow drift)
        "poly"     : polynomial fit deg=poly_deg (accelerating)
        "spline"   : UnivariateSpline fit        (accelerating/curvy)
    - spline_s: smoothing factor for spline; if None, auto-ish (see below)
    - max_step_px: optional jump filter; if provided, drops segments with step > max_step_px
    Returns:
        overall_u_xy_px, per_track_table_df
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, pd.DataFrame()

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d["cx"] = pd.to_numeric(d["cx"], errors="coerce")
    d["cy"] = pd.to_numeric(d["cy"], errors="coerce")

    # time base
    if "t_s" in d.columns and d["t_s"].notna().any():
        d["t"] = pd.to_numeric(d["t_s"], errors="coerce")
    else:
        if fps and fps > 0:
            d["t"] = d["frame"] / float(fps)
        else:
            d["t"] = d["frame"]  

    d = d.dropna(subset=["id", "t", "cx", "cy"]).copy()

    rows = []
    all_r2 = []  # store r^2 across all selected samples (for global u_xy)

    for tid, g in d.groupby("id"):
        g = g.sort_values("t")
        t = g["t"].to_numpy(float)
        x = g["cx"].to_numpy(float)
        y = g["cy"].to_numpy(float)

        m = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
        t, x, y = t[m], x[m], y[m]
        if t.size < min_points:
            continue

        
        if max_step_px is not None and t.size >= 3:
            dx = np.diff(x)
            dy = np.diff(y)
            step = np.hypot(dx, dy)
            ok = np.ones_like(x, dtype=bool)
            ok[1:] = step <= float(max_step_px)
            t, x, y = t[ok], x[ok], y[ok]
            if t.size < min_points:
                continue

        if fit_mode == "constant":
            xfit = np.full_like(x, np.mean(x))
            yfit = np.full_like(y, np.mean(y))

        elif fit_mode == "linear":
            px = np.polyfit(t, x, 1)
            py = np.polyfit(t, y, 1)
            xfit = np.polyval(px, t)
            yfit = np.polyval(py, t)

        elif fit_mode == "poly":
            deg = int(poly_deg)
            deg = max(1, min(deg, 5))  
            px = np.polyfit(t, x, deg)
            py = np.polyfit(t, y, deg)
            xfit = np.polyval(px, t)
            yfit = np.polyval(py, t)

        elif fit_mode == "spline":
            
            if spline_s is None:
                s = t.size * (0.5 ** 2)
            else:
                s = float(spline_s)

            sx = UnivariateSpline(t, x, s=s)
            sy = UnivariateSpline(t, y, s=s)
            xfit = sx(t)
            yfit = sy(t)

        else:
            raise ValueError(f"Unknown fit_mode: {fit_mode}")

        # residual distance per sample
        rx = x - xfit
        ry = y - yfit
        r2 = rx**2 + ry**2

        # Eq 3.17 for this track
        N = r2.size
        u_xy = np.sqrt(np.sum(r2) / (N - 1)) if N > 1 else np.nan

        rows.append({
            "id": tid,
            "N": int(N),
            "u_xy_px": float(u_xy) if np.isfinite(u_xy) else np.nan,
            "u_x_px": float(np.std(rx, ddof=1)) if N > 1 else np.nan,
            "u_y_px": float(np.std(ry, ddof=1)) if N > 1 else np.nan,
        })
        all_r2.extend(list(r2))

    per_track = pd.DataFrame(rows)
    all_r2 = np.asarray(all_r2, float)
    all_r2 = all_r2[np.isfinite(all_r2)]

    # Global u_xy across all included samples (matches Eq. 3.17 idea at dataset level)
    # Use (N_total - 1) in denominator
    if all_r2.size > 1:
        u_xy_global = float(np.sqrt(np.sum(all_r2) / (all_r2.size - 1)))
    else:
        u_xy_global = np.nan

    return u_xy_global, per_track

def run_centroid_uncertainty(df_tracks, tabs_dir, out_dir, fps, scale_in_per_px):
    """
    Computes and saves centroid uncertainty (Eq. 3.17) from tracked trajectories.
    Saves:
      - centroid_uncertainty_per_track.csv
      - centroid_uncertainty_summary.csv
    Returns: (u_xy_px, u_xy_in)
    """
    try:
        need = ["id", "frame", "cx", "cy"]
        for c in need:
            if c not in df_tracks.columns:
                raise ValueError(f"Centroid uncertainty needs column '{c}' but it was missing.")

        cols = ["id", "frame", "cx", "cy"]
        if "t_s" in df_tracks.columns:
            cols.append("t_s")

        u_xy_px, per_track = centroid_uncertainty_from_fit(
            df_tracks[cols].copy(),
            fps=fps,
            fit_mode="poly",
            poly_deg=3,
            min_points=20,
            max_step_px=25
        )

        per_track.to_csv(os.path.join(tabs_dir, "centroid_uncertainty_per_track.csv"), index=False)
        pd.DataFrame([{
            "centroid_u_xy_px": u_xy_px,
            "fit_mode": "poly3",
            "min_points": 20,
            "max_step_px": 25
        }]).to_csv(os.path.join(tabs_dir, "centroid_uncertainty_summary.csv"), index=False)

        u_xy_in = (u_xy_px * scale_in_per_px) if (scale_in_per_px and np.isfinite(u_xy_px)) else np.nan
        return u_xy_px, u_xy_in

    except Exception as e:
        log_error(out_dir, f"[CENTROID UNC FAIL] {type(e).__name__}: {e}")
        return np.nan, np.nan

def drop_injection_zone_by_y(df, cy_col="cy", frac_from_bottom=0.20, frame_h=None):
    """
    Drop samples whose centroid is within the bottom frac_from_bottom of the image.

    Example: frac_from_bottom=0.15 means "remove bottom 15% of the frame".
    Requires frame_h in pixels.
    """
    if df is None or df.empty:
        return df
    if frame_h is None:
        # can't define "bottom X%" without knowing frame height
        return df

    d = df.copy()
    y = pd.to_numeric(d[cy_col], errors="coerce")

    # keep everything ABOVE the injection zone
    # injection zone starts at y_cut and goes to the bottom of the frame
    y_cut = float(frame_h) * (1.0 - float(frac_from_bottom))

    keep = np.isfinite(y) & (y < y_cut)
    return d[keep].copy()

# ============================================================
# Averages/settings reader (header line 25)
# ============================================================
def read_averages_with_settings(path, header_line_1_indexed=None):
    """
    Reads the *_averages.csv file written by the tracking GUI.

    Format:
      - settings rows at the top
      - blank line
      - summary table beginning with header row, usually:
        id, avg_vel_x_px_s, avg_vel_y_px_s, ...
    """
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.read().splitlines()

    # --- Parse settings until the first blank line ---
    settings = {}
    header_idx = None

    for i, ln in enumerate(lines):
        s = ln.strip()

        # Blank line usually separates settings from summary table
        if not s:
            continue

        first = s.split(",", 1)[0].strip()

        # Detect actual summary header
        if first == "id" and ("avg_" in s or "diam" in s or "vel" in s):
            header_idx = i
            break

        # Parse setting rows before the table
        if "," in s:
            k, v = s.split(",", 1)
            k = k.strip()
            v = v.strip()
            if k and not k.startswith("#"):
                settings[k] = v
        elif ":" in s:
            k, v = s.split(":", 1)
            k = k.strip()
            v = v.strip()
            if k and not k.startswith("#"):
                settings[k] = v

    if header_idx is None:
        # Fallback for older files
        if header_line_1_indexed is not None:
            header_idx = int(header_line_1_indexed) - 1
        else:
            raise ValueError(f"Could not find averages table header in: {path}")

    df_avg = pd.read_csv(
        path,
        header=header_idx,
        engine="python",
        sep=",",
        on_bad_lines="skip"
    )

    return settings, df_avg

def infer_scale_in_per_px(settings=None, df_avg=None):
    if settings:
        for k in ["SCALE_IN_PER_PX", "scale_in_per_px"]:
            if k in settings:
                val = _safe_float(settings[k])
                if val and val > 0:
                    return val
        pairs = [("SCALE_IN_INPUT", "SCALE_PX_INPUT"), ("SCALE_IN", "SCALE_PX")]
        for a, b in pairs:
            if a in settings and b in settings:
                A = _safe_float(settings[a])
                B = _safe_float(settings[b])
                if A and B and B > 0:
                    return A / B

    if df_avg is not None and ("diam_in" in df_avg.columns) and ("diam_px" in df_avg.columns):
        r = pd.to_numeric(df_avg["diam_in"], errors="coerce") / pd.to_numeric(df_avg["diam_px"], errors="coerce")
        r = r[np.isfinite(r) & (r > 0)]
        if len(r):
            return float(np.median(r))
    return None

def infer_fps(settings=None):
    if not settings:
        return None

    # Prefer still FPS, but allow fallback to captured video FPS
    for k in ["FPS_STILL", "CAP_FPS_VIDEO", "FPS"]:
        v = settings.get(k, None)
        try:
            v = float(v) if v not in (None, "", "None", "nan") else None
        except Exception:
            v = None

        if v is not None and np.isfinite(v) and v > 0:
            return v

    return None


# ============================================================
# Flexible column mapping
# ============================================================
def pick_col(df, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def map_gt(df_gt):
    m = {}
    m["frame"] = pick_col(df_gt, ["frame", "Frame"])
    m["cx"]    = pick_col(df_gt, ["gt_cx_px", "gt_cx", "cx", "x"])
    m["cy"]    = pick_col(df_gt, ["gt_cy_px", "gt_cy", "cy", "y"])
    m["diam"]  = pick_col(df_gt, ["gt_diam_equiv_px", "gt_diam_eq", "gt_diam_px", "diam_px", "diam"])
    m["vx"]    = pick_col(df_gt, ["gt_vx_px_s", "gt_vx", "vx"])
    m["vy"]    = pick_col(df_gt, ["gt_vy_px_s", "gt_vy", "vy"])
    m["spd"]   = pick_col(df_gt, ["gt_speed_px_s", "gt_speed", "speed_px_s", "speed"])
    req = [m["frame"], m["cx"], m["cy"], m["diam"]]
    if any(v is None for v in req):
        raise ValueError(f"GT missing required columns. Found: {list(df_gt.columns)}")
    return m

def map_tracked(df_m):
    m = {}
    m["frame"] = pick_col(df_m, ["frame", "Frame"])
    m["id"]    = pick_col(df_m, ["id", "ID", "track_id"])
    m["cx"]    = pick_col(df_m, ["cx", "x", "centroid_x"])
    m["cy"]    = pick_col(df_m, ["cy", "y", "centroid_y"])
    m["diam"]  = pick_col(df_m, ["diam_px", "diam", "diameter_px"])
    m["vx"]    = pick_col(df_m, ["vel_x_px_s", "vx_px_s", "vx"])
    m["vy"]    = pick_col(df_m, ["vel_y_px_s", "vy_px_s", "vy"])
    m["spd"]   = pick_col(df_m, ["vel_px_s", "speed_px_s", "vel", "speed"])
    req = [m["frame"], m["id"], m["cx"], m["cy"], m["diam"]]
    if any(v is None for v in req):
        raise ValueError(f"TRACKED missing required columns. Found: {list(df_m.columns)}")
    return m


# ============================================================
# Matching (Hungarian per frame)
# ============================================================
def match_frame(gtf, mf, max_dist_px):
    gx = gtf["gt_cx_px"].to_numpy(float)
    gy = gtf["gt_cy_px"].to_numpy(float)
    mx = mf["cx"].to_numpy(float)
    my = mf["cy"].to_numpy(float)

    if gx.size == 0 or mx.size == 0:
        return []

    D = np.sqrt((gx[:, None] - mx[None, :])**2 + (gy[:, None] - my[None, :])**2)
    BIG = 1e9
    D[~np.isfinite(D)] = BIG
    D[D > max_dist_px] = BIG

    r, c = linear_sum_assignment(D)
    pairs = []
    for i, j in zip(r, c):
        if D[i, j] < BIG:
            pairs.append((int(i), int(j), float(D[i, j])))
    return pairs

# ============================================================
# Grace diagram overlay (hard-coded image + axis mapping)
#   - Single-run: plots ONE point (avg_Eo, avg_Re)
#   - Batch: plots ONE point per run for ALL runs on one map
# ============================================================

# Grace diagram image path (script-safe + notebook-safe)
try:
    _BASE_DIR = os.path.dirname(__file__)
except NameError:
    _BASE_DIR = os.getcwd()

GRACE_IMG_PATH = os.path.join(_BASE_DIR, "GraceDiagram.png")

# Hard-coded plot box in the provided image (pixels)
# Detected from the image: left/right border columns ~76/547, top/bottom ~18/585
GRACE_PLOT_X0 = 76
GRACE_PLOT_X1 = 547
GRACE_PLOT_Y0 = 18
GRACE_PLOT_Y1 = 585

# Axis limits shown on the diagram
GRACE_EO_MIN = 1e-2
GRACE_EO_MAX = 1e3
GRACE_RE_MIN = 1e-1
GRACE_RE_MAX = 1e5
DEBUG_GRACE = False


def _grace_xy_from_eo_re(eo, re_):
    """
    Map (Eo, Re) to pixel coords on the Grace diagram image.
    Both axes are log10.
    Returns (x_pix, y_pix) arrays (float).
    """
    eo = np.asarray(eo, float)
    re_ = np.asarray(re_, float)

    m = np.isfinite(eo) & np.isfinite(re_) & (eo > 0) & (re_ > 0)
    eo = eo[m]
    re_ = re_[m]
    if eo.size == 0:
        return np.array([]), np.array([])

    x = (np.log10(eo) - np.log10(GRACE_EO_MIN)) / (np.log10(GRACE_EO_MAX) - np.log10(GRACE_EO_MIN))
    y = (np.log10(re_) - np.log10(GRACE_RE_MIN)) / (np.log10(GRACE_RE_MAX) - np.log10(GRACE_RE_MIN))

    x = np.clip(x, 0.0, 1.0)
    y = np.clip(y, 0.0, 1.0)

    x_pix = GRACE_PLOT_X0 + x * (GRACE_PLOT_X1 - GRACE_PLOT_X0)
    y_pix = GRACE_PLOT_Y1 - y * (GRACE_PLOT_Y1 - GRACE_PLOT_Y0)

    return x_pix, y_pix


def _load_grace_bg_or_raise():
    if not os.path.exists(GRACE_IMG_PATH):
        raise FileNotFoundError(f"GraceDiagram.png not found at: {GRACE_IMG_PATH}")
    import matplotlib.image as mpimg
    return mpimg.imread(GRACE_IMG_PATH)

def debug_grace_mapping(out_png, test_points=None, show_box=True):
    """
    Writes a diagnostic PNG that overlays:
      - plot box corners
      - a few known (Eo, Re) test points + their mapped pixel locations
    This is only for verifying your pixel mapping. It does NOT affect thesis plots.
    """
    set_thesis_style()
    bg = _load_grace_bg_or_raise()

    if test_points is None:
        # Use corners + a couple midpoints across the expected axis ranges
        test_points = [
            ("min/min", GRACE_EO_MIN, GRACE_RE_MIN),
            ("max/min", GRACE_EO_MAX, GRACE_RE_MIN),
            ("min/max", GRACE_EO_MIN, GRACE_RE_MAX),
            ("max/max", GRACE_EO_MAX, GRACE_RE_MAX),
            ("mid", np.sqrt(GRACE_EO_MIN * GRACE_EO_MAX), np.sqrt(GRACE_RE_MIN * GRACE_RE_MAX)),
            ("Eo=1,Re=1", 1.0, 1.0),
            ("Eo=10,Re=100", 10.0, 100.0),
        ]

    labels = [t[0] for t in test_points]
    eo = np.array([t[1] for t in test_points], float)
    re_ = np.array([t[2] for t in test_points], float)

    x_pix, y_pix = _grace_xy_from_eo_re(eo, re_)

    fig, ax = plt.subplots(figsize=(10.5, 7.0))
    ax.imshow(bg)
    ax.axis("off")

    # draw the plot box you hard-coded
    if show_box:
        xs = [GRACE_PLOT_X0, GRACE_PLOT_X1, GRACE_PLOT_X1, GRACE_PLOT_X0, GRACE_PLOT_X0]
        ys = [GRACE_PLOT_Y0, GRACE_PLOT_Y0, GRACE_PLOT_Y1, GRACE_PLOT_Y1, GRACE_PLOT_Y0]
        ax.plot(xs, ys, linewidth=2.5)
        ax.text(GRACE_PLOT_X0, GRACE_PLOT_Y0, "TOP-LEFT", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.9))
        ax.text(GRACE_PLOT_X1, GRACE_PLOT_Y1, "BOT-RIGHT", fontsize=10, ha="right",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="black", alpha=0.9))

    # plot test points
    ax.scatter(x_pix, y_pix, s=120, edgecolor="black", linewidth=0.8, alpha=0.95)

    # annotate with (Eo,Re) and pixel coords
    for lab, eo_i, re_i, xp, yp in zip(labels, eo, re_, x_pix, y_pix):
        ax.text(
            xp + 6, yp + 6,
            f"{lab}\nEo={eo_i:g}, Re={re_i:g}\n(x={xp:.1f}, y={yp:.1f})",
            fontsize=9,
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.85)
        )

    ax.text(
        0.5, 0.985,
        "DEBUG: Grace Mapping Check (pixel box + test points)",
        transform=ax.transAxes,
        ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.95)
    )

    savefig(out_png)


def _extract_avg_eo_re(df_avg=None, df_tracked=None, settings=None, scale_in_per_px=None):
    # ---------- Preferred: MEDIAN-based Bo/Re ----------
    if (df_tracked is not None) and (not df_tracked.empty) and scale_in_per_px:
        liq, gas = infer_liquid_gas(settings)

        
        if liq in LIQ_PROPS:
            # choose diameter and velocity columns (prefer in-units)
            dcol = "diam_in" if "diam_in" in df_tracked.columns else None
            vcol = "vel_in_s" if "vel_in_s" in df_tracked.columns else None

            if dcol and vcol:
                D = pd.to_numeric(df_tracked[dcol], errors="coerce").to_numpy(float)
                V = pd.to_numeric(df_tracked[vcol], errors="coerce").to_numpy(float)
                m = np.isfinite(D) & np.isfinite(V) & (D > 0) & (V > 0)
                if np.any(m):
                    D_med = float(np.median(D[m]))
                    V_med = float(np.median(V[m]))
                    Re_med, Bo_med = compute_re_bo_from_medians(D_med, V_med, liq, gas)
                    if np.isfinite(Re_med) and np.isfinite(Bo_med) and Re_med > 0 and Bo_med > 0:
                        # Return in (Eo, Re) slot expected by Grace mapping
                        return Bo_med, Re_med

    # ---------- Fallback 1: averages.csv columns ----------
    if df_avg is not None and not df_avg.empty:
        eo_col = pick_col(df_avg, ["avg_Eo", "Eo_avg", "Eo", "avg_eo"])
        re_col = pick_col(df_avg, ["avg_Re", "Re_avg", "Re", "avg_re"])
        if eo_col and re_col:
            eo0 = pd.to_numeric(df_avg[eo_col], errors="coerce").dropna()
            re0 = pd.to_numeric(df_avg[re_col], errors="coerce").dropna()
            if len(eo0) and len(re0):
                return float(eo0.iloc[0]), float(re0.iloc[0])

    # ---------- Fallback 2: tracked file already has Eo/Re per sample ----------
    if df_tracked is not None and not df_tracked.empty:
        eo_col = pick_col(df_tracked, ["Eo", "eo", "EOTVOS", "eotvos"])
        re_col = pick_col(df_tracked, ["Re", "re", "REYNOLDS", "reynolds"])
        if eo_col and re_col:
            eo = pd.to_numeric(df_tracked[eo_col], errors="coerce").to_numpy(float)
            re_ = pd.to_numeric(df_tracked[re_col], errors="coerce").to_numpy(float)
            m = np.isfinite(eo) & np.isfinite(re_) & (eo > 0) & (re_ > 0)
            eo, re_ = eo[m], re_[m]
            if eo.size:
                return float(np.mean(eo)), float(np.mean(re_))

    return np.nan, np.nan



def plot_grace_avg_point(df_tracked, out_png, df_avg=None, label="Run", annotate=False, settings=None, scale_in_per_px=None):

    """
    Single-run Grace diagram plot:
    - ONE point (avg Eo, avg Re)
    - Legend placed outside (right side)
    - No text drawn on the image itself
    """
    set_thesis_style()

    try:
        bg = _load_grace_bg_or_raise()
    except Exception as e:
        fig, ax = plt.subplots(figsize=(7.6, 6.6))
        ax.text(0.5, 0.5, str(e), ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        savefig(out_png)
        return

    # Extract averages
    Eo_avg, Re_avg = _extract_avg_eo_re(
        df_avg=df_avg,
        df_tracked=df_tracked,
        settings=settings,
        scale_in_per_px=scale_in_per_px
    )


    fig, ax = plt.subplots(figsize=(10.5, 7.0))  # wider for legend
    ax.imshow(bg)
    ax.axis("off")

    if not (np.isfinite(Eo_avg) and np.isfinite(Re_avg) and Eo_avg > 0 and Re_avg > 0):
        ax.text(0.5, 0.5, "No valid avg Eo/Re values",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black"))
        savefig(out_png)
        return

    x_pix, y_pix = _grace_xy_from_eo_re([Eo_avg], [Re_avg])

    # Plot point
    sc = ax.scatter(
        x_pix, y_pix,
        s=160,
        alpha=0.95,
        edgecolor="black",
        linewidth=0.9,
        color="tab:red"
    )

    # Figure title (not on image)
    ax.text(
        0.5, 0.985,
        "Grace Diagram (Median Eo / Re)",
        transform=ax.transAxes,
        ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.95)
    )

    # Legend 
    legend_label = f"{label}  (Eo={Eo_avg:.3g}, Re={Re_avg:.3g})"
    ax.legend(
        [sc],
        [legend_label],
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        framealpha=0.98,
        title="Run"
    )

    # Reserve space for legend
    fig.subplots_adjust(right=0.68)

    savefig(out_png)



def plot_grace_avg_points_all_runs(run_points, out_png, title="Grace Diagram (Avg Eo/Re per run)"):
    """
    Batch/combined plot: ONE point per run on ONE Grace diagram.
    - Each run gets a different color
    - Legend is outside the image (right side)
    run_points: list of dicts like:
        {"label": "circle_bubble", "Eo": 12.3, "Re": 456.7}
    """
    set_thesis_style()
    bg = _load_grace_bg_or_raise()

    # Filter to valid points
    clean = []
    for d in run_points:
        lab = str(d.get("label", "run"))
        eo = d.get("Eo", np.nan)
        re_ = d.get("Re", np.nan)
        if np.isfinite(eo) and np.isfinite(re_) and eo > 0 and re_ > 0:
            clean.append({"label": lab, "Eo": float(eo), "Re": float(re_)})

    fig, ax = plt.subplots(figsize=(10.5, 7.0))  # wider to make room for legend
    ax.imshow(bg)
    ax.axis("off")

    if not clean:
        ax.text(0.5, 0.5, "No valid avg Eo/Re points to plot",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black"))
        savefig(out_png)
        return

    labels = [d["label"] for d in clean]
    eo = np.array([d["Eo"] for d in clean], float)
    re_ = np.array([d["Re"] for d in clean], float)

    x_pix, y_pix = _grace_xy_from_eo_re(eo, re_)

    # Distinct colors
    cmap = plt.get_cmap("tab20")
    colors = [cmap(i % cmap.N) for i in range(len(clean))]

    handles = []
    for xp, yp, col, d in zip(x_pix, y_pix, colors, clean):
        sc = ax.scatter([xp], [yp], s=120, alpha=0.95,
                        edgecolor="black", linewidth=0.7, color=col)

        # Legend text
        lab = d["label"]
        lab_txt = f"{lab}  (Eo={d['Eo']:.3g}, Re={d['Re']:.3g})"
        handles.append((sc, lab_txt))

    # Title as a figure-level box
    ax.text(0.5, 0.985, title, transform=ax.transAxes, ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.95))

    # Legend outside on the right
    legend_handles = [h for (h, _) in handles]
    legend_labels  = [t for (_, t) in handles]

    ax.legend(
        legend_handles, legend_labels,
        bbox_to_anchor=(1.02, 1.0),   # outside to the right
        borderaxespad=0.0,
        framealpha=0.98,
        title="Runs"
    )

    # Give room for legend on the right
    fig.subplots_adjust(right=0.68)

    savefig(out_png)


# ============================================================
# Plotting 
# ============================================================
def _find_raw_video(base_dir, stem):
    """
    Try to find the raw video corresponding to this run.
    Looks for common naming patterns and falls back to "any mp4 that starts with stem"
    excluding derived videos like _filtered/_pred_mask/_gt_mask.
    """
    candidates = [
        os.path.join(base_dir, stem + ".mp4"),
        os.path.join(base_dir, stem + "_raw.mp4"),
        os.path.join(base_dir, stem + "_input.mp4"),
        os.path.join(base_dir, stem + "_video.mp4"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    # fallback: any stem*.mp4 excluding derived outputs
    all_mp4 = sorted(glob.glob(os.path.join(base_dir, stem + "*.mp4")))
    bad_suffixes = ("_filtered.mp4", "_pred_mask.mp4", "_gt_mask.mp4")
    all_mp4 = [p for p in all_mp4 if not any(p.endswith(s) for s in bad_suffixes)]
    return all_mp4[0] if all_mp4 else None


def _read_frame_at(cap, frame_idx):
    if cap is None:
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    return frame if ok else None


def make_qualitative_panel_from_videos(raw_mp4, pred_mask_mp4, filtered_mp4, df_tracked,
                                       out_png, start_frame=0, n_frames=5,
                                       gt_mask_mp4=None, show_gt_row=False):
    """
    Creates ONE PNG with n_frames consecutive frames.
    Rows:
      1) raw
      2) pred mask
      3) filtered overlay (with IDs drawn)
      4) (optional) GT mask if show_gt_row=True and gt_mask_mp4 exists
    """
    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read videos for qualitative panel.")

    if not raw_mp4 or not os.path.exists(raw_mp4):
        raise FileNotFoundError(f"Raw video not found: {raw_mp4}")
    if not pred_mask_mp4 or not os.path.exists(pred_mask_mp4):
        raise FileNotFoundError(f"Pred mask video not found: {pred_mask_mp4}")
    if not filtered_mp4 or not os.path.exists(filtered_mp4):
        raise FileNotFoundError(f"Filtered video not found: {filtered_mp4}")

    if show_gt_row and (not gt_mask_mp4 or not os.path.exists(gt_mask_mp4)):
        show_gt_row = False  # silently disable if missing

    set_thesis_style()

    cap_raw = cv2.VideoCapture(raw_mp4)
    cap_pm  = cv2.VideoCapture(pred_mask_mp4)
    cap_f   = cv2.VideoCapture(filtered_mp4)
    cap_gt  = cv2.VideoCapture(gt_mask_mp4) if show_gt_row else None

    # Choose frames
    frs = list(range(int(start_frame), int(start_frame) + int(n_frames)))

    n_rows = 4 if show_gt_row else 3
    fig, axes = plt.subplots(n_rows, len(frs), figsize=(4.0 * len(frs), 3.4 * n_rows))
    if n_rows == 1:
        axes = np.array([axes])
    if len(frs) == 1:
        axes = axes.reshape(n_rows, 1)

    # Make sure we have the columns we need for labels
    dft = df_tracked.copy()
    if "frame" in dft.columns:
        dft["frame"] = pd.to_numeric(dft["frame"], errors="coerce").astype("Int64")
    if "id" in dft.columns:
        dft["id"] = dft["id"].astype(str)

    for j, fr in enumerate(frs):
        # --- RAW ---
        raw_bgr = _read_frame_at(cap_raw, fr)
        if raw_bgr is not None:
            raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
            axes[0, j].imshow(raw_rgb)
        axes[0, j].set_title(f"Frame {fr}")
        axes[0, j].axis("off")

        # --- PRED MASK ---
        pm_bgr = _read_frame_at(cap_pm, fr)
        if pm_bgr is not None:
            if pm_bgr.ndim == 3:
                pm_gray = cv2.cvtColor(pm_bgr, cv2.COLOR_BGR2GRAY)
            else:
                pm_gray = pm_bgr
            axes[1, j].imshow(pm_gray, cmap="gray", vmin=0, vmax=255)
        axes[1, j].axis("off")

        # --- FILTERED (overlay) + IDs ---
        f_bgr = _read_frame_at(cap_f, fr)
        if f_bgr is not None:
            f_rgb = cv2.cvtColor(f_bgr, cv2.COLOR_BGR2RGB)
            axes[2, j].imshow(f_rgb)

            # Draw IDs on the filtered frame using tracked CSV
            if ("frame" in dft.columns) and ("cx" in dft.columns) and ("cy" in dft.columns) and ("id" in dft.columns):
                d = dft[dft["frame"].astype("Int64") == int(fr)]
                for _, r in d.iterrows():
                    cx = float(r["cx"])
                    cy = float(r["cy"])
                    tid = str(r["id"])
                    axes[2, j].plot(cx, cy, marker="o", markersize=5)
                    axes[2, j].text(cx + 4, cy + 4, tid, fontsize=9, weight="bold",
                                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
        axes[2, j].axis("off")

        # --- OPTIONAL GT MASK ROW ---
        if show_gt_row:
            gt_bgr = _read_frame_at(cap_gt, fr) if cap_gt is not None else None
            if gt_bgr is not None:
                if gt_bgr.ndim == 3:
                    gt_gray = cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2GRAY)
                else:
                    gt_gray = gt_bgr
                axes[3, j].imshow(gt_gray, cmap="gray", vmin=0, vmax=255)
            axes[3, j].axis("off")

    # Row labels on the left
    row_names = ["Raw", "Pred Mask", "Filtered + IDs"] + (["GT Mask"] if show_gt_row else [])
    for i, name in enumerate(row_names):
        axes[i, 0].set_ylabel(name, fontsize=14, rotation=90, labelpad=18)

    cap_raw.release()
    cap_pm.release()
    cap_f.release()
    if cap_gt is not None:
        cap_gt.release()

    fig.suptitle("Qualitative Detection & Tracking (5 Consecutive Frames)", y=1.01, fontsize=18)
    savefig(out_png)

def plot_delta_r_histogram(dr_steps_px, out_png):
    """
    Histogram of frame-to-frame centroid displacement Δr (pixels).
    Used to define representative Δr for velocity uncertainty propagation.
    """
    set_thesis_style()

    fig, ax = plt.subplots(figsize=FIG_STANDARD)

    dr = np.asarray(dr_steps_px, float)
    dr = dr[np.isfinite(dr)]

    if dr.size == 0:
        ax.text(0.5, 0.5, "No Δr data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # Robust clipping to avoid one bad ID jump dominating the axis
    dr_plot, lo, hi, n_total, n_kept = clip_for_hist(dr, 1, 99)
    n_clipped = n_total - n_kept

    ax.hist(dr_plot, bins=20, edgecolor="black", linewidth=0.6)
    ax.set_xlabel("Frame-to-frame displacement Δr (px)")
    ax.set_ylabel("Count")
    ax.set_title("Centroid Frame-to-Frame Displacement")

    ax.grid(True)

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(dr):.3g} px",
        f"Mean = {np.mean(dr):.3g} px",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}] px")

    add_statbox(ax, lines)
    savefig(out_png)

def plot_distribution(x, out_png, title, x_label, units, bins=20, legend_outside=False):
    set_thesis_style()
    x = _clean(x)

    # ---- NEW: clip plotted range
    x_plot, lo, hi, n_total, n_kept = clip_for_hist(x, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=FIG_STANDARD)
    if x_plot.size:
        ax.hist(x_plot, bins=bins, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"{x_label} ({units})")
    ax.set_ylabel("Count")

    st = stats_dict(x)  # stats on FULL data (not clipped)
    lines = [
        f"N = {st['N']}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Mean = {st['mean']:.4g} {units}" if st["N"] else "Mean = NA",
        f"Median = {st['median']:.4g} {units}" if st["N"] else "Median = NA",
        f"Std = {st['std']:.4g} {units}" if st["N"] else "Std = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}]")

    add_statbox(ax, lines)
    savefig(out_png)


def plot_distribution_overlay(series_list, out_png, title, x_label, units, bins=20):
    set_thesis_style()
    fig, ax = plt.subplots(figsize=(10.5, 5.6))  # wider, like your Grace plot
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"{x_label} ({units})")
    ax.set_ylabel("Count")

    allv = np.concatenate([_clean(v) for _, v in series_list if _clean(v).size]) if series_list else np.array([])
    if allv.size == 0:
        ax.text(0.5, 0.5, "No data to overlay", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    lo, hi = robust_limits(allv, 1, 99, 0.10)
    bins_edges = np.linspace(lo, hi, bins + 1)

    # collect handles/labels like your Grace function does
    legend_handles = []
    legend_labels  = []

    for label, v in series_list:
        v = _clean(v)
        if v.size == 0:
            continue

        n, edges, patches = ax.hist(
            v,
            bins=bins_edges,
            histtype="step",
            linewidth=2.0
        )

        # histtype="step" returns a list of Polygon patches; use the first as legend handle
        if patches:
            legend_handles.append(patches[0])
            legend_labels.append(f"{label} (N={v.size})")

    ax.legend(
        legend_handles, legend_labels,
        bbox_to_anchor=(1.02, 1.0),   # outside right
        borderaxespad=0.0,
        framealpha=0.98,
        title="Runs"
    )

    # reserve space on the right for legend (same pattern as Grace)
    fig.subplots_adjust(right=0.68)

    savefig(out_png)   # use the same saver as Grace for consistent layout
    plt.close(fig)


def plot_distribution_zoom_full(x, out_png, title, x_label, units, bins_zoom=30, bins_full=55):
    set_thesis_style()
    x = _clean(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 5.4))

    if x.size:
        z0, z1 = robust_limits(x, 1, 99, 0.10)
        ax1.hist(x, bins=bins_zoom, edgecolor="black", linewidth=0.6)
        ax1.set_xlim(z0, z1)
    ax1.grid(True)
    ax1.set_title("Zoomed (1–99th percentile)")
    ax1.set_xlabel(f"{x_label} ({units})")
    ax1.set_ylabel("Count")

    st = stats_dict(x)
    if st["N"]:
        ax1.axvline(st["mean"], linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax1.axvline(st["median"], linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax1.legend(loc="lower left", framealpha=1.0)

    add_statbox(ax1, [
        f"N = {st['N']}",
        f"MAE = {mae(x):.4g} {units}" if st["N"] else "MAE = NA",
        f"Bias = {bias(x):.4g} {units}" if st["N"] else "Bias = NA",
        f"Std = {st['std']:.4g} {units}" if st["N"] else "Std = NA",
    ])

    if x.size:
        ax2.hist(x, bins=bins_full, edgecolor="black", linewidth=0.6)
    ax2.grid(True)
    ax2.set_title("Full range")
    ax2.set_xlabel(f"{x_label} ({units})")
    ax2.set_ylabel("Count")
    if st["N"]:
        ax2.axvline(st["mean"], linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax2.axvline(st["median"], linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax2.legend(loc="lower left", framealpha=1.0)

    fig.suptitle(clean_plot_title(title), y=1.02, fontsize=18)
    savefig(out_png)

def _add_one_to_one_and_pct_band(ax, xmin, xmax, pct=0.10,
                                line_kw=None, band_kw=None):
    """
    Draw y=x and y=(1±pct)x over [xmin, xmax].
    """
    if line_kw is None:
        line_kw = dict(linestyle="--", linewidth=2.0)
    if band_kw is None:
        band_kw = dict(linestyle=":", linewidth=1.8, alpha=0.9)

    xx = np.array([xmin, xmax], float)

    # 1:1
    ax.plot(xx, xx, label="1:1 line", **line_kw)

    # ±pct lines
    #ax.plot(xx, (1.0 + pct) * xx, label=f"+{int(pct*100)}%", **band_kw)
    #ax.plot(xx, (1.0 - pct) * xx, label=f"-{int(pct*100)}%", **band_kw)


def plot_measured_vs_true(x_true, y_meas, out_png, title, units,
                          y_pos_px=None,            
                          y_pos_label="y position (px)",
                          pct_band=0.10,           
                          cmap="viridis"):
    set_thesis_style()
    x_true = np.asarray(x_true, float)
    y_meas = np.asarray(y_meas, float)

    if y_pos_px is not None:
        y_pos_px = np.asarray(y_pos_px, float)
        m = np.isfinite(x_true) & np.isfinite(y_meas) & np.isfinite(y_pos_px)
        x_true, y_meas, y_pos_px = x_true[m], y_meas[m], y_pos_px[m]
    else:
        m = np.isfinite(x_true) & np.isfinite(y_meas)
        x_true, y_meas = x_true[m], y_meas[m]

    fig, ax = plt.subplots(figsize=(7.4, 5.8))

    if x_true.size == 0:
        ax.text(0.5, 0.5, "No matched samples", ha="center", va="center", transform=ax.transAxes)
        ax.grid(True)
        ax.set_title(clean_plot_title(title))
        ax.set_xlabel(f"True ({units})")
        ax.set_ylabel(f"Measured ({units})")
        savefig(out_png)
        return

    # --- scatter ---
    if y_pos_px is not None:
        norm = mpl.colors.Normalize(vmin=np.nanmin(y_pos_px), vmax=np.nanmax(y_pos_px))
        sc = ax.scatter(
            x_true, y_meas,
            c=y_pos_px, cmap=cmap, norm=norm,
            s=30, alpha=0.85,
            edgecolor="black", linewidth=0.35,
            label="Matched samples"
        )
        cbar = plt.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label(y_pos_label)
    else:
        ax.scatter(
            x_true, y_meas,
            s=30, alpha=0.85,
            edgecolor="black", linewidth=0.35,
            label="Matched samples"
        )

    # --- limits for 1:1 + bands ---
    xmin = float(min(np.min(x_true), np.min(y_meas)))
    xmax = float(max(np.max(x_true), np.max(y_meas)))
    if xmin == xmax:
        xmin -= 1.0
        xmax += 1.0

    # --- 1:1 and ±10% lines ---
    _add_one_to_one_and_pct_band(ax, xmin, xmax, pct=pct_band)

    pad = 0.05 * (xmax - xmin)   
    xmin -= pad
    xmax += pad
    
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"True ({units})")
    ax.set_ylabel(f"Measured ({units})")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(xmin, xmax)
    ax.set_aspect("equal", adjustable="box")

    # --- stats box stays the same ---
    err = y_meas - x_true
    add_statbox(ax, [
        f"N = {int(np.isfinite(err).sum())}",
        f"MAE = {mae(err):.4g} {units}",
        f"RMSE = {rmse(err):.4g} {units}",
        f"Bias = {bias(err):.4g} {units}",
    ])

    ax.legend(loc="lower left", framealpha=1.0)
    savefig(out_png)

def plot_velocity_vs_diameter(
    diam, vel, out_png, units_len, units_vel,
    clip_vel_high_pct=99.5,          # for plotting only (same spirit as your current clipping)
    do_regression=True,
    regression_use_clipped=False,    # False = regression on true values (recommended)
):
    set_thesis_style()

    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.stats import linregress

    diam = np.asarray(diam, float)
    vel  = np.asarray(vel, float)

    # basic clean
    m = np.isfinite(diam) & np.isfinite(vel)
    diam, vel = diam[m], vel[m]

    fig, ax = plt.subplots(figsize=FIG_WIDE)

    if diam.size == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.grid(True)
        ax.set_title("Rise Velocity vs Diameter")
        ax.set_xlabel(f"Diameter ({units_len})")
        ax.set_ylabel(f"Rise velocity ({units_vel})")
        savefig(out_png)
        return

    # ---- plotting clip (does NOT have to affect regression) ----
    yhi = np.nanpercentile(vel, clip_vel_high_pct) if (clip_vel_high_pct is not None) else np.nan
    do_clip = np.isfinite(yhi)

    vel_plot = np.minimum(vel, yhi) if do_clip else vel

    ax.scatter(
        diam, vel_plot,
        s=26, alpha=0.70,
        edgecolor="black", linewidth=0.25,
        label="Instantaneous samples"
    )

    if do_clip:
        ax.set_ylim(bottom=min(0.0, np.nanpercentile(vel_plot, 0.5)), top=yhi)
        ax.text(
            0.99, 0.02,
            f"Plot clipped at {clip_vel_high_pct:.1f}th pct ({yhi:.3g} {units_vel})",
            transform=ax.transAxes,
            ha="right", va="bottom"
        )

    # ---- regression ----
    if do_regression and diam.size >= 3:
        y_for_fit = vel_plot if (regression_use_clipped and do_clip) else vel
        x_for_fit = diam

        mm = np.isfinite(x_for_fit) & np.isfinite(y_for_fit)
        x_for_fit = x_for_fit[mm]
        y_for_fit = y_for_fit[mm]

        if x_for_fit.size >= 3:
            reg = linregress(x_for_fit, y_for_fit)
            slope = reg.slope
            intercept = reg.intercept
            r2 = reg.rvalue**2
            pval = reg.pvalue

            # draw fit line across data range
            xx = np.array([np.min(diam), np.max(diam)], float)
            yy = intercept + slope * xx
            ax.plot(xx, yy, linestyle="--", linewidth=2.8, color="orange", label="Linear regression")

            # stats box (keep it simple/defensible)
            add_statbox(ax, [
                f"N = {int(x_for_fit.size)}",
                f"Slope = {slope:.4g} ({units_vel})/({units_len})",
                f"Intercept = {intercept:.4g} {units_vel}",
                f"R² = {r2:.4g}",
                f"p = {pval:.4g}",
            ])

    ax.grid(True)
    ax.set_title("Rise Velocity vs Diameter")
    ax.set_xlabel(f"Diameter ({units_len})")
    ax.set_ylabel(f"Rise velocity ({units_vel})")
    ax.legend(
        bbox_to_anchor=(1.03, 0.55),
        borderaxespad=0.0,
        framealpha=1.0
    )

    savefig(out_png)


def plot_track_lengths(track_len_frames, fps, out_png):
    set_thesis_style()
    L = np.asarray(track_len_frames, float)
    L = L[np.isfinite(L)]
    dur = (L / fps) if (fps and fps > 0) else None

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15.5, 5.4))

    if L.size:
        ax1.hist(L, bins=20, edgecolor="black", linewidth=0.6)
    ax1.grid(True)
    ax1.set_title("Track length distribution")
    ax1.set_xlabel("Track length (frames)")
    ax1.set_ylabel("Count")
    st = stats_dict(L)
    add_statbox(ax1, [
        f"N tracks = {st['N']}",
        f"Mean = {st['mean']:.4g} frames" if st["N"] else "Mean = NA",
        f"Median = {st['median']:.4g} frames" if st["N"] else "Median = NA",
        f"Std = {st['std']:.4g} frames" if st["N"] else "Std = NA",
    ])

    if dur is not None:
        if dur.size:
            ax2.hist(dur, bins=20, edgecolor="black", linewidth=0.6)
        ax2.grid(True)
        ax2.set_title("Track duration distribution")
        ax2.set_xlabel("Track duration (s)")
        ax2.set_ylabel("Count")
        st2 = stats_dict(dur)
        add_statbox(ax2, [
            f"N tracks = {st2['N']}",
            f"Mean = {st2['mean']:.4g} s" if st2["N"] else "Mean = NA",
            f"Median = {st2['median']:.4g} s" if st2["N"] else "Median = NA",
            f"Std = {st2['std']:.4g} s" if st2["N"] else "Std = NA",
        ])
    else:
        ax2.axis("off")
        ax2.text(0.5, 0.5, "FPS not available\n(duration plot skipped)", ha="center", va="center")

    savefig(out_png)

def plot_error_vs_time_timecolored(x, err, out_png, title, xlab, units_y):
    set_thesis_style()
    x = np.asarray(x, float)
    e = np.asarray(err, float)
    m = np.isfinite(x) & np.isfinite(e)
    x, e = x[m], e[m]

    fig, ax = plt.subplots(figsize=(7.8, 5.4))
    if x.size:
        sc = ax.scatter(
            x, e,
            c=x, cmap="viridis",
            s=26, alpha=0.85,
            edgecolor="black", linewidth=0.25,
            label="Error samples"
        )
        cbar = plt.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label(xlab)

    ax.axhline(0, linestyle="--", linewidth=2, label="Zero error")
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(xlab)
    ax.set_ylabel(f"Error ({units_y})")

    st = stats_dict(e)
    add_statbox(
        ax,
        [
            f"N = {st['N']}",
            f"MAE = {mae(e):.4g} {units_y}",
            f"RMSE = {rmse(e):.4g} {units_y}",
            f"Bias = {bias(e):.4g} {units_y}",
        ],
    )

    ax.legend(loc="lower left", framealpha=1.0)
    savefig(out_png)

def plot_trajectories_2d_timecolored(df, out_png, units_len,
                                    max_tracks=25, min_points=8,
                                    upward_frac_min=0.70,
                                    show_longest_contiguous_segment=True,
                                    jump_max_in=0.12,      
                                    jump_max_px=25.0,      
                                    gap_max_s=0.03,        
                                    gap_max_frames=2,      
                                    linewidth=2.0, alpha=0.95):

    set_thesis_style()

    import numpy as np
    import pandas as pd
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection

    # -------------------- guard --------------------
    if df is None or df.empty:
        fig, ax = plt.subplots(figsize=(8.2, 6.0))
        ax.set_title("2D Trajectories Colored by Time")
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # -------------------- columns --------------------
    xcol = "x_in" if "x_in" in df.columns else "cx"
    ycol = "y_in" if "y_in" in df.columns else "cy"

    has_time = ("t_s" in df.columns) and df["t_s"].notna().any()
    tcol = "t_s" if has_time else ("time_s" if "time_s" in df.columns else ("frame" if "frame" in df.columns else None))
    if tcol is None:

        df = df.copy()
        df["frame"] = np.arange(len(df))
        tcol = "frame"
        has_time = False

    xlab = f"x ({units_len})" if xcol.endswith("_in") else "x (px)"
    ylab = f"y ({units_len})" if ycol.endswith("_in") else "y (px)"

    d0 = df.copy()
    for c in ("id", xcol, ycol, tcol):
        if c in d0.columns:
            d0[c] = pd.to_numeric(d0[c], errors="coerce")
    d0 = d0[np.isfinite(d0["id"]) & np.isfinite(d0[xcol]) & np.isfinite(d0[ycol]) & np.isfinite(d0[tcol])].copy()
    if d0.empty:
        fig, ax = plt.subplots(figsize=(8.2, 6.0))
        ax.set_title("Representative Bubble Trajectories")
        ax.text(0.5, 0.5, "No valid numeric data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # thresholds depend on units
    jump_max = float(jump_max_in if xcol.endswith("_in") else jump_max_px)
    gap_max  = float(gap_max_s  if has_time else gap_max_frames)

    # -------------------- helpers --------------------
    def _longest_contiguous_run(x, y, t, jump_max, gap_max):
        """Return x,y,t for the longest contiguous segment (no teleports or big time gaps)."""
        if len(x) < 2:
            return x, y, t

        dx = np.diff(x); dy = np.diff(y); dt = np.diff(t)
        dr = np.hypot(dx, dy)

        if has_time:
            bad = (dr > jump_max) | (dt > gap_max) | (dt <= 0)
        else:
            # frames should increase by ~1; allow small skips
            bad = (dr > jump_max) | (dt > gap_max) | (dt <= 0)

        # segments are separated at bad edges
        starts = [0]
        for i, b in enumerate(bad):
            if b:
                starts.append(i + 1)
        starts = np.array(starts, dtype=int)

        ends = np.r_[starts[1:], len(x)]
        lens = ends - starts
        k = int(np.argmax(lens))
        s0, s1 = int(starts[k]), int(ends[k])
        return x[s0:s1], y[s0:s1], t[s0:s1]

    def _stability_score(g):
        """Higher = more stable upward trajectory, fewer teleports, smoother."""
        g = g.sort_values(tcol)
        x = g[xcol].to_numpy(float)
        y = g[ycol].to_numpy(float)
        t = g[tcol].to_numpy(float)

        if len(x) < min_points:
            return None

        dx = np.diff(x); dy = np.diff(y); dt = np.diff(t)
        dr = np.hypot(dx, dy)

        good = np.isfinite(dr) & np.isfinite(dt) & (dt > 0)
        if good.sum() < (min_points - 1):
            return None

        dx = dx[good]; dy = dy[good]; dt = dt[good]; dr = dr[good]

        # rising = y decreases (image coords)
        up = (dy < 0)
        up_frac = float(up.mean())
        if up_frac < upward_frac_min:
            return None

        tele = float((dr > jump_max).mean())
        gap  = float((dt > gap_max).mean()) if (has_time or "frame" in d0.columns) else 0.0

        # smoothness / jaggedness via heading changes
        ang = np.arctan2(dy, dx)
        dang = np.abs(np.diff(ang))
        # wrap to [0, pi]
        dang = np.minimum(dang, np.pi - (dang % np.pi))
        jag = float(np.nanmedian(dang)) if dang.size else 0.0

        # sideways-ness (lower is better)
        lat_ratio = float(np.median(np.abs(dx) / (np.abs(dy) + 1e-9)))

        # vertical speed consistency 
        vy = dy / dt
        vy_med = np.median(vy)
        vy_mad = np.median(np.abs(vy - vy_med)) + 1e-9
        v_consistency = 1.0 / (1.0 + vy_mad)

        L = len(g)

        score = (
            3.0 * up_frac
            + 0.02 * L
            + 0.6 * v_consistency
            - 3.0 * tele
            - 2.0 * gap
            - 0.8 * jag
            - 0.6 * lat_ratio
        )
        return score

    # -------------------- choose IDs by stability --------------------
    scores = []
    ids = []
    for tid, g in d0.groupby("id", sort=False):
        sc = _stability_score(g)
        if sc is None:
            continue
        ids.append(int(tid))
        scores.append(float(sc))

    if scores:
        order = np.argsort(-np.asarray(scores))
        top_ids = [ids[i] for i in order[:max_tracks]]
    else:
        # fallback to longest if nothing passes
        lens = d0.groupby("id").size().sort_values(ascending=False)
        top_ids = list(lens.head(min(max_tracks, len(lens))).index.astype(int))

    # -------------------- normalize time/frame for colormap --------------------
    tvals = d0[d0["id"].isin(top_ids)][tcol].to_numpy(float)
    tvals = tvals[np.isfinite(tvals)]
    if tvals.size == 0:
        tmin, tmax = 0.0, 1.0
    else:
        tmin, tmax = float(np.min(tvals)), float(np.max(tvals))
        if tmin == tmax:
            tmax = tmin + 1.0

    norm = mpl.colors.Normalize(vmin=tmin, vmax=tmax)
    cmap = plt.get_cmap("viridis")

    # -------------------- plot --------------------
    fig, ax = plt.subplots(figsize=(8.6, 6.3))

    drawn = 0
    for tid in top_ids:
        g = d0[d0["id"] == tid].sort_values(tcol)
        x = g[xcol].to_numpy(float)
        y = g[ycol].to_numpy(float)
        t = g[tcol].to_numpy(float)

        if show_longest_contiguous_segment:
            x, y, t = _longest_contiguous_run(x, y, t, jump_max=jump_max, gap_max=gap_max)

        if len(x) < min_points:
            continue

        pts = np.column_stack([x, y])
        segs = np.stack([pts[:-1], pts[1:]], axis=1)

        lc = LineCollection(segs, cmap=cmap, norm=norm, linewidth=linewidth, alpha=alpha)
        lc.set_array(t[:-1])
        ax.add_collection(lc)

        # start / end markers
        ax.scatter([x[0]], [y[0]], s=22, edgecolor="black", linewidth=0.25, marker="o", zorder=5)
        ax.scatter([x[-1]], [y[-1]], s=28, edgecolor="black", linewidth=0.25, marker="s", zorder=5)
        drawn += 1

    ax.autoscale()
    ax.grid(True)
    ax.set_title("Representative Bubble Trajectories")
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.invert_yaxis()

    if drawn == 0:
        ax.text(0.5, 0.5,
                "No valid tracks after stability filtering\n(try lowering upward_frac_min or min_points)",
                ha="center", va="center",
                transform=ax.transAxes,
                bbox=dict(boxstyle="round", fc="white", ec="black"))

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Time (s)" if has_time else "Frame")

    savefig(out_png)

def plot_mean_velocity_vs_mean_diameter_per_track(
    df_tracks,
    out_png,
    units_len,
    units_vel,
    diam_col=None,     # optional override
    vel_col=None,      # optional override
    min_points_per_track=20,
    do_regression=True
):
    """
    One point per track:
      x = mean diameter (per track)
      y = mean velocity (per track)

    Requires df_tracks columns: id + diam_col + vel_col.
    If diam_col/vel_col are None, it will auto-pick based on what's present.
    """
    set_thesis_style()

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy.stats import linregress

    if df_tracks is None or df_tracks.empty:
        fig, ax = plt.subplots(figsize=FIG_WIDE)
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.grid(True)
        ax.set_title("Mean Rise Velocity vs Mean Diameter (Per Track)")
        ax.set_xlabel(f"Mean diameter ({units_len})")
        ax.set_ylabel(f"Mean rise velocity ({units_vel})")
        savefig(out_png)
        return None

    d = df_tracks.copy()

    # ---- choose columns automatically if not provided ----
    if diam_col is None:
        diam_col = "diam_in" if "diam_in" in d.columns else ("diam_px" if "diam_px" in d.columns else None)
    if vel_col is None:
        vel_col = "vel_in_s" if "vel_in_s" in d.columns else ("vel_px_s" if "vel_px_s" in d.columns else None)

    if ("id" not in d.columns) or (diam_col is None) or (vel_col is None):
        fig, ax = plt.subplots(figsize=FIG_WIDE)
        ax.text(
            0.5, 0.5,
            f"Missing required cols.\nNeed: id, {diam_col}, {vel_col}",
            ha="center", va="center", transform=ax.transAxes
        )
        ax.axis("off")
        savefig(out_png)
        return None

    d["id"] = d["id"].astype(str)
    d[diam_col] = pd.to_numeric(d[diam_col], errors="coerce")
    d[vel_col]  = pd.to_numeric(d[vel_col],  errors="coerce")

    # ---- per-track means + N ----
    g = d.groupby("id", sort=False).agg(
        N=(diam_col, "count"),
        mean_diam=(diam_col, "mean"),
        mean_vel=(vel_col, "mean")
    ).reset_index()

    # need both to be finite
    g["N_vel"] = d.groupby("id", sort=False)[vel_col].count().values
    g["N"] = np.minimum(g["N"].to_numpy(int), g["N_vel"].to_numpy(int))

    g = g[np.isfinite(g["mean_diam"]) & np.isfinite(g["mean_vel"])].copy()
    g = g[g["N"] >= int(min_points_per_track)].copy()

    fig, ax = plt.subplots(figsize=(7.8, 5.8))

    if g.empty:
        ax.text(0.5, 0.5, "No valid tracks after filtering", ha="center", va="center", transform=ax.transAxes)
        ax.grid(True)
        ax.set_title("Mean Rise Velocity vs Mean Diameter (Per Track)")
        ax.set_xlabel(f"Mean diameter ({units_len})")
        ax.set_ylabel(f"Mean rise velocity ({units_vel})")
        savefig(out_png)
        return None

    ax.scatter(
        g["mean_diam"].to_numpy(float),
        g["mean_vel"].to_numpy(float),
        s=40, alpha=0.80,
        edgecolor="black", linewidth=0.35,
        label=f"Tracks (N≥{min_points_per_track})"
    )

    # ---- optional regression ----
    if do_regression and len(g) >= 3:
        x = g["mean_diam"].to_numpy(float)
        y = g["mean_vel"].to_numpy(float)
        reg = linregress(x, y)

        xx = np.array([np.min(x), np.max(x)], float)
        yy = reg.intercept + reg.slope * xx
        ax.plot(xx, yy, linestyle="--", linewidth=2.8, color="orange", label="Linear regression")

        add_statbox(ax, [
            f"N tracks = {len(g)}",
            f"Slope = {reg.slope:.4g} ({units_vel})/({units_len})",
            f"Intercept = {reg.intercept:.4g} {units_vel}",
            f"R² = {(reg.rvalue**2):.4g}",
            f"p = {reg.pvalue:.4g}",
        ])
    else:
        add_statbox(ax, [f"N tracks = {len(g)}"])

    ax.grid(True)
    ax.set_title("Mean Rise Velocity vs Mean Diameter Per Track")
    ax.set_xlabel(f"Mean diameter ({units_len})")
    ax.set_ylabel(f"Mean rise velocity ({units_vel})")
    ax.legend(
        bbox_to_anchor=(1.05, 0.55),
        borderaxespad=0.0,
        framealpha=1.0
    )

    savefig(out_png)

    return g 

def plot_velocity_vs_time(df, out_png, units_vel,
                          max_tracks=6,
                          min_points=8,
                          upward_frac_min=0.70,
                          jump_max_in=0.12,     
                          jump_max_px=25.0,    
                          gap_max_s=0.03,      
                          gap_max_frames=2,     
                          clip_high_quantile=0.995,
                          linewidth=2.0,alpha=0.9):
 
    set_thesis_style()

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    if df is None or df.empty or (("vel_px_s" not in df.columns) and ("vel_in_s" not in df.columns)):
        fig, ax = plt.subplots(figsize=(7.8, 5.4))
        ax.text(0.5, 0.5, "No velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # ----- time column -----
    has_time = ("t_s" in df.columns) and df["t_s"].notna().any()
    tcol = "t_s" if has_time else ("time_s" if "time_s" in df.columns else "frame")
    xlabel = "Time (s)" if (tcol in ("t_s", "time_s")) else "Frame"

    # ----- pick vel column -----
    vel_col = "vel_px_s" if units_vel == "px/s" else "vel_in_s"
    if vel_col not in df.columns:
        # fallback if requested units missing
        vel_col = "vel_in_s" if "vel_in_s" in df.columns else "vel_px_s"

    # ----- choose position columns  -----
    xcol = "x_in" if "x_in" in df.columns else ("cx" if "cx" in df.columns else None)
    ycol = "y_in" if "y_in" in df.columns else ("cy" if "cy" in df.columns else None)

    # If we can't teleport-detect (missing x/y), we still do stability selection by vel only.
    can_jump = (xcol is not None) and (ycol is not None)

    d0 = df.copy()
    for c in ("id", tcol, vel_col, xcol, ycol):
        if c and c in d0.columns:
            d0[c] = pd.to_numeric(d0[c], errors="coerce")

    # keep only rows with id/time/vel
    req = np.isfinite(d0["id"]) & np.isfinite(d0[tcol]) & np.isfinite(d0[vel_col])
    if can_jump:
        req &= np.isfinite(d0[xcol]) & np.isfinite(d0[ycol])
    d0 = d0[req].copy()

    if d0.empty:
        fig, ax = plt.subplots(figsize=(7.8, 5.4))
        ax.text(0.5, 0.5, "No valid numeric velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    jump_max = float(jump_max_in if (can_jump and xcol.endswith("_in")) else jump_max_px)
    gap_max  = float(gap_max_s if has_time else gap_max_frames)

    def _segments_by_gaps(x, y, t):
        """Return list of (i0,i1) segments that are contiguous (no teleports, no big dt)."""
        n = len(t)
        if n < 2:
            return [(0, n)]
        dx = np.diff(x); dy = np.diff(y); dt = np.diff(t)
        dr = np.hypot(dx, dy)
        if has_time:
            bad = (dr > jump_max) | (dt > gap_max) | (dt <= 0)
        else:
            bad = (dr > jump_max) | (dt > gap_max) | (dt <= 0)
        cuts = [0]
        for i, b in enumerate(bad):
            if b:
                cuts.append(i + 1)
        cuts = cuts + [n]
        segs = []
        for a, b in zip(cuts[:-1], cuts[1:]):
            if (b - a) >= min_points:
                segs.append((a, b))
        return segs if segs else [(0, n)]

    def _stability_score(g):
        g = g.sort_values(tcol)
        t = g[tcol].to_numpy(float)
        v = g[vel_col].to_numpy(float)

        if len(t) < min_points:
            return None

        v_med = np.nanmedian(v)
        v_mad = np.nanmedian(np.abs(v - v_med)) + 1e-9
        spike_frac = float(np.mean(np.abs(v - v_med) > 8.0 * v_mad))

        up_frac = 1.0
        tele_frac = 0.0
        gap_frac = 0.0
        longest_seg = len(t)

        if can_jump:
            x = g[xcol].to_numpy(float)
            y = g[ycol].to_numpy(float)

            # segment stats
            dx = np.diff(x); dy = np.diff(y); dt = np.diff(t)
            dr = np.hypot(dx, dy)
            good = np.isfinite(dr) & np.isfinite(dt) & (dt > 0)
            if good.sum() < (min_points - 1):
                return None
            dy_g = dy[good]
            up_frac = float((dy_g < 0).mean())
            if up_frac < upward_frac_min:
                return None

            tele_frac = float((dr[good] > jump_max).mean())
            gap_frac  = float((dt[good] > gap_max).mean())

            # longest contiguous segment length
            segs = _segments_by_gaps(x, y, t)
            longest_seg = max((b - a) for a, b in segs) if segs else len(t)

        score = (
            3.0 * up_frac
            + 0.03 * longest_seg
            - 2.5 * spike_frac
            - 3.0 * tele_frac
            - 2.0 * gap_frac
        )
        return score

    # ----- choose representative IDs by stability -----
    scored = []
    for tid, g in d0.groupby("id", sort=False):
        sc = _stability_score(g)
        if sc is None:
            continue
        scored.append((int(tid), float(sc)))

    if scored:
        scored.sort(key=lambda x: x[1], reverse=True)
        ids = [tid for tid, _ in scored[:max_tracks]]
    else:
        # fallback: longest
        lens = d0.groupby("id").size().sort_values(ascending=False)
        ids = list(lens.head(max_tracks).index.astype(int))

    # ----- compute plot y-clip threshold -----
    clip_thr = None
    if clip_high_quantile is not None:
        vv = d0[vel_col].to_numpy(float)
        vv = vv[np.isfinite(vv)]
        if vv.size:
            clip_thr = float(np.quantile(vv, clip_high_quantile))

    # ----- plot -----
    fig, ax = plt.subplots(figsize=(7.8, 5.4))

    for tid in ids:
        g = d0[d0["id"] == tid].sort_values(tcol)

        t = g[tcol].to_numpy(float)
        v = g[vel_col].to_numpy(float)

        if can_jump:
            x = g[xcol].to_numpy(float)
            y = g[ycol].to_numpy(float)
            segs = _segments_by_gaps(x, y, t)
        else:
            segs = [(0, len(t))]

        # plot each contiguous segment separately 
        for a, b in segs:
            tt = t[a:b]
            vv = v[a:b]

            if clip_thr is not None:
                vv = np.minimum(vv, clip_thr)

            ax.plot(tt, vv, linewidth=linewidth, alpha=alpha)

    ax.grid(True)
    ax.set_title("Bubble Rise Velocity vs Time")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"Rise velocity ({units_vel})")

    if clip_thr is not None:
        ax.text(0.99, 0.01,
                f"Plot clipped at {clip_high_quantile*100:.1f}th pct ({clip_thr:.2f} {units_vel})",
                ha="right", va="bottom", transform=ax.transAxes, fontsize=9)

    savefig(out_png)


def plot_velocity_smoothness(df, out_png, units_vel):
    set_thesis_style()

    if df is None or df.empty or "vel_px_s" not in df.columns:
        fig, ax = plt.subplots(figsize=FIG_STANDARD)
        ax.text(0.5, 0.5, "No velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    dv_all = []
    for _, d in df.groupby("id"):
        v = d["vel_px_s"].to_numpy(float)
        v = v[np.isfinite(v)]
        if len(v) >= 2:
            dv_all.extend(np.abs(np.diff(v)))

    dv_all = np.asarray(dv_all, float)

    # ---- clip plotting range to avoid one giant outlier ruining the axis
    dv_plot, lo, hi, n_total, n_kept = clip_for_hist(dv_all, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=FIG_STANDARD)
    if dv_plot.size:
        ax.hist(dv_plot, bins=20, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title("Velocity Change Between Frames")
    ax.set_xlabel(f"|Δv| ({units_vel})")
    ax.set_ylabel("Count")

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(dv_all):.4g} {units_vel}" if n_total else "Median = NA",
        f"Mean = {np.mean(dv_all):.4g} {units_vel}" if n_total else "Mean = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}]")

    add_statbox(ax, lines)
    savefig(out_png)


def plot_relative_error(err, true, out_png):
    set_thesis_style()

    err = np.asarray(err, float)
    true = np.asarray(true, float)

    m = np.isfinite(err) & np.isfinite(true) & (true != 0)
    rel_pct = (err[m] / true[m]) * 100.0  # percent

    rel_plot, lo, hi, n_total, n_kept = clip_for_hist(rel_pct, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=FIG_STANDARD)
    if rel_plot.size:
        ax.hist(rel_plot, bins=20, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title("Relative Diameter Error Distribution")
    ax.set_xlabel("Relative error (%)")
    ax.set_ylabel("Count")
    ax.axvline(0, linestyle="--", linewidth=2)

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(rel_pct):.2f} %" if n_total else "Median = NA",
        f"Mean = {np.mean(rel_pct):.2f} %" if n_total else "Mean = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.2f}%, {hi:.2f}%]")

    add_statbox(ax, lines)
    savefig(out_png)



# ============================================================
# Mask IoU (semantic)
# ============================================================
def compute_iou(mask_pred, mask_true):
    inter = np.logical_and(mask_pred, mask_true).sum()
    union = np.logical_or(mask_pred, mask_true).sum()
    return inter / union if union > 0 else np.nan

def _read_mask_frame(cap):
    ok, frame = cap.read()
    if not ok:
        return None
    if frame.ndim == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return frame

def _binarize_mask(img_gray, thresh=128):
    if img_gray is None:
        return None
    return (img_gray >= thresh)

def compute_iou_video_semantic(gt_mask_mp4, pred_mask_mp4, out_csv,
                               thresh=128, max_frames=None,
                               crop_top=0, crop_bottom=0):

    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read mask videos.")

    if not (gt_mask_mp4 and os.path.exists(gt_mask_mp4)):
        return (np.nan, np.nan, 0)
    if not (pred_mask_mp4 and os.path.exists(pred_mask_mp4)):
        return (np.nan, np.nan, 0)

    cap_gt = cv2.VideoCapture(gt_mask_mp4)
    cap_pr = cv2.VideoCapture(pred_mask_mp4)

    rows = []
    k = 0
    while True:
        gt_gray = _read_mask_frame(cap_gt)
        pr_gray = _read_mask_frame(cap_pr)
        if gt_gray is None or pr_gray is None:
            break

        if pr_gray.shape != gt_gray.shape:
            pr_gray = cv2.resize(pr_gray, (gt_gray.shape[1], gt_gray.shape[0]), interpolation=cv2.INTER_NEAREST)

        if crop_top or crop_bottom:
            H = gt_gray.shape[0]
            y0 = int(crop_top)
            y1 = int(H - crop_bottom) if crop_bottom else H
            gt_gray = gt_gray[y0:y1, :]
            pr_gray = pr_gray[y0:y1, :]

        gt_bin = _binarize_mask(gt_gray, thresh=thresh)
        pr_bin = _binarize_mask(pr_gray, thresh=thresh)

        iou = compute_iou(pr_bin, gt_bin)
        rows.append({"frame_idx": k, "iou": float(iou) if np.isfinite(iou) else np.nan})

        k += 1
        if max_frames is not None and k >= int(max_frames):
            break

    cap_gt.release()
    cap_pr.release()

    df_iou = pd.DataFrame(rows)
    ensure_dir(os.path.dirname(out_csv))
    df_iou.to_csv(out_csv, index=False)

    v = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    v = v[np.isfinite(v)]
    return (float(np.mean(v)) if v.size else np.nan,
            float(np.median(v)) if v.size else np.nan,
            int(v.size))

def plot_iou_vs_time(df_iou, out_png, fps=None):
    set_thesis_style()
    if df_iou is None or df_iou.empty or "iou" not in df_iou.columns:
        fig, ax = plt.subplots(figsize=(7.8, 5.2))
        ax.text(0.5, 0.5, "No IoU data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    x = pd.to_numeric(df_iou.get("frame_idx", df_iou.index), errors="coerce").to_numpy(float)
    y = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if fps and fps > 0:
        x_plot = x / fps
        xlab = "Time (s)"
    else:
        x_plot = x
        xlab = "Frame"

    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    if len(x_plot):
        ax.plot(x_plot, y, linewidth=1.8)
    ax.grid(True)
    ax.set_title("Segmentation IoU vs Time/Frame")
    ax.set_xlabel(xlab)
    ax.set_ylabel("IoU")

    yy = y[np.isfinite(y)]
    if yy.size: 
        q1, q3 = np.percentile(yy, [25, 75])
        add_statbox(ax, [
            f"N = {yy.size}",
            f"Mean = {np.mean(yy):.3f}",
            f"Median = {np.median(yy):.3f}",
            f"IQR = {q3-q1:.3f}",
            f"Min/Max = {np.min(yy):.3f} / {np.max(yy):.3f}",
        ],)

    savefig(out_png)

def plot_iou_histogram(df_iou, out_png, bins=20):
    set_thesis_style()
    if df_iou is None or df_iou.empty or "iou" not in df_iou.columns:
        fig, ax = plt.subplots(figsize=(7.8, 5.2))
        ax.text(0.5, 0.5, "No IoU data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    y = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    y = y[np.isfinite(y)]

    fig, ax = plt.subplots(figsize=(7.8, 5.2))
    if y.size:
        ax.hist(y, bins=bins, edgecolor="black", linewidth=0.6)
        ax.axvline(np.mean(y), linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax.axvline(np.median(y), linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax.legend(loc="lower left", framealpha=1.0)

    ax.grid(True)
    ax.set_title("Segmentation IoU Distribution")
    ax.set_xlabel("IoU")
    ax.set_ylabel("Count")

    ax.set_xlim(0.0, 1.0)

    savefig(out_png)


# ============================================================
# MOT metrics 
# ============================================================
def compute_mot_metrics(df_gt, df_m, max_dist_px):
    df_gt = df_gt.copy()
    df_m  = df_m.copy()
    df_gt["frame"] = pd.to_numeric(df_gt["frame"], errors="coerce").astype("Int64")
    df_m["frame"]  = pd.to_numeric(df_m["frame"], errors="coerce").astype("Int64")
    df_gt["gt_id"] = df_gt["gt_id"].astype(str)
    df_m["id"]     = df_m["id"].astype(str)

    frames = sorted(set(df_gt["frame"].dropna().astype(int)) | set(df_m["frame"].dropna().astype(int)))

    FP = 0
    FN = 0
    IDSW = 0
    dist_sum = 0.0
    match_count = 0
    gt_total = 0

    last_match_for_gt = {}
    pair_hits = {}
    gt_hits = {}
    trk_hits = {}

    for fr in frames:
        gtf = df_gt[df_gt["frame"].astype(int) == fr].dropna(subset=["gt_cx_px","gt_cy_px","gt_id"])
        mf  = df_m[df_m["frame"].astype(int) == fr].dropna(subset=["cx","cy","id"])

        gt_total += len(gtf)

        if len(gtf) == 0 and len(mf) == 0:
            continue
        if len(gtf) == 0:
            FP += len(mf)
            continue
        if len(mf) == 0:
            FN += len(gtf)
            continue

        gtf2 = pd.DataFrame({
            "gt_cx_px": pd.to_numeric(gtf["gt_cx_px"], errors="coerce"),
            "gt_cy_px": pd.to_numeric(gtf["gt_cy_px"], errors="coerce"),
        })
        mf2 = pd.DataFrame({
            "cx": pd.to_numeric(mf["cx"], errors="coerce"),
            "cy": pd.to_numeric(mf["cy"], errors="coerce"),
        })

        pairs = match_frame(gtf2, mf2, max_dist_px=max_dist_px)

        matched_gt_idx = set()
        matched_m_idx  = set()

        for gi, mi, dist in pairs:
            matched_gt_idx.add(gi)
            matched_m_idx.add(mi)

            dist_sum += dist
            match_count += 1

            gt_id = str(gtf.iloc[gi]["gt_id"])
            trk_id = str(mf.iloc[mi]["id"])

            if gt_id in last_match_for_gt and last_match_for_gt[gt_id] != trk_id:
                IDSW += 1
            last_match_for_gt[gt_id] = trk_id

            pair_hits[(gt_id, trk_id)] = pair_hits.get((gt_id, trk_id), 0) + 1
            gt_hits[gt_id] = gt_hits.get(gt_id, 0) + 1
            trk_hits[trk_id] = trk_hits.get(trk_id, 0) + 1

        FN += (len(gtf) - len(matched_gt_idx))
        FP += (len(mf)  - len(matched_m_idx))

    MOTP = (dist_sum / match_count) if match_count > 0 else np.nan
    MOTA = 1.0 - (FN + FP + IDSW) / gt_total if gt_total > 0 else np.nan

    gt_ids = list(gt_hits.keys())
    trk_ids = list(trk_hits.keys())

    if len(gt_ids) == 0 or len(trk_ids) == 0:
        IDF1 = np.nan
        IDTP = 0
        IDFP = FP
        IDFN = FN
    else:
        H = np.zeros((len(gt_ids), len(trk_ids)), dtype=float)
        gt_index = {g:i for i,g in enumerate(gt_ids)}
        tr_index = {t:j for j,t in enumerate(trk_ids)}
        for (g, t), cnt in pair_hits.items():
            if g in gt_index and t in tr_index:
                H[gt_index[g], tr_index[t]] = cnt

        r, c = linear_sum_assignment(-H)

        IDTP = float(H[r, c].sum())
        total_trk_matched = float(sum(trk_hits.values()))
        total_gt_matched  = float(sum(gt_hits.values()))

        IDFP = total_trk_matched - IDTP
        IDFN = total_gt_matched  - IDTP

        IDF1 = (2*IDTP) / (2*IDTP + IDFP + IDFN) if (2*IDTP + IDFP + IDFN) > 0 else np.nan

    return {
        "MOTA": float(MOTA) if np.isfinite(MOTA) else np.nan,
        "MOTP_px": float(MOTP) if np.isfinite(MOTP) else np.nan,
        "IDF1": float(IDF1) if np.isfinite(IDF1) else np.nan,
        "FP": int(FP),
        "FN": int(FN),
        "IDSW": int(IDSW),
        "GT_total": int(gt_total),
        "matches": int(match_count),
    }

def compare_manual_auto(df_manual, df_auto):
    err_d = df_auto["diam"] - df_manual["diam"]
    err_v = df_auto["vel"] - df_manual["vel"]

    return {
        "diam_mean_err": np.mean(err_d),
        "diam_std_err": np.std(err_d, ddof=1),
        "diam_rmse": rmse(err_d),
        "vel_mean_err": np.mean(err_v),
        "vel_std_err": np.std(err_v, ddof=1),
        "vel_rmse": rmse(err_v),
        "N": len(err_d)
    }

def _try_manual_crosscheck(manual_csv, df_auto, out_dir, units_len, units_vel):
    if not manual_csv or (not os.path.exists(manual_csv)):
        return

    try:
        df_m = pd.read_csv(manual_csv)
    except Exception as e:
        log_error(out_dir, f"[MANUAL READ FAIL] {type(e).__name__}: {e}")
        return

    fcol = pick_col(df_m, ["frame", "Frame"])
    dcol = pick_col(df_m, ["diam", "diam_in", "diam_px", "D_manual", "D"])
    vcol = pick_col(df_m, ["vel", "vel_in_s", "vel_px_s", "v_manual", "v"])

    if fcol is None or dcol is None or vcol is None:
        log_error(out_dir, f"[MANUAL SKIP] manual csv missing required cols. Found: {list(df_m.columns)}")
        return

    df_m = df_m.copy()
    df_m["frame"] = pd.to_numeric(df_m[fcol], errors="coerce").astype("Int64")
    df_m["diam"]  = pd.to_numeric(df_m[dcol], errors="coerce")
    df_m["vel"]   = pd.to_numeric(df_m[vcol], errors="coerce")

    if "frame" not in df_auto.columns or "diam_cmp" not in df_auto.columns or "vel_cmp" not in df_auto.columns:
        log_error(out_dir, "[MANUAL SKIP] auto dataframe missing diam_cmp/vel_cmp.")
        return

    ma = df_m.groupby("frame")[["diam", "vel"]].mean().reset_index()
    au = df_auto.groupby("frame")[["diam_cmp", "vel_cmp"]].mean().reset_index()

    df_join = pd.merge(ma, au, on="frame", how="inner")
    if df_join.empty:
        log_error(out_dir, "[MANUAL SKIP] no overlapping frames between manual and auto.")
        return

    df_manual2 = pd.DataFrame({
        "diam": df_join["diam"].to_numpy(float),
        "vel":  df_join["vel"].to_numpy(float),
    })
    df_auto2 = pd.DataFrame({
        "diam": df_join["diam_cmp"].to_numpy(float),
        "vel":  df_join["vel_cmp"].to_numpy(float),
    })

    summary = compare_manual_auto(df_manual2, df_auto2)
    tabs = ensure_dir(os.path.join(out_dir, "tables"))
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "manual_vs_auto_summary.csv"), index=False)
    df_join.to_csv(os.path.join(tabs, "manual_vs_auto_joined_by_frame.csv"), index=False)

# ============================================================
# Core evaluation: NO GT 
# ============================================================
def evaluate_no_gt(tracked_csv, out_dir, avg_csv=None, frames_dir=None, manual_csv=None):
    ensure_dir(out_dir)
    figs = ensure_dir(os.path.join(out_dir, "figs"))
    tabs = ensure_dir(os.path.join(out_dir, "tables"))

    df = pd.read_csv(tracked_csv)
    m = map_tracked(df)

    df["frame"] = pd.to_numeric(df[m["frame"]], errors="coerce").astype("Int64")
    df["id"]    = df[m["id"]].astype(str)
    df["cx"]    = pd.to_numeric(df[m["cx"]], errors="coerce")
    df["cy"]    = pd.to_numeric(df[m["cy"]], errors="coerce")
    df["diam_px"] = pd.to_numeric(df[m["diam"]], errors="coerce")

    if m["spd"]:
        df["vel_px_s"] = pd.to_numeric(df[m["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df[m["vx"]], errors="coerce") if m["vx"] else np.nan
        vy = pd.to_numeric(df[m["vy"]], errors="coerce") if m["vy"] else np.nan
        df["vel_px_s"] = np.hypot(vx, vy)


    settings, df_avg = (None, None)
    if avg_csv and os.path.exists(avg_csv):
        try:
            settings, df_avg = read_averages_with_settings(avg_csv)
        except Exception as e:
            log_error(out_dir, f"[AVG READ FAIL] {type(e).__name__}: {e}")
            settings, df_avg = None, None

    scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
    fps = infer_fps(settings=settings)

    ignore_top = infer_ignore_top_px(settings)
    bottom_bar = BOTTOM_BAR_PX
    
    stem = os.path.basename(tracked_csv).replace("_tracked.csv", "")
    base = os.path.dirname(tracked_csv)
    pred_mask_mp4 = os.path.join(base, stem + "_pred_mask.mp4")
    filtered_mp4  = os.path.join(base, stem + "_filtered.mp4")
    raw_mp4       = _find_raw_video(base, stem)
    
    frame_w, frame_h = (None, None)
    for vp in [pred_mask_mp4, filtered_mp4, raw_mp4]:
        fw, fh = infer_frame_wh_from_video(vp)
        if (fw is not None) and (fh is not None):
            frame_w, frame_h = fw, fh
            break

    
    units_len = "in" if scale_in_per_px else "px"
    units_vel = "in/s" if scale_in_per_px else "px/s"
    units_len_fname = fname_units(units_len)
    units_vel_fname = fname_units(units_vel)

    if "time_s" in df.columns and df["time_s"].notna().any():
        df["t_s"] = pd.to_numeric(df["time_s"], errors="coerce")
    elif fps and fps > 0 and df["frame"].notna().any():
        df["t_s"] = df["frame"].astype(float) / fps
    else:
        df["t_s"] = np.nan

    
    if scale_in_per_px:
        df["diam_in"]  = df["diam_px"] * scale_in_per_px
        df["vel_in_s"] = df["vel_px_s"] * scale_in_per_px
        df["x_in"]     = df["cx"] * scale_in_per_px
        df["y_in"]     = df["cy"] * scale_in_per_px

    track_len = df.groupby("id").size()
    pd.DataFrame({"id": track_len.index, "track_len_frames": track_len.values}) \
        .to_csv(os.path.join(tabs, "track_stats.csv"), index=False)

    
    diam_col = "diam_in" if scale_in_per_px else "diam_px"
    vel_col  = "vel_in_s" if scale_in_per_px else "vel_px_s"

    df_plot = drop_injection_zone_by_y(df,cy_col="cy", frac_from_bottom=0.15, frame_h=frame_h)
    
    # Per-bubble (per-track) standard deviations + stats
    per_bubble_cols = [diam_col, vel_col]
    
    if "cx" in df.columns: per_bubble_cols.append("cx")
    if "cy" in df.columns: per_bubble_cols.append("cy")
    if "t_s" in df.columns: per_bubble_cols.append("t_s")
    
    df_per_bubble = compute_per_bubble_stats(
        df,
        id_col="id",
        cols=per_bubble_cols,
        min_points=2
    )
    
    df_per_bubble.to_csv(os.path.join(tabs, "per_bubble_stats.csv"), index=False)
    
    per_track_means = df.groupby("id", sort=False).agg(
        N=(diam_col, "count"),
        mean_diam=(diam_col, "mean"),
        mean_vel=(vel_col, "mean")
    ).reset_index()
    
    # Save the table too (super helpful for debugging + thesis)
    per_track_means.to_csv(os.path.join(tabs, "per_track_mean_diam_vel.csv"), index=False)
    
    # Plot it
    safe_plot(
        out_dir,
        "fig_mean_vel_vs_mean_diam_per_track",
        os.path.join(figs, f"fig_mean_vel_vs_mean_diam_per_track_{units_len_fname}.png"),
        lambda: plot_mean_velocity_vs_mean_diameter_per_track(
            df_tracks=df,
            out_png=os.path.join(figs, f"fig_mean_vel_vs_mean_diam_per_track_{units_len_fname}.png"),
            units_len=units_len,
            units_vel=units_vel,
            diam_col=diam_col,
            vel_col=vel_col,
            min_points_per_track=20,
            do_regression=True
        )
    )
    
    std_diam_key = f"std_{diam_col}"
    std_vel_key  = f"std_{vel_col}"
    
    per_bubble_std_diam_median = float(df_per_bubble[std_diam_key].median()) if (not df_per_bubble.empty and std_diam_key in df_per_bubble.columns) else np.nan
    per_bubble_std_vel_median  = float(df_per_bubble[std_vel_key].median())  if (not df_per_bubble.empty and std_vel_key  in df_per_bubble.columns) else np.nan

    
    diam = pd.to_numeric(df_plot[diam_col], errors="coerce").to_numpy(float)
    vel  = pd.to_numeric(df_plot[vel_col],  errors="coerce").to_numpy(float)
    
    m = np.isfinite(diam) & np.isfinite(vel)
    diam = diam[m]
    vel  = vel[m]
    
    dens_summary, df_dens_pf = compute_bubble_density_per_run(
        df_det=df,
        frame_col="frame",
        cy_col="cy",
        ignore_top_px=ignore_top,
        bottom_bar_px=bottom_bar,
        frame_h=frame_h,
        frame_w=frame_w,
        scale_in_per_px=scale_in_per_px
    )
    
    df_dens_pf.to_csv(os.path.join(tabs, "bubble_density_per_frame.csv"), index=False)
    pd.DataFrame([dens_summary]).to_csv(os.path.join(tabs, "bubble_density_summary.csv"), index=False)

    # centroid uncertainty
    centroid_u_xy_px, centroid_u_xy_in = run_centroid_uncertainty(
        df, tabs, out_dir, fps, scale_in_per_px
    )

    diam_u_D_px, diam_u_D_in = run_diameter_uncertainty(
        df, tabs, out_dir, fps, scale_in_per_px
    )
    
    delta_r_med_px, delta_r_mean_px, n_steps, dr_steps = compute_delta_r_stats(
        df[["id", "frame", "cx", "cy"]].copy(),
        require_consecutive=True,
        max_step_px=100
    )
    plot_delta_r_histogram(dr_steps, os.path.join(figs, "fig_delta_r_histogram.png"))
    pd.DataFrame([{
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "require_consecutive": True,
        "max_step_px": 100,
        "note": "Δr is reported as motion statistic; not used in uncertainty propagation"
    }]).to_csv(os.path.join(tabs, "delta_r_summary.csv"), index=False)

    # --- Video-based qualitative panel (no frames_dir needed) ---
    stem = os.path.basename(tracked_csv).replace("_tracked.csv", "")
    base = os.path.dirname(tracked_csv)
    
    raw_mp4       = _find_raw_video(base, stem)
    pred_mask_mp4 = os.path.join(base, stem + "_pred_mask.mp4")
    filtered_mp4  = os.path.join(base, stem + "_filtered.mp4")
    
    # ---- choose start frame near the middle of tracked data ----
    if df["frame"].notna().any():
        fmin = int(df["frame"].dropna().min())
        fmax = int(df["frame"].dropna().max())
        start_frame = max(fmin, (fmin + fmax)//2 - 2)  # centers a 5-frame window
    else:
        start_frame = 0
    
    safe_plot(
        out_dir,
        "fig_qualitative_sequence",
        os.path.join(figs, "fig_qualitative_sequence.png"),
        lambda: make_qualitative_panel_from_videos(
            raw_mp4=raw_mp4,
            pred_mask_mp4=pred_mask_mp4,
            filtered_mp4=filtered_mp4,
            df_tracked=df,
            out_png=os.path.join(figs, "fig_qualitative_sequence.png"),
            start_frame=start_frame,
            n_frames=5,
            show_gt_row=False
        )
    )

    safe_plot(out_dir, "fig_diameter_hist",
              os.path.join(figs, f"fig_diameter_hist_{units_len_fname}.png"),
              lambda: plot_distribution(
                  diam, os.path.join(figs, f"fig_diameter_hist_{units_len_fname}.png"),
                  "Bubble Diameter Distribution (Measured)", "Equivalent diameter", units_len))

    safe_plot(out_dir, "fig_velocity_hist",
              os.path.join(figs, f"fig_velocity_hist_{units_vel_fname}.png"),
              lambda: plot_distribution(
                  vel, os.path.join(figs, f"fig_velocity_hist_{units_vel_fname}.png"),
                  "Bubble Rise Velocity Distribution (Measured)", "Rise velocity", units_vel))

    safe_plot(out_dir, "fig_track_lengths",
              os.path.join(figs, "fig_track_lengths.png"),
              lambda: plot_track_lengths(track_len.values, fps, os.path.join(figs, "fig_track_lengths.png")))

    safe_plot(out_dir, "fig_trajectories_2d_timecolored",
              os.path.join(figs, "fig_trajectories_2d_timecolored.png"),
              lambda: plot_trajectories_2d_timecolored(
                  df, os.path.join(figs, "fig_trajectories_2d_timecolored.png"), units_len))

    safe_plot(out_dir, "fig_velocity_vs_diameter",
              os.path.join(figs, f"fig_velocity_vs_diameter_{units_len_fname}.png"),
              lambda: plot_velocity_vs_diameter(
                  diam, vel, os.path.join(figs, f"fig_velocity_vs_diameter_{units_len_fname}.png"),
                  units_len, units_vel))

    safe_plot(out_dir, "fig_velocity_vs_time",
              os.path.join(figs, f"fig_velocity_vs_time_{units_vel_fname}.png"),
              lambda: plot_velocity_vs_time(
                  df, os.path.join(figs, f"fig_velocity_vs_time_{units_vel_fname}.png"),
                  units_vel))

    safe_plot(out_dir, "fig_velocity_smoothness",
              os.path.join(figs, f"fig_velocity_smoothness_{units_vel_fname}.png"),
              lambda: plot_velocity_smoothness(
                  df, os.path.join(figs, f"fig_velocity_smoothness_{units_vel_fname}.png"),
                  units_vel))
   
    safe_plot(out_dir, "fig_grace_avg_point",
          os.path.join(figs, "fig_grace_avg_point.png"),
          lambda: plot_grace_avg_point(df, os.path.join(figs, "fig_grace_avg_point.png"),
                                       df_avg=df_avg, label="Auto (no GT)", annotate=True,
                                       settings=settings, scale_in_per_px=scale_in_per_px))
    
    safe_plot(out_dir, "fig_velocity_vs_time_smoothed",
          os.path.join(figs, f"fig_velocity_vs_time_smoothed_{units_vel_fname}.png"),
          lambda: plot_velocity_vs_time_smoothed(
            df,
            os.path.join(figs, f"fig_velocity_vs_time_smoothed_{units_vel_fname}.png"),
            units_vel,
            median_window=15,
            mean_window=11,
            plot_every=3))


    if DEBUG_GRACE:
        safe_plot(out_dir, "fig_grace_debug_mapping",
                  os.path.join(figs, "fig_grace_debug_mapping.png"),
                  lambda: debug_grace_mapping(os.path.join(figs, "fig_grace_debug_mapping.png")))


    sd = stats_dict(diam)  
    sv = stats_dict(vel)

    summary = {
        "mode": "NO_GT",
        "N_tracks": int(track_len.size),
        "N_samples": int(np.isfinite(diam).sum()),
        "scale_in_per_px": scale_in_per_px if scale_in_per_px else "",
        "fps": fps if fps else "",
        f"diam_mean_{units_len}": sd["mean"],
        f"diam_median_{units_len}": sd["median"],
        f"diam_std_{units_len}": sd["std"],
        f"vel_mean_{units_vel}": sv["mean"],
        f"vel_median_{units_vel}": sv["median"],
        f"vel_std_{units_vel}": sv["std"],
        "mean_track_len_frames": float(track_len.mean()) if track_len.size else np.nan,
        "centroid_u_xy_px": centroid_u_xy_px,
        "centroid_u_xy_in": centroid_u_xy_in if scale_in_per_px else "",
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "units_len": units_len,
        "units_vel": units_vel,
        "diameter_u_D_px": diam_u_D_px,
        "diameter_u_D_in": diam_u_D_in if scale_in_per_px else "",
        "roi_area_in2": dens_summary.get("roi_area_in2", np.nan),
        "mean_bubbles_per_frame": dens_summary.get("mean_bubbles_per_frame", np.nan),
        "median_bubbles_per_frame": dens_summary.get("median_bubbles_per_frame", np.nan),
        "mean_density_bubbles_in2": dens_summary.get("mean_density_bubbles_in2", np.nan),
        "median_density_bubbles_in2": dens_summary.get("median_density_bubbles_in2", np.nan),
        "density_frames_used": dens_summary.get("N_frames_used", 0),

    }
    
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "run_summary.csv"), index=False)
    return summary


# ============================================================
# Core evaluation: WITH GT 
# ============================================================
def evaluate_with_gt(gt_csv, tracked_csv, out_dir, max_dist_px, avg_csv=None, frames_dir=None, manual_csv=None):
    ensure_dir(out_dir)
    figs = ensure_dir(os.path.join(out_dir, "figs"))
    tabs = ensure_dir(os.path.join(out_dir, "tables"))

    df_gt = pd.read_csv(gt_csv)
    df_m  = pd.read_csv(tracked_csv)

    gtm = map_gt(df_gt)
    trm = map_tracked(df_m)

    settings, df_avg = (None, None)
    if avg_csv and os.path.exists(avg_csv):
        try:
            settings, df_avg = read_averages_with_settings(avg_csv)
        except Exception as e:
            log_error(out_dir, f"[AVG READ FAIL] {type(e).__name__}: {e}")
            settings, df_avg = None, None

    ignore_top = infer_ignore_top_px(settings)
    bottom_bar = BOTTOM_BAR_PX

    scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
    fps = infer_fps(settings=settings)

    units_len = "in" if scale_in_per_px else "px"
    units_vel = "in/s" if scale_in_per_px else "px/s"
    units_len_fname = fname_units(units_len)
    units_vel_fname = fname_units(units_vel)

    # --- Build mask video paths early (used for frame_h + IoU) ---
    stem = os.path.basename(tracked_csv).replace("_tracked.csv", "")
    base = os.path.dirname(tracked_csv)

    gt_mask_mp4   = os.path.join(base, stem + "_gt_mask.mp4")
    pred_mask_mp4 = os.path.join(base, stem + "_pred_mask.mp4")  # binary masks (correct for IoU)

    # overlay video exists but is NOT used for IoU
    filtered_mp4  = os.path.join(base, stem + "_filtered.mp4")

    # --- Determine frame height BEFORE ROI filtering ---
    frame_w, frame_h = (None, None)
    if _HAS_CV2 and os.path.exists(gt_mask_mp4):
        cap0 = cv2.VideoCapture(gt_mask_mp4)
        ok0, fr0 = cap0.read()
        cap0.release()
        if ok0 and fr0 is not None:
            frame_h, frame_w = fr0.shape[0], fr0.shape[1]


    # --- Standardize GT dataframe ---
    df_gt["frame"] = pd.to_numeric(df_gt[gtm["frame"]], errors="coerce").astype("Int64")
    if "gt_id" not in df_gt.columns:
        raise ValueError(f"GT csv must include gt_id column. Found: {list(df_gt.columns)}")
    df_gt["gt_id"] = df_gt["gt_id"].astype(str)

    df_gt["gt_cx_px"] = pd.to_numeric(df_gt[gtm["cx"]], errors="coerce")
    df_gt["gt_cy_px"] = pd.to_numeric(df_gt[gtm["cy"]], errors="coerce")
    df_gt["gt_diam_px"] = pd.to_numeric(df_gt[gtm["diam"]], errors="coerce")
    if gtm["spd"]:
        df_gt["gt_speed_px_s"] = pd.to_numeric(df_gt[gtm["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df_gt[gtm["vx"]], errors="coerce") if gtm["vx"] else np.nan
        vy = pd.to_numeric(df_gt[gtm["vy"]], errors="coerce") if gtm["vy"] else np.nan
        df_gt["gt_speed_px_s"] = np.hypot(vx, vy)

    # --- Standardize tracked dataframe ---
    df_m["frame"] = pd.to_numeric(df_m[trm["frame"]], errors="coerce").astype("Int64")
    df_m["id"]    = df_m[trm["id"]].astype(str)
    df_m["cx"]    = pd.to_numeric(df_m[trm["cx"]], errors="coerce")
    df_m["cy"]    = pd.to_numeric(df_m[trm["cy"]], errors="coerce")
    df_m["diam_px"] = pd.to_numeric(df_m[trm["diam"]], errors="coerce")
    if trm["spd"]:
        df_m["meas_speed_px_s"] = pd.to_numeric(df_m[trm["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df_m[trm["vx"]], errors="coerce") if trm["vx"] else np.nan
        vy = pd.to_numeric(df_m[trm["vy"]], errors="coerce") if trm["vy"] else np.nan
        df_m["meas_speed_px_s"] = np.hypot(vx, vy)




    if fps and fps > 0 and df_m["frame"].notna().any():
        df_m["t_s"] = df_m["frame"].astype(float) / fps
    else:
        df_m["t_s"] = np.nan

    # --- Apply ROI to BOTH GT and tracked ---
    df_gt = apply_roi_filter(df_gt, "gt_cy_px", ignore_top_px=ignore_top, bottom_bar_px=bottom_bar, frame_h=frame_h)
    df_m  = apply_roi_filter(df_m,  "cy",      ignore_top_px=ignore_top, bottom_bar_px=bottom_bar, frame_h=frame_h)

    # Bubble density (bubbles/in^2) using tracked detections
    dens_summary, df_dens_pf = compute_bubble_density_per_run(
        df_det=df_m,
        frame_col="frame",
        cy_col="cy",
        ignore_top_px=ignore_top,
        bottom_bar_px=bottom_bar,
        frame_h=frame_h,
        frame_w=frame_w,
        scale_in_per_px=scale_in_per_px
    )
    
    df_dens_pf.to_csv(os.path.join(tabs, "bubble_density_per_frame.csv"), index=False)
    pd.DataFrame([dens_summary]).to_csv(os.path.join(tabs, "bubble_density_summary.csv"), index=False)

    
    # --- Per-frame matching ---
    rows = []
    common_frames = sorted(set(df_gt["frame"].dropna().astype(int)) & set(df_m["frame"].dropna().astype(int)))
    for fr in common_frames:
        gtf = df_gt[df_gt["frame"].astype(int) == fr].dropna(subset=["gt_cx_px", "gt_cy_px", "gt_diam_px"])
        mf  = df_m[df_m["frame"].astype(int) == fr].dropna(subset=["cx", "cy", "diam_px"])
        if len(gtf) == 0 or len(mf) == 0:
            continue

        pairs = match_frame(gtf, mf, max_dist_px=max_dist_px)
        for gi, mi, dist in pairs:
            g = gtf.iloc[gi]
            mrow = mf.iloc[mi] 
            ms= mrow["meas_speed_px_s"]
            gs = g["gt_speed_px_s"]
            rows.append({
                "frame": fr,
                "match_dist_px": dist,
                "gt_cy_px": g["gt_cy_px"],          
                "meas_cy_px": mrow["cy"],
                "gt_diam_px": g["gt_diam_px"],
                "meas_diam_px": mrow["diam_px"],
                "gt_speed_px_s": g["gt_speed_px_s"],
                "meas_speed_px_s": ms,
                "err_speed_px_s": (ms - gs) if np.isfinite(ms) and np.isfinite(gs) else np.nan,
                "err_diam_px": mrow["diam_px"] - g["gt_diam_px"],
            })

    df_match = pd.DataFrame(rows)
    df_match.to_csv(os.path.join(tabs, "matched_pairs.csv"), index=False)

    # --- Keep a clean speed-valid subset for speed metrics/KS2 ---
    df_match_speed = df_match[np.isfinite(df_match["gt_speed_px_s"]) & np.isfinite(df_match["meas_speed_px_s"])].copy()
    
    # --- Tracking metrics (MOTA/MOTP/IDF1) ---
    mot = compute_mot_metrics(
        df_gt[["frame","gt_id","gt_cx_px","gt_cy_px"]],
        df_m[["frame","id","cx","cy"]],
        max_dist_px=max_dist_px
    )
    pd.DataFrame([mot]).to_csv(os.path.join(tabs, "tracking_metrics.csv"), index=False)

    # centroid uncertainty (Eq. 3.17) on the tracked data
    centroid_u_xy_px, centroid_u_xy_in = run_centroid_uncertainty(
        df_m, tabs, out_dir, fps, scale_in_per_px
    )
        
    # --- Centroid contribution to velocity uncertainty (raw, no smoothing) ---
    df_v = df_m.copy()
    
    # Force the velocity column we want to use (measured speed) into a single clean Series
    if "meas_speed_px_s" in df_v.columns:
        df_v["vel_px_s"] = pd.to_numeric(df_v["meas_speed_px_s"], errors="coerce")
    elif "vel_px_s" in df_v.columns:
        df_v["vel_px_s"] = pd.to_numeric(df_v["vel_px_s"], errors="coerce")
    else:
        df_v["vel_px_s"] = np.nan
    
    # Drop duplicate columns by name (keeps the first occurrence)
    df_v = df_v.loc[:, ~df_v.columns.duplicated()]
    
    delta_r_med_px, delta_r_mean_px, n_steps, dr_steps = compute_delta_r_stats(
    df_m[["id", "frame", "cx", "cy"]].copy(),
    require_consecutive=True,
    max_step_px=100
    )

    plot_delta_r_histogram(
        dr_steps,
        os.path.join(figs, "fig_delta_r_histogram.png")
    )

    pd.DataFrame([{
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "require_consecutive": True,
        "max_step_px": 100
    }]).to_csv(os.path.join(tabs, "delta_r_summary.csv"), index=False)
    
    # --- IoU validation (semantic masks) ---
    if not os.path.exists(gt_mask_mp4):
        log_error(out_dir, f"[IOU SKIP] Missing GT mask video: {gt_mask_mp4}")
    if not os.path.exists(pred_mask_mp4):
        log_error(out_dir, f"[IOU SKIP] Missing PRED mask video: {pred_mask_mp4}")
        if os.path.exists(filtered_mp4):
            log_error(out_dir, f"[NOTE] Found overlay video (not valid for IoU): {filtered_mp4}")

    mean_iou, median_iou, n_iou = compute_iou_video_semantic(
        gt_mask_mp4,
        pred_mask_mp4,
        out_csv=os.path.join(tabs, "iou_per_frame.csv"),
        thresh=128,
        crop_top=ignore_top,
        crop_bottom=bottom_bar
    )

    diam_col = "diam_px"
    vel_col  = "vel_px_s" 
    
    df_per_bubble = compute_per_bubble_stats(
        df_m,
        id_col="id",
        cols=[diam_col, vel_col, "cx", "cy", "t_s"],
        min_points=2
    )
    df_per_bubble.to_csv(os.path.join(tabs, "per_bubble_stats.csv"), index=False)

    
    df_iou = None
    iou_csv = os.path.join(tabs, "iou_per_frame.csv")
    if os.path.exists(iou_csv):
        try:
            df_iou = pd.read_csv(iou_csv)
        except Exception as e:
            log_error(out_dir, f"[IOU CSV READ FAIL] {type(e).__name__}: {e}")

    safe_plot(out_dir, "fig_iou_vs_time",
              os.path.join(figs, "fig_iou_vs_time.png"),
              lambda: plot_iou_vs_time(df_iou, os.path.join(figs, "fig_iou_vs_time.png"), fps=fps))

    safe_plot(out_dir, "fig_iou_histogram",
              os.path.join(figs, "fig_iou_histogram.png"),
              lambda: plot_iou_histogram(df_iou, os.path.join(figs, "fig_iou_histogram.png"), bins=20))

    # --- Convert match units if scale exists ---
    if scale_in_per_px and (not df_match.empty):
        df_match["gt_diam_in"] = df_match["gt_diam_px"] * scale_in_per_px
        df_match["meas_diam_in"] = df_match["meas_diam_px"] * scale_in_per_px
        df_match["err_diam_in"] = df_match["err_diam_px"] * scale_in_per_px

        df_match["gt_speed_in_s"] = df_match["gt_speed_px_s"] * scale_in_per_px
        df_match["meas_speed_in_s"] = df_match["meas_speed_px_s"] * scale_in_per_px
        df_match["err_speed_in_s"] = df_match["err_speed_px_s"] * scale_in_per_px

    if scale_in_per_px and (not df_match.empty):
        true_d = df_match["gt_diam_in"].values
        meas_d = df_match["meas_diam_in"].values
        err_d  = df_match["err_diam_in"].values
        true_v = (df_match_speed["gt_speed_px_s"] * scale_in_per_px).values
        meas_v = (df_match_speed["meas_speed_px_s"] * scale_in_per_px).values
        err_v  = ((df_match_speed["meas_speed_px_s"] - df_match_speed["gt_speed_px_s"]) * scale_in_per_px).values
    else:
        true_d = df_match["gt_diam_px"].values if not df_match.empty else np.array([])
        meas_d = df_match["meas_diam_px"].values if not df_match.empty else np.array([])
        err_d  = df_match["err_diam_px"].values if not df_match.empty else np.array([])
        true_v = df_match_speed["gt_speed_px_s"].values if not df_match_speed.empty else np.array([])
        meas_v = df_match_speed["meas_speed_px_s"].values if not df_match_speed.empty else np.array([])
        err_v  = df_match_speed["err_speed_px_s"].values if not df_match_speed.empty else np.array([])

    # Defaults so summary never NameErrors even if something fails below
    sum_err_d = np.nan
    sum_err_v = np.nan

    mu_gt_d = sd_gt_d = mu_me_d = sd_me_d = np.nan
    mu_gt_v = sd_gt_v = mu_me_v = sd_me_v = np.nan

    # ============================================================
    # Error sum check + Excel-style standardization ("standard normal")
    # ============================================================
    def _clean1d(x):
        x = np.asarray(x, dtype=float).ravel()
        return x[np.isfinite(x)]

    def _mean_std(x):
        x = _clean1d(x)
        if x.size < 2:
            return (np.nan, np.nan, int(x.size))
        return (float(np.mean(x)), float(np.std(x, ddof=1)), int(x.size))

    # Signed errors (meas - gt) and "add them up"
    err_d_c = _clean1d(err_d)
    err_v_c = _clean1d(err_v)

    sum_err_d = float(np.sum(err_d_c)) if err_d_c.size else np.nan
    sum_err_v = float(np.sum(err_v_c)) if err_v_c.size else np.nan

    # Means/std devs for GT and Measured (needed for STANDARDIZE)
    mu_gt_d, sd_gt_d, n_gt_d = _mean_std(true_d)
    mu_me_d, sd_me_d, n_me_d = _mean_std(meas_d)

    mu_gt_v, sd_gt_v, n_gt_v = _mean_std(true_v)
    mu_me_v, sd_me_v, n_me_v = _mean_std(meas_v)

    # Excel STANDARDIZE(x, mean, std_dev) == (x - mean) / std_dev
    def _standardize(x, mu, sd):
        x = _clean1d(x)
        if not np.isfinite(mu) or not np.isfinite(sd) or sd <= 0:
            return np.full(x.shape, np.nan, dtype=float)
        return (x - mu) / sd

    z_gt_d   = _standardize(true_d, mu_gt_d, sd_gt_d)
    z_meas_d = _standardize(meas_d, mu_me_d, sd_me_d)

    z_gt_v   = _standardize(true_v, mu_gt_v, sd_gt_v)
    z_meas_v = _standardize(meas_v, mu_me_v, sd_me_v)

    # Save outputs
    pd.DataFrame([{
        "metric": f"diam_{units_len}",
        "sum_error(meas-gt)": sum_err_d,
        "mu_gt": mu_gt_d, "sd_gt": sd_gt_d, "N_gt": n_gt_d,
        "mu_meas": mu_me_d, "sd_meas": sd_me_d, "N_meas": n_me_d,
    },{
        "metric": f"speed_{units_vel}",
        "sum_error(meas-gt)": sum_err_v,
        "mu_gt": mu_gt_v, "sd_gt": sd_gt_v, "N_gt": n_gt_v,
        "mu_meas": mu_me_v, "sd_meas": sd_me_v, "N_meas": n_me_v,
    }]).to_csv(os.path.join(tabs, "standardize_summary.csv"), index=False)

    # Save z-scores separately because diam and speed can have different N
    pd.DataFrame({
        "z_gt_diam": z_gt_d,
        "z_meas_diam": z_meas_d,
    }).to_csv(os.path.join(tabs, "standardize_values_diam.csv"), index=False)

    pd.DataFrame({
        "z_gt_speed": z_gt_v,
        "z_meas_speed": z_meas_v,
    }).to_csv(os.path.join(tabs, "standardize_values_speed.csv"), index=False)

    # DEBUG Print
    if true_d.size > 0 and meas_d.size > 0:
        print("\n--- Diameter stats ---")
        print("GT min/max:", np.min(true_d), np.max(true_d))
        print("MEAS min/max:", np.min(meas_d), np.max(meas_d))
    
    if true_v.size > 0 and meas_v.size > 0:
        print("\n--- Speed stats ---")
        print("GT min/max:", np.min(true_v), np.max(true_v))
        print("MEAS min/max:", np.min(meas_v), np.max(meas_v))
    

    cy_color = df_match["meas_cy_px"].values if not df_match.empty else None

    # --- KS2 distribution checks (GT vs Measured) ---
    ks_D_diam, ks_p_diam, ks_reject_diam, ks_n_true_d, ks_n_meas_d = ks2_test(true_d, meas_d, alpha=0.05)
    ks_D_speed, ks_p_speed, ks_reject_speed, ks_n_true_v, ks_n_meas_v = ks2_test(true_v, meas_v, alpha=0.05)
    
    # Save to a small CSV (optional but nice for records)
    pd.DataFrame([{
        "metric": f"diam_{units_len}",
        "KS_D": ks_D_diam,
        "p_value": ks_p_diam,
        "reject_same_dist_p<0.05": ks_reject_diam,
        "N_true": ks_n_true_d,
        "N_measured": ks_n_meas_d,
    },{
        "metric": f"speed_{units_vel}",
        "KS_D": ks_D_speed,
        "p_value": ks_p_speed,
        "reject_same_dist_p<0.05": ks_reject_speed,
        "N_true": ks_n_true_v,
        "N_measured": ks_n_meas_v,
    }]).to_csv(os.path.join(tabs, "ks2_results.csv"), index=False)
    
    safe_plot(out_dir, "fig_measured_vs_true_diameter",
              os.path.join(figs, f"fig_measured_vs_true_diameter_{units_len_fname}.png"),
              lambda: plot_measured_vs_true(
                  true_d, meas_d,
                  os.path.join(figs, f"fig_measured_vs_true_diameter_{units_len_fname}.png"),
                  "Measured vs True Diameter (GT Validation)", units_len,
                  y_pos_px=cy_color,
                  y_pos_label="Vertical Centroid (px)"
              ))
    
    safe_plot(out_dir, "fig_measured_vs_true_speed",
              os.path.join(figs, f"fig_measured_vs_true_speed_{units_vel_fname}.png"),
              lambda: plot_measured_vs_true(
                  true_v, meas_v,
                  os.path.join(figs, f"fig_measured_vs_true_speed_{units_vel_fname}.png"),
                  "Measured vs True Speed (GT Validation)", units_vel,
                  y_pos_px=cy_color,
                  y_pos_label="Vertical Centroid (px)"
              ))



    safe_plot(out_dir, "fig_error_hist_diameter",
              os.path.join(figs, f"fig_error_hist_diameter_{units_len_fname}.png"),
              lambda: plot_distribution_zoom_full(
                  err_d, os.path.join(figs, f"fig_error_hist_diameter_{units_len_fname}.png"),
                  "Diameter Error Distribution (Synthetic Validation)", "Error", units_len))

    safe_plot(out_dir, "fig_error_hist_speed",
              os.path.join(figs, f"fig_error_hist_speed_{units_vel_fname}.png"),
              lambda: plot_distribution_zoom_full(
                  err_v, os.path.join(figs, f"fig_error_hist_speed_{units_vel_fname}.png"),
                  "Speed Error Distribution (Synthetic Validation)", "Error", units_vel))

    safe_plot(out_dir, "fig_relative_error_diameter",
              os.path.join(figs, f"fig_relative_error_diameter.png"),
              lambda: plot_relative_error(
                  err_d, true_d,
                  os.path.join(figs, f"fig_relative_error_diameter.png")))

    safe_plot(out_dir, "fig_grace_avg_point",
          os.path.join(figs, "fig_grace_avg_point.png"),
          lambda: plot_grace_avg_point(df_m, os.path.join(figs, "fig_grace_avg_point.png"),
                                       df_avg=df_avg, label="Auto (GT runs)", annotate=True,
                                       settings=settings, scale_in_per_px=scale_in_per_px))

    
    if DEBUG_GRACE:
        safe_plot(out_dir, "fig_grace_debug_mapping",
                  os.path.join(figs, "fig_grace_debug_mapping.png"),
                  lambda: debug_grace_mapping(os.path.join(figs, "fig_grace_debug_mapping.png")))



    # Time-colored error vs time/frame
    if not df_match.empty:
        if fps and fps > 0:
            x = df_match["frame"].values / fps
            xlab = "Time (s)"
        else:
            x = df_match["frame"].values
            xlab = "Frame"
    else:
        x = np.array([])
        xlab = "Frame"

    safe_plot(out_dir, "fig_error_vs_time_diameter",
              os.path.join(figs, f"fig_error_vs_time_diameter_{units_len_fname}.png"),
              lambda: plot_error_vs_time_timecolored(
                  x, err_d, os.path.join(figs, f"fig_error_vs_time_diameter_{units_len_fname}.png"),
                  "Diameter Error vs Time/Frame (Synthetic)", xlab, units_len))

    safe_plot(out_dir, "fig_error_vs_time_speed",
              os.path.join(figs, f"fig_error_vs_time_speed_{units_vel_fname}.png"),
              lambda: plot_error_vs_time_timecolored(
                  x, err_v, os.path.join(figs, f"fig_error_vs_time_speed_{units_vel_fname}.png"),
                  "Speed Error vs Time/Frame (Synthetic)", xlab, units_vel))

    safe_plot(out_dir, "fig_trajectories_2d_timecolored",
              os.path.join(figs, "fig_trajectories_2d_timecolored.png"),
              lambda: plot_trajectories_2d_timecolored(
                  df_m, os.path.join(figs, "fig_trajectories_2d_timecolored.png"), units_len))

    # --- Video-based qualitative panel (raw + pred mask + filtered + IDs; optionally GT mask) ---
    raw_mp4 = _find_raw_video(base, stem)
    if df_m["frame"].notna().any():
        fmin = int(df_m["frame"].dropna().min())
        fmax = int(df_m["frame"].dropna().max())
        start_frame = max(fmin, (fmin + fmax)//2 - 2)  # centers a 5-frame window
    else:
        start_frame = 0
    
    safe_plot(out_dir, "fig_qualitative_sequence",
              os.path.join(figs, "fig_qualitative_sequence.png"),
              lambda: make_qualitative_panel_from_videos(
                  raw_mp4=raw_mp4,
                  pred_mask_mp4=pred_mask_mp4,
                  filtered_mp4=filtered_mp4,
                  gt_mask_mp4=gt_mask_mp4,
                  show_gt_row=True,  # set False if you don’t want the GT row
                  df_tracked=df_m,
                  out_png=os.path.join(figs, "fig_qualitative_sequence.png"),
                  start_frame=start_frame,#int(df_m["frame"].dropna().min()) if df_m["frame"].notna().any() else 0,
                  n_frames=5
              ))

    # --- GT vs Pred mask difference panel (white=GT, red=over, blue=under) ---
    safe_plot(out_dir, "fig_mask_diff_sequence",
              os.path.join(figs, "fig_mask_diff_sequence.png"),
              lambda: make_mask_diff_panel_from_videos(
                  gt_mask_mp4=gt_mask_mp4,
                  pred_mask_mp4=pred_mask_mp4,
                  out_png=os.path.join(figs, "fig_mask_diff_sequence.png"),
                  start_frame=start_frame,
                  n_frames=5,
                  thresh=128,
                  crop_top=ignore_top,
                  crop_bottom=bottom_bar
              ))


    def stdev(x):
        x = _clean(x)
        return float(np.std(x, ddof=1)) if x.size > 1 else np.nan

    summary = {
        "mode": "GT",
        "N_matches": int(np.isfinite(err_d).sum()) if err_d.size else 0,
        "scale_in_per_px": scale_in_per_px if scale_in_per_px else "",
        "fps": fps if fps else "",
        f"diam_MAE_{units_len}": mae(err_d),
        f"diam_RMSE_{units_len}": rmse(err_d),
        f"diam_Bias_{units_len}": bias(err_d),
        f"diam_Std_{units_len}": stdev(err_d),
        f"speed_MAE_{units_vel}": mae(err_v),
        f"speed_RMSE_{units_vel}": rmse(err_v),
        f"speed_Bias_{units_vel}": bias(err_v),
        f"speed_Std_{units_vel}": stdev(err_v),
        "IoU_mean": mean_iou,
        "IoU_median": median_iou,
        "IoU_N_frames": n_iou,
        "MOTA": mot["MOTA"],
        "MOTP_px": mot["MOTP_px"],
        "IDF1": mot["IDF1"],
        "FP": mot["FP"],
        "FN": mot["FN"],
        "IDSW": mot["IDSW"],
        "centroid_u_xy_px": centroid_u_xy_px,
        "centroid_u_xy_in": centroid_u_xy_in if scale_in_per_px else "",
        "delta_r_median_px": delta_r_med_px,
        "roi_area_in2": dens_summary.get("roi_area_in2", np.nan),
        "mean_bubbles_per_frame": dens_summary.get("mean_bubbles_per_frame", np.nan),
        "median_bubbles_per_frame": dens_summary.get("median_bubbles_per_frame", np.nan),
        "mean_density_bubbles_in2": dens_summary.get("mean_density_bubbles_in2", np.nan),
        "median_density_bubbles_in2": dens_summary.get("median_density_bubbles_in2", np.nan),
        f"KS2_D_diam_{units_len}": ks_D_diam,
        f"KS2_p_diam_{units_len}": ks_p_diam,
        f"KS2_reject_diam_{units_len}": ks_reject_diam,
        f"KS2_D_speed_{units_vel}": ks_D_speed,
        f"KS2_p_speed_{units_vel}": ks_p_speed,
        f"KS2_reject_speed_{units_vel}": ks_reject_speed,
        f"diam_sum_error_{units_len}": sum_err_d,
        f"speed_sum_error_{units_vel}": sum_err_v,

        f"diam_mu_gt_{units_len}": mu_gt_d,
        f"diam_sd_gt_{units_len}": sd_gt_d,
        f"diam_mu_meas_{units_len}": mu_me_d,
        f"diam_sd_meas_{units_len}": sd_me_d,

        f"speed_mu_gt_{units_vel}": mu_gt_v,
        f"speed_sd_gt_{units_vel}": sd_gt_v,
        f"speed_mu_meas_{units_vel}": mu_me_v,
        f"speed_sd_meas_{units_vel}": sd_me_v,
    }
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "run_summary.csv"), index=False)

    # manual cross-check 
    df_for_manual = df_m.copy()
    if scale_in_per_px:
        df_for_manual["diam_cmp"] = df_for_manual["diam_px"] * scale_in_per_px
        df_for_manual["vel_cmp"]  = df_for_manual["meas_speed_px_s"] * scale_in_per_px
    else:
        df_for_manual["diam_cmp"] = df_for_manual["diam_px"]
        df_for_manual["vel_cmp"]  = df_for_manual["meas_speed_px_s"]
    _try_manual_crosscheck(manual_csv, df_for_manual.rename(columns={"meas_speed_px_s":"vel_px_s"}), out_dir, units_len, units_vel)

    return summary


# ============================================================
# Batch overlay plots 
# ============================================================
def overlay_from_batch(tracked_files, out_dir_overlay, max_runs=10):
    ensure_dir(out_dir_overlay)

    diam_series = []
    vel_series = []
    units_len = "px"
    units_vel = "px/s"

    for f in tracked_files[:max_runs]:
        stem = os.path.basename(f).replace("_tracked.csv", "")
        base = os.path.dirname(f)
        avg = os.path.join(base, stem + "_averages.csv")

        df = pd.read_csv(f)
        m = map_tracked(df)

        df["frame"] = pd.to_numeric(df[m["frame"]], errors="coerce").astype("Int64")
        df["id"] = df[m["id"]].astype(str)
        df["diam_px"] = pd.to_numeric(df[m["diam"]], errors="coerce")

        if m["spd"]:
            df["vel_px_s"] = pd.to_numeric(df[m["spd"]], errors="coerce")
        else:
            vx = pd.to_numeric(df[m["vx"]], errors="coerce") if m["vx"] else np.nan
            vy = pd.to_numeric(df[m["vy"]], errors="coerce") if m["vy"] else np.nan
            df["vel_px_s"] = np.hypot(vx, vy)

        settings, df_avg = (None, None)
        if os.path.exists(avg):
            try:
                settings, df_avg = read_averages_with_settings(avg, header_line_1_indexed=25)
            except Exception:
                settings, df_avg = None, None

        scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
        if scale_in_per_px:
            d = df["diam_px"].to_numpy(float) * scale_in_per_px
            v = df["vel_px_s"].to_numpy(float) * scale_in_per_px
            units_len = "in"
            units_vel = "in/s"
        else:
            d = df["diam_px"].to_numpy(float)
            v = df["vel_px_s"].to_numpy(float)

        diam_series.append((stem, d))
        vel_series.append((stem, v))

    plot_distribution_overlay(
        diam_series,
        os.path.join(out_dir_overlay, "fig_batch_overlay_diameter.png"),
        "Diameter Distribution Overlay (Batch)",
        "Equivalent diameter",
        units_len,
        bins=20
    )

    plot_distribution_overlay(
        vel_series,
        os.path.join(out_dir_overlay, "fig_batch_overlay_velocity.png"),
        "Velocity Distribution Overlay (Batch)",
        "Rise velocity",
        units_vel,
        bins=20
    )

def plot_batch_accuracy_from_master(master_csv, out_dir):
    ensure_dir(out_dir)
    set_thesis_style()

    df = pd.read_csv(master_csv)
    df = df[df["mode"].astype(str).str.upper().eq("GT")].copy()

    if df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.grid(True)
        ax.set_title("Batch Accuracy (GT) - No GT rows found")
        ax.text(0.5, 0.5, "No GT cases in master_summary.csv", ha="center", va="center", transform=ax.transAxes)
        savefig(os.path.join(out_dir, "fig_batch_accuracy_NONE.png"))
        return

    diam_mae_cols = [c for c in df.columns if c.startswith("diam_MAE_")]
    spd_mae_cols  = [c for c in df.columns if c.startswith("speed_MAE_")]
    if not diam_mae_cols or not spd_mae_cols:
        raise ValueError("master_summary.csv missing diam_MAE_* or speed_MAE_* columns.")

    units_len = diam_mae_cols[0].split("diam_MAE_", 1)[1]
    units_vel = spd_mae_cols[0].split("speed_MAE_", 1)[1]

    d_mae  = f"diam_MAE_{units_len}"
    d_rmse = f"diam_RMSE_{units_len}"
    d_bias = f"diam_Bias_{units_len}"
    d_std  = f"diam_Std_{units_len}"

    v_mae  = f"speed_MAE_{units_vel}"
    v_rmse = f"speed_RMSE_{units_vel}"
    v_bias = f"speed_Bias_{units_vel}"
    v_std  = f"speed_Std_{units_vel}"

    def sort_key(s):
        s = str(s).lower()
        for i, k in enumerate(["circle", "ellipse", "wobble", "multi"]):
            if k in s:
                return i
        return 99

    if "case" in df.columns:
        df = df.sort_values(by="case", key=lambda s: s.map(sort_key))
        cases = df["case"].astype(str).tolist()
    else:
        cases = [str(i) for i in range(len(df))]

    x = np.arange(len(cases))

    def bar_with_err(ycol, ecol, title, ylabel, fname):
        y = pd.to_numeric(df[ycol], errors="coerce").to_numpy(float)
        e = pd.to_numeric(df[ecol], errors="coerce").to_numpy(float) if ecol in df.columns else None
        n = pd.to_numeric(df.get("N_matches", np.nan), errors="coerce").to_numpy(float)

        fig, ax = plt.subplots(figsize=(10.8, 5.2))
        ax.grid(True)
        ax.bar(x, y)
        ax.set_title(clean_plot_title(title))
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(cases, rotation=20, ha="right")

        for xi, yi, ni in zip(x, y, n):
            if np.isfinite(yi):
                ax.text(xi, yi, f"N={int(ni) if np.isfinite(ni) else ''}", ha="center", va="bottom", fontsize=10)

        savefig(os.path.join(out_dir, fname))

    def bias_plot(ycol, title, ylabel, fname):
        y = pd.to_numeric(df[ycol], errors="coerce").to_numpy(float)

        fig, ax = plt.subplots(figsize=(10.8, 5.2))
        ax.grid(True)
        ax.axhline(0, linestyle="--", linewidth=2)
        ax.bar(x, y)
        ax.set_title(clean_plot_title(title))
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(cases, rotation=20, ha="right")
        savefig(os.path.join(out_dir, fname))

    bar_with_err(d_mae,  d_std,  "Diameter MAE by Video (GT)",  f"MAE ({units_len})",  "fig_batch_diam_MAE.png")
    bar_with_err(d_rmse, d_std,  "Diameter RMSE by Video (GT)", f"RMSE ({units_len})", "fig_batch_diam_RMSE.png")
    bias_plot(d_bias,    "Diameter Bias by Video (GT)",         f"Bias ({units_len})", "fig_batch_diam_Bias.png")

    bar_with_err(v_mae,  v_std,  "Speed MAE by Video (GT)",     f"MAE ({units_vel})",  "fig_batch_speed_MAE.png")
    bar_with_err(v_rmse, v_std,  "Speed RMSE by Video (GT)",    f"RMSE ({units_vel})", "fig_batch_speed_RMSE.png")
    bias_plot(v_bias,    "Speed Bias by Video (GT)",            f"Bias ({units_vel})", "fig_batch_speed_Bias.png")


# ============================================================
# GUI
# ============================================================
class BubbleEvalGUI(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bubble Evaluation Tool (Thesis-ready v3)")
        self.geometry("900x460")
        self.resizable(False, False)

        self.tracked = tk.StringVar()
        self.folder  = tk.StringVar()
        self.outroot = tk.StringVar()
        self.frames_dir = tk.StringVar()
        self.max_dist = tk.StringVar(value="65")

        self._build()

    def _build(self):
        pad = {"padx": 8, "pady": 6}

        ttk.Label(self, text="Single Run: select *_tracked.csv").grid(row=0, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.tracked, width=78).grid(row=0, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_tracked).grid(row=0, column=2, **pad)

        ttk.Label(self, text="Batch Folder: contains *_tracked.csv").grid(row=1, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.folder, width=78).grid(row=1, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_folder).grid(row=1, column=2, **pad)

        ttk.Label(self, text="Output Root Folder").grid(row=2, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.outroot, width=78).grid(row=2, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_outroot).grid(row=2, column=2, **pad)

        ttk.Label(self, text="(Optional) Frames folder for qualitative panel").grid(row=3, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.frames_dir, width=78).grid(row=3, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_frames).grid(row=3, column=2, **pad)

        ttk.Label(self, text="Max match distance (px) [GT only]").grid(row=4, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.max_dist, width=12).grid(row=4, column=1, sticky="w", **pad)

        ttk.Button(self, text="Run Single", command=self.run_single).grid(row=5, column=0, columnspan=3, pady=(12, 6))
        ttk.Button(self, text="Run Batch Folder (+ overlay plots)", command=self.run_batch).grid(row=6, column=0, columnspan=3)

        self.status = ttk.Label(self, text="Ready.", foreground="gray")
        self.status.grid(row=7, column=0, columnspan=3, pady=(10, 0))

        note = "Tip: set Output Root to something like C:\\Users\\Lab_Staff\\Desktop\\BubbleEvalOutputs"
        ttk.Label(self, text=note, foreground="gray").grid(row=8, column=0, columnspan=3, pady=(6, 0))

    def pick_tracked(self):
        self.tracked.set(filedialog.askopenfilename(filetypes=[("CSV", "*.csv")]))

    def pick_folder(self):
        self.folder.set(filedialog.askdirectory())

    def pick_outroot(self):
        self.outroot.set(filedialog.askdirectory())

    def pick_frames(self):
        self.frames_dir.set(filedialog.askdirectory())

    def _run_one(self, tracked_path, outroot):
        if not tracked_path or not os.path.isfile(tracked_path):
            raise ValueError("Invalid tracked.csv path.")
        if not outroot or not os.path.isdir(outroot):
            raise ValueError("Invalid output root folder.")

        try:
            maxd = float(self.max_dist.get())
        except Exception:
            maxd = 65.0

        stem = os.path.basename(tracked_path).replace("_tracked.csv", "")
        base = os.path.dirname(tracked_path)

        gt     = os.path.join(base, stem + "_gt.csv")
        avg    = os.path.join(base, stem + "_averages.csv")
        manual = os.path.join(base, stem + "_manual.csv")  # optional

        out_dir = ensure_dir(os.path.join(outroot, stem))

        if os.path.exists(gt):
            return out_dir, evaluate_with_gt(
                gt, tracked_path, out_dir, maxd, avg_csv=avg,
                frames_dir=self.frames_dir.get().strip() or None,
                manual_csv=manual if os.path.exists(manual) else None
            )
        else:
            return out_dir, evaluate_no_gt(
                tracked_path, out_dir, avg_csv=avg,
                frames_dir=self.frames_dir.get().strip() or None,
                manual_csv=manual if os.path.exists(manual) else None
            )

    def run_single(self):
        if not self.tracked.get() or not self.outroot.get():
            messagebox.showerror("Missing input", "Select *_tracked.csv and an Output Root folder.")
            return
        self.status.config(text="Running single...")
        self.update_idletasks()

        try:
            out_dir, summary = self._run_one(self.tracked.get(), self.outroot.get())
        except Exception as e:
            messagebox.showerror("Run failed", f"{type(e).__name__}: {e}")
            self.status.config(text="Failed.")
            return

        self.status.config(text=f"Done. Outputs: {out_dir}")
        messagebox.showinfo("Done", f"Completed.\nMode: {summary.get('mode')}\nOutputs in:\n{out_dir}")

    def run_batch(self):
        if not self.folder.get() or not self.outroot.get():
            messagebox.showerror("Missing input", "Select a Batch Folder and an Output Root folder.")
            return

        self.status.config(text="Batch running...")
        self.update_idletasks()

        tracked_files = sorted(glob.glob(os.path.join(self.folder.get(), "*_tracked.csv")))
        if not tracked_files:
            messagebox.showerror("No runs found", "No *_tracked.csv files found in batch folder.")
            self.status.config(text="Ready.")
            return

        rows = []
        for f in tracked_files:
            stem = os.path.basename(f).replace("_tracked.csv", "")
            try:
                out_dir, summary = self._run_one(f, self.outroot.get())
                summary["case"] = stem
                rows.append(summary)
            except Exception as e:
                rows.append({"case": stem, "mode": "ERROR", "error": f"{type(e).__name__}: {e}"})

        # Create overlay output folder FIRST (so everything can use it)
        overlay_dir = ensure_dir(os.path.join(self.outroot.get(), "batch_overlays"))

        # --- Grace diagram: all runs on one map (avg/median Eo/Re per run) ---
        try:
            run_points = []
            for f in tracked_files:
                stem = os.path.basename(f).replace("_tracked.csv", "")
                base = os.path.dirname(f)
                avg  = os.path.join(base, stem + "_averages.csv")
        
                df_tr = pd.read_csv(f)
        
                settings_i = None
                df_av = None
                if os.path.exists(avg):
                    try:
                        settings_i, df_av = read_averages_with_settings(avg, header_line_1_indexed=25)
                    except Exception:
                        settings_i, df_av = None, None
        
                # infer scale for THIS run (needed for median-based Re/Bo)
                scale_i = infer_scale_in_per_px(settings=settings_i, df_avg=df_av)
        
                # ONE correct call
                Eo_i, Re_i = _extract_avg_eo_re(
                    df_avg=df_av,
                    df_tracked=df_tr,
                    settings=settings_i,
                    scale_in_per_px=scale_i
                )
        
                run_points.append({"label": stem, "Eo": Eo_i, "Re": Re_i})
        
            grace_all_png = os.path.join(overlay_dir, "fig_grace_all_runs_avg.png")
            plot_grace_avg_points_all_runs(run_points, grace_all_png)
        
        except Exception as e:
            log_error(overlay_dir, f"[GRACE ALL RUNS FAIL] {type(e).__name__}: {e}")


        master_df = pd.DataFrame(rows)

        master_csv = os.path.join(self.outroot.get(), "master_summary.csv")
        try:
            master_df.to_csv(master_csv, index=False)
        except PermissionError:
            # Most common cause: file open in Excel or folder not writable.
            # Fall back to overlay_dir with a unique name.
            alt_csv = os.path.join(overlay_dir, f"master_summary_{pd.Timestamp.now():%Y%m%d_%H%M%S}.csv")
            master_df.to_csv(alt_csv, index=False)
            log_error(overlay_dir, f"[MASTER CSV PERMISSION] Could not write {master_csv}; wrote {alt_csv} instead.")


        acc_dir = ensure_dir(os.path.join(overlay_dir, "accuracy"))
        try:
            plot_batch_accuracy_from_master(master_csv, acc_dir)
        except Exception as e:
            log_error(overlay_dir, f"[BATCH ACCURACY PLOTS FAIL] {type(e).__name__}: {e}")

        try:
            overlay_from_batch(tracked_files, overlay_dir, max_runs=12)
        except Exception as e:
            log_error(overlay_dir, f"[BATCH OVERLAY FAIL] {type(e).__name__}: {e}")

        self.status.config(text="Batch complete.")
        messagebox.showinfo("Done", f"Batch complete.\nSaved master_summary.csv\nOverlay plots in:\n{overlay_dir}")

        

if __name__ == "__main__":
    BubbleEvalGUI().mainloop()


In [16]:
# ============================================================
# Bubble Evaluation Tool (THESIS-READY) - FULL SCRIPT v3 (FIXED)
# ============================================================
# Fixes included:
#  - FIX: frame_h defined BEFORE ROI filtering in GT mode (prevents NameError)
#  - FIX: IoU uses _pred_mask.mp4 only (binary). _filtered.mp4 is overlay and not used.
#  - FIX: removed duplicate mask-path assignments; added clear logging when masks missing
#  - Keeps your outputs + behavior the same
# ============================================================

import os, glob, traceback, random, re
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.collections import LineCollection

from scipy.optimize import linear_sum_assignment
from scipy.interpolate import UnivariateSpline

import tkinter as tk
from tkinter import ttk, filedialog, messagebox

try:
    import cv2
    _HAS_CV2 = True
except Exception:
    _HAS_CV2 = False

BOTTOM_BAR_PX = 30  # your timestamp bar height

# ============================================================
# Helpers / style / IO
# ============================================================

def _binarize_from_bgr_or_gray(frame, thresh=128):
    """Return boolean mask from a cv2 frame (BGR or gray)."""
    if frame is None:
        return None
    if frame.ndim == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame
    return (gray >= thresh)

def _apply_vertical_crop(mask_bool, crop_top=0, crop_bottom=0):
    """Crop boolean mask in y (top/bottom)."""
    if mask_bool is None:
        return None
    H = mask_bool.shape[0]
    y0 = int(max(0, crop_top))
    y1 = int(H - crop_bottom) if crop_bottom else H
    y1 = int(min(H, y1))
    if y1 <= y0:
        return mask_bool  # fail-safe: no crop if invalid
    return mask_bool[y0:y1, :]

def _mask_diff_rgb(gt_bool, pr_bool):
    """
    Build RGB diff image:
      - GT pixels (including overlap) = white
      - Over-prediction (pred=1, gt=0) = red
      - Under-prediction (gt=1, pred=0) = blue
      - Background = black
    """
    if gt_bool is None or pr_bool is None:
        return None

    if pr_bool.shape != gt_bool.shape:
        # Resize pred to GT size (nearest for masks)
        pr_u8 = pr_bool.astype(np.uint8) * 255
        pr_u8 = cv2.resize(pr_u8, (gt_bool.shape[1], gt_bool.shape[0]), interpolation=cv2.INTER_NEAREST)
        pr_bool = (pr_u8 >= 128)

    gt = gt_bool.astype(bool)
    pr = pr_bool.astype(bool)

    over  = pr & (~gt)   # false positives
    under = gt & (~pr)   # false negatives
    both  = gt & pr      # overlap (also GT)

    rgb = np.zeros((gt.shape[0], gt.shape[1], 3), dtype=np.uint8)

    # Start with GT white (includes overlap)
    rgb[gt] = (255, 255, 255)

    # Under-prediction: blue (override GT-white where GT-only)
    rgb[under] = (0, 0, 255)

    # Over-prediction: red
    rgb[over] = (255, 0, 0)

    # Overlap remains white (because gt set to white and we didn't override with blue/red)
    return rgb

def make_mask_diff_panel_from_videos(gt_mask_mp4, pred_mask_mp4, out_png,
                                    start_frame=0, n_frames=5,
                                    thresh=128, crop_top=0, crop_bottom=0):
    """
    Creates ONE PNG with n_frames consecutive frames showing mask agreement:
      white = GT (incl overlap)
      red   = over-pred (pred=1, gt=0)
      blue  = under-pred (gt=1, pred=0)
    """
    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read mask videos.")
    if not gt_mask_mp4 or not os.path.exists(gt_mask_mp4):
        raise FileNotFoundError(f"GT mask video not found: {gt_mask_mp4}")
    if not pred_mask_mp4 or not os.path.exists(pred_mask_mp4):
        raise FileNotFoundError(f"Pred mask video not found: {pred_mask_mp4}")

    set_thesis_style()

    cap_gt = cv2.VideoCapture(gt_mask_mp4)
    cap_pr = cv2.VideoCapture(pred_mask_mp4)

    frs = list(range(int(start_frame), int(start_frame) + int(n_frames)))

    fig, axes = plt.subplots(1, len(frs), figsize=(4.0 * len(frs), 4.0))
    if len(frs) == 1:
        axes = np.array([axes])

    for j, fr in enumerate(frs):
        gt_bgr = _read_frame_at(cap_gt, fr)
        pr_bgr = _read_frame_at(cap_pr, fr)

        gt_bool = _binarize_from_bgr_or_gray(gt_bgr, thresh=thresh)
        pr_bool = _binarize_from_bgr_or_gray(pr_bgr, thresh=thresh)

        gt_bool = _apply_vertical_crop(gt_bool, crop_top=crop_top, crop_bottom=crop_bottom)
        pr_bool = _apply_vertical_crop(pr_bool, crop_top=crop_top, crop_bottom=crop_bottom)

        rgb = _mask_diff_rgb(gt_bool, pr_bool)

        if rgb is not None:
            axes[j].imshow(rgb)  # rgb already
        axes[j].set_title(f"Frame {fr}")
        axes[j].axis("off")

    # One legend block (text) to avoid adding a matplotlib legend for image pixels
    fig.suptitle("Mask Difference Overlay (GT vs Pred)", y=1.02, fontsize=18)
    fig.text(
        0.5, 0.02,
        "White: GT (incl overlap)    Red: Over-prediction (FP)    Blue: Under-prediction (FN)",
        ha="center", va="bottom",
        bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="black", alpha=0.95)
    )

    cap_gt.release()
    cap_pr.release()

    savefig(out_png)

def _safe_float(v):
    try:
        return float(v)
    except Exception:
        return None

def infer_ignore_top_px(settings):
    if not settings:
        return 0
    # common variants you might have
    for k in ["IGNORE_TOP_PX", "IGNORE_TOP", "IGNORE_T", "IGNORE_TEXT_PX", "IGNORE_TEXT_TOP_PX"]:
        if k in settings:
            v = _safe_float(settings[k])
            if v is not None:
                return int(round(v))
    # fallback: try fuzzy match
    for k, v in settings.items():
        kk = str(k).upper()
        if "IGNORE" in kk and "TOP" in kk:
            vv = _safe_float(v)
            if vv is not None:
                return int(round(vv))
    return 0

def apply_roi_filter(df, cy_col, ignore_top_px=0, bottom_bar_px=0, frame_h=None):
    """
    Keep only rows whose centroid y is inside the detectable region:
      y >= ignore_top_px  AND  y < (frame_h - bottom_bar_px)
    If frame_h is None, only applies the top filter.
    """
    if df is None or df.empty:
        return df
    df = df.copy()
    y = pd.to_numeric(df[cy_col], errors="coerce")
    keep = (y >= float(ignore_top_px))
    if frame_h is not None:
        keep &= (y < float(frame_h - bottom_bar_px))
    return df[keep].copy()

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

def set_thesis_style():
    plt.rcParams.update({
        "figure.dpi": 140,
        "savefig.dpi": 300,
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.labelsize": 14,
        "legend.fontsize": 11,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "axes.linewidth": 1.2,
        "grid.alpha": 0.25,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
    })

def log_error(out_dir, msg):
    try:
        with open(os.path.join(out_dir, "errors.log"), "a", encoding="utf-8") as f:
            f.write(msg.rstrip() + "\n")
    except Exception:
        pass

def savefig(path):
    fig = plt.gcf()
    try:
        fig.tight_layout()
    except Exception:
        pass
    ensure_dir(os.path.dirname(path))
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)

def safe_plot(out_dir, name, out_png, fn):
    """
    Always writes a PNG:
      - success -> saves out_png
      - failure -> placeholder PNG + logs errors.log
    """
    try:
        fn()
    except Exception as e:
        log_error(out_dir, f"[PLOT FAIL] {name}: {type(e).__name__}: {e}")
        log_error(out_dir, traceback.format_exc())
        try:
            set_thesis_style()
            fig, ax = plt.subplots(figsize=(7.8, 5.4))
            ax.grid(True)
            ax.set_title(name)
            ax.text(
                0.5, 0.5,
                f"Plot failed:\n{type(e).__name__}\n{e}",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black")
            )
            savefig(out_png)
        except Exception:
            pass

def fname_units(units: str) -> str:
    """Windows-safe units string for filenames (no slashes)."""
    if units is None:
        return "unitless"
    u = str(units).replace("/", "_per_").replace("\\", "_").replace(" ", "")
    u = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", u)
    return u

def clip_by_percentile(x, lo=1, hi=99):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return x, None, None
    a, b = np.percentile(x, [lo, hi])
    return x[(x >= a) & (x <= b)], a, b

def clip_for_hist(x, lo=1, hi=99):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    n_total = int(x.size)
    if n_total == 0:
        return x, np.nan, np.nan, 0, 0
    a, b = np.percentile(x, [lo, hi])
    keep = (x >= a) & (x <= b)
    x2 = x[keep]
    return x2, float(a), float(b), n_total, int(x2.size)

def compute_delta_r_stats(df_tracks, require_consecutive=True, max_step_px=None):
    """
    Compute frame-to-frame displacement Δr for tracked centroids.

    df_tracks must have columns: id, frame, cx, cy (numeric-able).
    - require_consecutive: only use steps where frame_{i+1} = frame_i + 1
    - max_step_px: optional jump filter to remove ID-swap leaps (e.g., 100 px)

    Returns:
      (delta_r_median, delta_r_mean, N_steps, steps_array)
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, np.nan, 0, np.array([])

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d["cx"] = pd.to_numeric(d["cx"], errors="coerce")
    d["cy"] = pd.to_numeric(d["cy"], errors="coerce")
    d = d.dropna(subset=["id", "frame", "cx", "cy"]).copy()

    steps = []

    for tid, g in d.groupby("id"):
        g = g.sort_values("frame")
        fr = g["frame"].to_numpy(float)
        x  = g["cx"].to_numpy(float)
        y  = g["cy"].to_numpy(float)

        if len(fr) < 2:
            continue

        dfr = np.diff(fr)
        dx  = np.diff(x)
        dy  = np.diff(y)
        dr  = np.hypot(dx, dy)

        # keep only consecutive frame steps (recommended)
        if require_consecutive:
            keep = (dfr == 1)
            dr = dr[keep]

        # optional jump filter (recommended for real data)
        if max_step_px is not None and dr.size:
            dr = dr[dr <= float(max_step_px)]

        if dr.size:
            steps.append(dr)

    if not steps:
        return np.nan, np.nan, 0, np.array([])

    steps = np.concatenate(steps).astype(float)
    steps = steps[np.isfinite(steps)]
    if steps.size == 0:
        return np.nan, np.nan, 0, np.array([])

    return float(np.median(steps)), float(np.mean(steps)), int(steps.size), steps
# ============================================================
# Stats
# ============================================================
def _clean(x):
    x = np.asarray(x, float)
    return x[np.isfinite(x)]

def stats_dict(x):
    x = _clean(x)
    if x.size == 0:
        return {"N": 0, "mean": np.nan, "median": np.nan, "std": np.nan, "min": np.nan, "max": np.nan}
    return {
        "N": int(x.size),
        "mean": float(np.mean(x)),
        "median": float(np.median(x)),
        "std": float(np.std(x, ddof=1)) if x.size > 1 else 0.0,
        "min": float(np.min(x)),
        "max": float(np.max(x)),
    }

def mae(x):
    x = _clean(x)
    return float(np.mean(np.abs(x))) if x.size else np.nan

def rmse(x):
    x = _clean(x)
    return float(np.sqrt(np.mean(x**2))) if x.size else np.nan

def bias(x):
    x = _clean(x)
    return float(np.mean(x)) if x.size else np.nan

def add_statbox(ax, lines, loc="upper right"):
    text = "\n".join(lines)
    bbox = dict(boxstyle="round,pad=0.4", fc="white", ec="black", lw=1.0)

    locs = {
        "upper left":  (0.02, 0.98, "top",    "left"),
        "upper right": (0.98, 0.98, "top",    "right"),
        "lower left":  (0.02, 0.02, "bottom", "left"),
        "lower right": (0.98, 0.02, "bottom", "right"),
    }

    x, y, va, ha = locs[loc]

    ax.text(
        x, y, text,
        transform=ax.transAxes,
        va=va, ha=ha,
        bbox=bbox
    )

def robust_limits(x, p_lo=1, p_hi=99, pad_frac=0.08):
    x = _clean(x)
    if x.size == 0:
        return (-1, 1)
    a = np.percentile(x, p_lo)
    b = np.percentile(x, p_hi)
    if a == b:
        pad = 1.0 if a == 0 else abs(a) * 0.2
        return (a - pad, b + pad)
    pad = (b - a) * pad_frac
    return (a - pad, b + pad)

def diameter_uncertainty_from_fit(df_tracks, fps=None,
                                  fit_mode="constant", poly_deg=2,
                                  spline_s=None,
                                  min_points=20,
                                  max_step_px=None,
                                  units_col="diam_px"):
    """
    Per-track diameter precision uncertainty from residuals around a smooth fit:
        u_D = sqrt( 1/(N-1) * sum_i (D_i - Dfit_i)^2 )

    df_tracks must have columns: id, frame, and units_col (diam_px or diam_in).
    Returns:
        u_D_global, per_track_df
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, pd.DataFrame()

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d[units_col] = pd.to_numeric(d[units_col], errors="coerce")

    # time base
    if "t_s" in d.columns and d["t_s"].notna().any():
        d["t"] = pd.to_numeric(d["t_s"], errors="coerce")
    else:
        if fps and fps > 0:
            d["t"] = d["frame"] / float(fps)
        else:
            d["t"] = d["frame"]

    d = d.dropna(subset=["id", "t", units_col]).copy()

    rows = []
    all_r2 = []

    for tid, g in d.groupby("id"):
        g = g.sort_values("t")
        t = g["t"].to_numpy(float)
        D = g[units_col].to_numpy(float)

        m = np.isfinite(t) & np.isfinite(D)
        t, D = t[m], D[m]
        if t.size < min_points:
            continue

        # optional jump filter using centroid steps (if provided)
        if max_step_px is not None and {"cx","cy"}.issubset(g.columns) and t.size >= 3:
            x = pd.to_numeric(g["cx"], errors="coerce").to_numpy(float)[m]
            y = pd.to_numeric(g["cy"], errors="coerce").to_numpy(float)[m]
            step = np.hypot(np.diff(x), np.diff(y))
            ok = np.ones_like(D, dtype=bool)
            ok[1:] = step <= float(max_step_px)
            t, D = t[ok], D[ok]
            if t.size < min_points:
                continue

        # ---- Fit D(t) ----
        if fit_mode == "constant":
            Dfit = np.full_like(D, np.mean(D))
        elif fit_mode == "linear":
            p = np.polyfit(t, D, 1)
            Dfit = np.polyval(p, t)
        elif fit_mode == "poly":
            deg = int(max(1, min(poly_deg, 5)))
            p = np.polyfit(t, D, deg)
            Dfit = np.polyval(p, t)
        elif fit_mode == "spline":
            s = float(spline_s) if spline_s is not None else (t.size * (0.5 ** 2))
            sp = UnivariateSpline(t, D, s=s)
            Dfit = sp(t)
        else:
            raise ValueError(f"Unknown fit_mode: {fit_mode}")

        r = D - Dfit
        r2 = r**2
        N = r2.size
        uD = np.sqrt(np.sum(r2) / (N - 1)) if N > 1 else np.nan

        rows.append({
            "id": tid,
            "N": int(N),
            f"u_D_{units_col}": float(uD) if np.isfinite(uD) else np.nan,
            f"D_mean_{units_col}": float(np.mean(D)) if np.isfinite(np.mean(D)) else np.nan,
            f"D_std_{units_col}": float(np.std(D, ddof=1)) if N > 1 else np.nan,
        })
        all_r2.extend(list(r2))

    per_track = pd.DataFrame(rows)

    all_r2 = np.asarray(all_r2, float)
    all_r2 = all_r2[np.isfinite(all_r2)]
    uD_global = float(np.sqrt(np.sum(all_r2) / (all_r2.size - 1))) if all_r2.size > 1 else np.nan

    return uD_global, per_track


def run_diameter_uncertainty(df_tracks, tabs_dir, out_dir, fps, scale_in_per_px):
    """
    Computes segmentation diameter precision uncertainty (no GT).
    Saves:
      - diameter_uncertainty_per_track.csv
      - diameter_uncertainty_summary.csv
    Returns: (uD_px, uD_in)
    """
    try:
        if "id" not in df_tracks.columns or "frame" not in df_tracks.columns or "diam_px" not in df_tracks.columns:
            raise ValueError("Diameter uncertainty needs columns: id, frame, diam_px")

        # ---- px-space always available ----
        uD_px, per_track_px = diameter_uncertainty_from_fit(
            df_tracks[["id","frame","diam_px","cx","cy","t_s"]].copy() if "t_s" in df_tracks.columns else
            df_tracks[["id","frame","diam_px","cx","cy"]].copy(),
            fps=fps,
            fit_mode="constant",   # bubble diameter should be ~constant over a short track
            min_points=20,
            max_step_px=25
        )
        per_track_px.to_csv(os.path.join(tabs_dir, "diameter_uncertainty_per_track_px.csv"), index=False)

        uD_in = np.nan
        if scale_in_per_px:
            df2 = df_tracks.copy()
            df2["diam_in"] = pd.to_numeric(df2["diam_px"], errors="coerce") * float(scale_in_per_px)

            uD_in, per_track_in = diameter_uncertainty_from_fit(
                df2[["id","frame","diam_in","cx","cy","t_s"]].copy() if "t_s" in df2.columns else
                df2[["id","frame","diam_in","cx","cy"]].copy(),
                fps=fps,
                fit_mode="constant",
                min_points=20,
                max_step_px=25,
                units_col="diam_in"
            )
            per_track_in.to_csv(os.path.join(tabs_dir, "diameter_uncertainty_per_track_in.csv"), index=False)

        pd.DataFrame([{
            "diameter_u_D_px": uD_px,
            "diameter_u_D_in": uD_in if scale_in_per_px else "",
            "fit_mode": "constant",
            "min_points": 20,
            "max_step_px": 25,
            "note": "Precision (repeatability) of diameter along each track; not GT accuracy."
        }]).to_csv(os.path.join(tabs_dir, "diameter_uncertainty_summary.csv"), index=False)

        return uD_px, (uD_in if scale_in_per_px else np.nan)

    except Exception as e:
        log_error(out_dir, f"[DIAM UNC FAIL] {type(e).__name__}: {e}")
        return np.nan, np.nan

def centroid_uncertainty_from_fit(df_tracks, fps=None,
                                  fit_mode="poly", poly_deg=3,
                                  spline_s=None,
                                  min_points=20,
                                  max_step_px=None):
    """
    Implements Eq. 3.17:
        u_xy = sqrt( 1/(N-1) * sum_i [ (x_i-xfit_i)^2 + (y_i-yfit_i)^2 ] )

    df_tracks must have columns: id, frame, cx, cy (and optionally t_s).
    - fit_mode:
        "constant" : xfit=mean(x), yfit=mean(y)  (quiescent)
        "linear"   : degree-1 poly fit           (slow drift)
        "poly"     : polynomial fit deg=poly_deg (accelerating)
        "spline"   : UnivariateSpline fit        (accelerating/curvy)
    - spline_s: smoothing factor for spline; if None, auto-ish (see below)
    - max_step_px: optional jump filter; if provided, drops segments with step > max_step_px
    Returns:
        overall_u_xy_px, per_track_table_df
    """
    if df_tracks is None or df_tracks.empty:
        return np.nan, pd.DataFrame()

    d = df_tracks.copy()
    d["id"] = d["id"].astype(str)
    d["frame"] = pd.to_numeric(d["frame"], errors="coerce")
    d["cx"] = pd.to_numeric(d["cx"], errors="coerce")
    d["cy"] = pd.to_numeric(d["cy"], errors="coerce")

    # time base
    if "t_s" in d.columns and d["t_s"].notna().any():
        d["t"] = pd.to_numeric(d["t_s"], errors="coerce")
    else:
        if fps and fps > 0:
            d["t"] = d["frame"] / float(fps)
        else:
            d["t"] = d["frame"]  # fallback (units = frames)

    d = d.dropna(subset=["id", "t", "cx", "cy"]).copy()

    rows = []
    all_r2 = []  # store r^2 across all selected samples (for global u_xy)

    for tid, g in d.groupby("id"):
        g = g.sort_values("t")
        t = g["t"].to_numpy(float)
        x = g["cx"].to_numpy(float)
        y = g["cy"].to_numpy(float)

        m = np.isfinite(t) & np.isfinite(x) & np.isfinite(y)
        t, x, y = t[m], x[m], y[m]
        if t.size < min_points:
            continue

        # Optional: filter out obvious ID-switch jumps
        if max_step_px is not None and t.size >= 3:
            dx = np.diff(x)
            dy = np.diff(y)
            step = np.hypot(dx, dy)
            ok = np.ones_like(x, dtype=bool)
            ok[1:] = step <= float(max_step_px)
            # keep only the "ok" points; (simple approach)
            t, x, y = t[ok], x[ok], y[ok]
            if t.size < min_points:
                continue

        # ----- Fit x(t), y(t) -----
        if fit_mode == "constant":
            xfit = np.full_like(x, np.mean(x))
            yfit = np.full_like(y, np.mean(y))

        elif fit_mode == "linear":
            px = np.polyfit(t, x, 1)
            py = np.polyfit(t, y, 1)
            xfit = np.polyval(px, t)
            yfit = np.polyval(py, t)

        elif fit_mode == "poly":
            deg = int(poly_deg)
            deg = max(1, min(deg, 5))  # keep sane
            px = np.polyfit(t, x, deg)
            py = np.polyfit(t, y, deg)
            xfit = np.polyval(px, t)
            yfit = np.polyval(py, t)

        elif fit_mode == "spline":
            # Choose smoothing if not given:
            # spline_s roughly controls how smooth the fit is.
            # If you don't know, start with something like s = N * (0.5 px)^2
            if spline_s is None:
                s = t.size * (0.5 ** 2)
            else:
                s = float(spline_s)

            sx = UnivariateSpline(t, x, s=s)
            sy = UnivariateSpline(t, y, s=s)
            xfit = sx(t)
            yfit = sy(t)

        else:
            raise ValueError(f"Unknown fit_mode: {fit_mode}")

        # residual distance per sample
        rx = x - xfit
        ry = y - yfit
        r2 = rx**2 + ry**2

        # Eq 3.17 for this track
        N = r2.size
        u_xy = np.sqrt(np.sum(r2) / (N - 1)) if N > 1 else np.nan

        rows.append({
            "id": tid,
            "N": int(N),
            "u_xy_px": float(u_xy) if np.isfinite(u_xy) else np.nan,
            "u_x_px": float(np.std(rx, ddof=1)) if N > 1 else np.nan,
            "u_y_px": float(np.std(ry, ddof=1)) if N > 1 else np.nan,
        })
        all_r2.extend(list(r2))

    per_track = pd.DataFrame(rows)
    all_r2 = np.asarray(all_r2, float)
    all_r2 = all_r2[np.isfinite(all_r2)]

    # Global u_xy across all included samples (matches Eq. 3.17 idea at dataset level)
    # Use (N_total - 1) in denominator
    if all_r2.size > 1:
        u_xy_global = float(np.sqrt(np.sum(all_r2) / (all_r2.size - 1)))
    else:
        u_xy_global = np.nan

    return u_xy_global, per_track

def run_centroid_uncertainty(df_tracks, tabs_dir, out_dir, fps, scale_in_per_px):
    """
    Computes and saves centroid uncertainty (Eq. 3.17) from tracked trajectories.
    Saves:
      - centroid_uncertainty_per_track.csv
      - centroid_uncertainty_summary.csv
    Returns: (u_xy_px, u_xy_in)
    """
    try:
        need = ["id", "frame", "cx", "cy"]
        for c in need:
            if c not in df_tracks.columns:
                raise ValueError(f"Centroid uncertainty needs column '{c}' but it was missing.")

        cols = ["id", "frame", "cx", "cy"]
        if "t_s" in df_tracks.columns:
            cols.append("t_s")

        u_xy_px, per_track = centroid_uncertainty_from_fit(
            df_tracks[cols].copy(),
            fps=fps,
            fit_mode="poly",
            poly_deg=3,
            min_points=20,
            max_step_px=25
        )

        per_track.to_csv(os.path.join(tabs_dir, "centroid_uncertainty_per_track.csv"), index=False)
        pd.DataFrame([{
            "centroid_u_xy_px": u_xy_px,
            "fit_mode": "poly3",
            "min_points": 20,
            "max_step_px": 25
        }]).to_csv(os.path.join(tabs_dir, "centroid_uncertainty_summary.csv"), index=False)

        u_xy_in = (u_xy_px * scale_in_per_px) if (scale_in_per_px and np.isfinite(u_xy_px)) else np.nan
        return u_xy_px, u_xy_in

    except Exception as e:
        log_error(out_dir, f"[CENTROID UNC FAIL] {type(e).__name__}: {e}")
        return np.nan, np.nan

# ============================================================
# Averages/settings reader (YOUR SPEC: header line 25)
# ============================================================
def read_averages_with_settings(path, header_line_1_indexed=25):
    """
    Averages file format:
      - run settings in lines 1..(header_line-1)
      - header row at line 25 (1-indexed)
      - data starts line 26
    """
    header_idx = int(header_line_1_indexed) - 1  # convert to 0-index

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        lines = f.read().splitlines()

    if len(lines) <= header_idx:
        raise ValueError(f"Averages file too short. Expected header at line {header_line_1_indexed}.")

    settings = {}
    for ln in lines[:header_idx]:
        s = ln.strip()
        if not s:
            continue
        if ":" in s:
            k, v = s.split(":", 1)
        elif "," in s:
            k, v = s.split(",", 1)
        elif "\t" in s:
            k, v = s.split("\t", 1)
        else:
            parts = re.split(r"\s{2,}", s)
            if len(parts) >= 2:
                k, v = parts[0], parts[1]
            else:
                continue
        k = k.strip()
        v = v.strip()
        if k:
            settings[k] = v

    df_avg = pd.read_csv(
        path,
        header=header_idx,
        engine="python",
        sep=None,
        on_bad_lines="skip"
    )
    return settings, df_avg

def infer_scale_in_per_px(settings=None, df_avg=None):
    if settings:
        for k in ["SCALE_IN_PER_PX", "scale_in_per_px"]:
            if k in settings:
                val = _safe_float(settings[k])
                if val and val > 0:
                    return val
        pairs = [("SCALE_IN_INPUT", "SCALE_PX_INPUT"), ("SCALE_IN", "SCALE_PX")]
        for a, b in pairs:
            if a in settings and b in settings:
                A = _safe_float(settings[a])
                B = _safe_float(settings[b])
                if A and B and B > 0:
                    return A / B

    if df_avg is not None and ("diam_in" in df_avg.columns) and ("diam_px" in df_avg.columns):
        r = pd.to_numeric(df_avg["diam_in"], errors="coerce") / pd.to_numeric(df_avg["diam_px"], errors="coerce")
        r = r[np.isfinite(r) & (r > 0)]
        if len(r):
            return float(np.median(r))
    return None

def infer_fps(settings=None):
    if not settings:
        return None
    v = settings.get("FPS_STILL", None)
    try:
        v = float(v) if v not in (None, "", "None") else None
    except Exception:
        v = None
    return v if (v is not None and v > 0) else None


# ============================================================
# Flexible column mapping
# ============================================================
def pick_col(df, candidates):
    cols = set(df.columns)
    for c in candidates:
        if c in cols:
            return c
    return None

def map_gt(df_gt):
    m = {}
    m["frame"] = pick_col(df_gt, ["frame", "Frame"])
    m["cx"]    = pick_col(df_gt, ["gt_cx_px", "gt_cx", "cx", "x"])
    m["cy"]    = pick_col(df_gt, ["gt_cy_px", "gt_cy", "cy", "y"])
    m["diam"]  = pick_col(df_gt, ["gt_diam_equiv_px", "gt_diam_eq", "gt_diam_px", "diam_px", "diam"])
    m["vx"]    = pick_col(df_gt, ["gt_vx_px_s", "gt_vx", "vx"])
    m["vy"]    = pick_col(df_gt, ["gt_vy_px_s", "gt_vy", "vy"])
    m["spd"]   = pick_col(df_gt, ["gt_speed_px_s", "gt_speed", "speed_px_s", "speed"])
    req = [m["frame"], m["cx"], m["cy"], m["diam"]]
    if any(v is None for v in req):
        raise ValueError(f"GT missing required columns. Found: {list(df_gt.columns)}")
    return m

def map_tracked(df_m):
    m = {}
    m["frame"] = pick_col(df_m, ["frame", "Frame"])
    m["id"]    = pick_col(df_m, ["id", "ID", "track_id"])
    m["cx"]    = pick_col(df_m, ["cx", "x", "centroid_x"])
    m["cy"]    = pick_col(df_m, ["cy", "y", "centroid_y"])
    m["diam"]  = pick_col(df_m, ["diam_px", "diam", "diameter_px"])
    m["vx"]    = pick_col(df_m, ["vel_x_px_s", "vx_px_s", "vx"])
    m["vy"]    = pick_col(df_m, ["vel_y_px_s", "vy_px_s", "vy"])
    m["spd"]   = pick_col(df_m, ["vel_px_s", "speed_px_s", "vel", "speed"])
    req = [m["frame"], m["id"], m["cx"], m["cy"], m["diam"]]
    if any(v is None for v in req):
        raise ValueError(f"TRACKED missing required columns. Found: {list(df_m.columns)}")
    return m


# ============================================================
# Matching (Hungarian per frame)
# ============================================================
def match_frame(gtf, mf, max_dist_px):
    gx = gtf["gt_cx_px"].to_numpy(float)
    gy = gtf["gt_cy_px"].to_numpy(float)
    mx = mf["cx"].to_numpy(float)
    my = mf["cy"].to_numpy(float)

    if gx.size == 0 or mx.size == 0:
        return []

    D = np.sqrt((gx[:, None] - mx[None, :])**2 + (gy[:, None] - my[None, :])**2)
    BIG = 1e9
    D[~np.isfinite(D)] = BIG
    D[D > max_dist_px] = BIG

    r, c = linear_sum_assignment(D)
    pairs = []
    for i, j in zip(r, c):
        if D[i, j] < BIG:
            pairs.append((int(i), int(j), float(D[i, j])))
    return pairs

# ============================================================
# Grace diagram overlay (hard-coded image + axis mapping)
#   - Single-run: plots ONE point (avg_Eo, avg_Re)
#   - Batch: plots ONE point per run for ALL runs on one map
# ============================================================

# Grace diagram image path (script-safe + notebook-safe)
try:
    _BASE_DIR = os.path.dirname(__file__)
except NameError:
    _BASE_DIR = os.getcwd()

GRACE_IMG_PATH = os.path.join(_BASE_DIR, "GraceDiagram.png")

# Hard-coded plot box in the provided image (pixels)
# Detected from the image: left/right border columns ~76/547, top/bottom ~18/585
GRACE_PLOT_X0 = 76
GRACE_PLOT_X1 = 547
GRACE_PLOT_Y0 = 18
GRACE_PLOT_Y1 = 585

# Axis limits shown on the diagram
GRACE_EO_MIN = 1e-2
GRACE_EO_MAX = 1e3
GRACE_RE_MIN = 1e-1
GRACE_RE_MAX = 1e5


def _grace_xy_from_eo_re(eo, re_):
    """
    Map (Eo, Re) to pixel coords on the Grace diagram image.
    Both axes are log10.
    Returns (x_pix, y_pix) arrays (float).
    """
    eo = np.asarray(eo, float)
    re_ = np.asarray(re_, float)

    m = np.isfinite(eo) & np.isfinite(re_) & (eo > 0) & (re_ > 0)
    eo = eo[m]
    re_ = re_[m]
    if eo.size == 0:
        return np.array([]), np.array([])

    x = (np.log10(eo) - np.log10(GRACE_EO_MIN)) / (np.log10(GRACE_EO_MAX) - np.log10(GRACE_EO_MIN))
    y = (np.log10(re_) - np.log10(GRACE_RE_MIN)) / (np.log10(GRACE_RE_MAX) - np.log10(GRACE_RE_MIN))

    x = np.clip(x, 0.0, 1.0)
    y = np.clip(y, 0.0, 1.0)

    x_pix = GRACE_PLOT_X0 + x * (GRACE_PLOT_X1 - GRACE_PLOT_X0)
    y_pix = GRACE_PLOT_Y1 - y * (GRACE_PLOT_Y1 - GRACE_PLOT_Y0)

    return x_pix, y_pix


def _load_grace_bg_or_raise():
    if not os.path.exists(GRACE_IMG_PATH):
        raise FileNotFoundError(f"GraceDiagram.png not found at: {GRACE_IMG_PATH}")
    import matplotlib.image as mpimg
    return mpimg.imread(GRACE_IMG_PATH)


def _extract_avg_eo_re(df_avg=None, df_tracked=None):
    """
    Prefer averages.csv (avg_Eo/avg_Re). If not present, fall back to mean of tracked Eo/Re.
    Returns (Eo_avg, Re_avg) or (nan, nan).
    """
    # Preferred: averages file
    if df_avg is not None and not df_avg.empty:
        eo_col = pick_col(df_avg, ["avg_Eo", "Eo_avg", "Eo", "avg_eo"])
        re_col = pick_col(df_avg, ["avg_Re", "Re_avg", "Re", "avg_re"])
        if eo_col and re_col:
            eo0 = pd.to_numeric(df_avg[eo_col], errors="coerce").dropna()
            re0 = pd.to_numeric(df_avg[re_col], errors="coerce").dropna()
            if len(eo0) and len(re0):
                return float(eo0.iloc[0]), float(re0.iloc[0])

    # Fallback: tracked file (if it already has Eo/Re per sample)
    if df_tracked is not None and not df_tracked.empty:
        eo_col = pick_col(df_tracked, ["Eo", "eo", "EOTVOS", "eotvos"])
        re_col = pick_col(df_tracked, ["Re", "re", "REYNOLDS", "reynolds"])
        if eo_col and re_col:
            eo = pd.to_numeric(df_tracked[eo_col], errors="coerce").to_numpy(float)
            re_ = pd.to_numeric(df_tracked[re_col], errors="coerce").to_numpy(float)
            m = np.isfinite(eo) & np.isfinite(re_) & (eo > 0) & (re_ > 0)
            eo, re_ = eo[m], re_[m]
            if eo.size:
                return float(np.mean(eo)), float(np.mean(re_))

    return np.nan, np.nan


def plot_grace_avg_point(df_tracked, out_png, df_avg=None, label="Run", annotate=False):
    """
    Single-run Grace diagram plot:
    - ONE point (avg Eo, avg Re)
    - Legend placed outside (right side)
    - No text drawn on the image itself
    """
    set_thesis_style()

    try:
        bg = _load_grace_bg_or_raise()
    except Exception as e:
        fig, ax = plt.subplots(figsize=(7.6, 6.6))
        ax.text(0.5, 0.5, str(e), ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        savefig(out_png)
        return

    # Extract averages
    Eo_avg, Re_avg = _extract_avg_eo_re(df_avg=df_avg, df_tracked=df_tracked)

    fig, ax = plt.subplots(figsize=(10.5, 7.0))  # wider for legend
    ax.imshow(bg)
    ax.axis("off")

    if not (np.isfinite(Eo_avg) and np.isfinite(Re_avg) and Eo_avg > 0 and Re_avg > 0):
        ax.text(0.5, 0.5, "No valid avg Eo/Re values",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black"))
        savefig(out_png)
        return

    x_pix, y_pix = _grace_xy_from_eo_re([Eo_avg], [Re_avg])

    # Plot point
    sc = ax.scatter(
        x_pix, y_pix,
        s=160,
        alpha=0.95,
        edgecolor="black",
        linewidth=0.9,
        color="tab:red"
    )

    # Figure title (not on image)
    ax.text(
        0.5, 0.985,
        "Grace Diagram (Average Eo / Re)",
        transform=ax.transAxes,
        ha="center", va="top",
        bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.95)
    )

    # Legend OUTSIDE
    legend_label = f"{label}  (Eo={Eo_avg:.3g}, Re={Re_avg:.3g})"
    ax.legend(
        [sc],
        [legend_label],
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0.0,
        framealpha=0.98,
        title="Run"
    )

    # Reserve space for legend
    fig.subplots_adjust(right=0.68)

    savefig(out_png)



def plot_grace_avg_points_all_runs(run_points, out_png, title="Grace Diagram (Avg Eo/Re per run)"):
    """
    Batch/combined plot: ONE point per run on ONE Grace diagram.
    - Each run gets a different color
    - Legend is outside the image (right side)
    run_points: list of dicts like:
        {"label": "circle_bubble", "Eo": 12.3, "Re": 456.7}
    """
    set_thesis_style()
    bg = _load_grace_bg_or_raise()

    # Filter to valid points
    clean = []
    for d in run_points:
        lab = str(d.get("label", "run"))
        eo = d.get("Eo", np.nan)
        re_ = d.get("Re", np.nan)
        if np.isfinite(eo) and np.isfinite(re_) and eo > 0 and re_ > 0:
            clean.append({"label": lab, "Eo": float(eo), "Re": float(re_)})

    fig, ax = plt.subplots(figsize=(10.5, 7.0))  # wider to make room for legend
    ax.imshow(bg)
    ax.axis("off")

    if not clean:
        ax.text(0.5, 0.5, "No valid avg Eo/Re points to plot",
                ha="center", va="center", transform=ax.transAxes,
                bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black"))
        savefig(out_png)
        return

    labels = [d["label"] for d in clean]
    eo = np.array([d["Eo"] for d in clean], float)
    re_ = np.array([d["Re"] for d in clean], float)

    x_pix, y_pix = _grace_xy_from_eo_re(eo, re_)

    # Distinct colors (works well up to ~20-30; beyond that it’s still okay but crowded)
    cmap = plt.get_cmap("tab20")
    colors = [cmap(i % cmap.N) for i in range(len(clean))]

    handles = []
    for xp, yp, col, d in zip(x_pix, y_pix, colors, clean):
        sc = ax.scatter([xp], [yp], s=120, alpha=0.95,
                        edgecolor="black", linewidth=0.7, color=col)

        # Legend text (keep short & readable)
        lab = d["label"]
        lab_txt = f"{lab}  (Eo={d['Eo']:.3g}, Re={d['Re']:.3g})"
        handles.append((sc, lab_txt))

    # Title as a figure-level box (not on the axis text since axis is off)
    ax.text(0.5, 0.985, title, transform=ax.transAxes, ha="center", va="top",
            bbox=dict(boxstyle="round,pad=0.25", fc="white", ec="black", alpha=0.95))

    # Legend OUTSIDE on the right
    legend_handles = [h for (h, _) in handles]
    legend_labels  = [t for (_, t) in handles]

    ax.legend(
        legend_handles, legend_labels,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),   # outside to the right
        borderaxespad=0.0,
        framealpha=0.98,
        title="Runs"
    )

    # Give room for legend on the right
    fig.subplots_adjust(right=0.68)

    savefig(out_png)


# ============================================================
# Plotting 
# ============================================================
def _find_raw_video(base_dir, stem):
    """
    Try to find the raw video corresponding to this run.
    Looks for common naming patterns and falls back to "any mp4 that starts with stem"
    excluding derived videos like _filtered/_pred_mask/_gt_mask.
    """
    candidates = [
        os.path.join(base_dir, stem + ".mp4"),
        os.path.join(base_dir, stem + "_raw.mp4"),
        os.path.join(base_dir, stem + "_input.mp4"),
        os.path.join(base_dir, stem + "_video.mp4"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p

    # fallback: any stem*.mp4 excluding derived outputs
    all_mp4 = sorted(glob.glob(os.path.join(base_dir, stem + "*.mp4")))
    bad_suffixes = ("_filtered.mp4", "_pred_mask.mp4", "_gt_mask.mp4")
    all_mp4 = [p for p in all_mp4 if not any(p.endswith(s) for s in bad_suffixes)]
    return all_mp4[0] if all_mp4 else None


def _read_frame_at(cap, frame_idx):
    if cap is None:
        return None
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
    ok, frame = cap.read()
    return frame if ok else None


def make_qualitative_panel_from_videos(raw_mp4, pred_mask_mp4, filtered_mp4, df_tracked,
                                       out_png, start_frame=0, n_frames=5,
                                       gt_mask_mp4=None, show_gt_row=False):
    """
    Creates ONE PNG with n_frames consecutive frames.
    Rows:
      1) raw
      2) pred mask
      3) filtered overlay (with IDs drawn)
      4) (optional) GT mask if show_gt_row=True and gt_mask_mp4 exists
    """
    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read videos for qualitative panel.")

    if not raw_mp4 or not os.path.exists(raw_mp4):
        raise FileNotFoundError(f"Raw video not found: {raw_mp4}")
    if not pred_mask_mp4 or not os.path.exists(pred_mask_mp4):
        raise FileNotFoundError(f"Pred mask video not found: {pred_mask_mp4}")
    if not filtered_mp4 or not os.path.exists(filtered_mp4):
        raise FileNotFoundError(f"Filtered video not found: {filtered_mp4}")

    if show_gt_row and (not gt_mask_mp4 or not os.path.exists(gt_mask_mp4)):
        show_gt_row = False  # silently disable if missing

    set_thesis_style()

    cap_raw = cv2.VideoCapture(raw_mp4)
    cap_pm  = cv2.VideoCapture(pred_mask_mp4)
    cap_f   = cv2.VideoCapture(filtered_mp4)
    cap_gt  = cv2.VideoCapture(gt_mask_mp4) if show_gt_row else None

    # Choose frames
    frs = list(range(int(start_frame), int(start_frame) + int(n_frames)))

    n_rows = 4 if show_gt_row else 3
    fig, axes = plt.subplots(n_rows, len(frs), figsize=(4.0 * len(frs), 3.4 * n_rows))
    if n_rows == 1:
        axes = np.array([axes])
    if len(frs) == 1:
        axes = axes.reshape(n_rows, 1)

    # Make sure we have the columns we need for labels
    dft = df_tracked.copy()
    if "frame" in dft.columns:
        dft["frame"] = pd.to_numeric(dft["frame"], errors="coerce").astype("Int64")
    if "id" in dft.columns:
        dft["id"] = dft["id"].astype(str)

    for j, fr in enumerate(frs):
        # --- RAW ---
        raw_bgr = _read_frame_at(cap_raw, fr)
        if raw_bgr is not None:
            raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
            axes[0, j].imshow(raw_rgb)
        axes[0, j].set_title(f"Frame {fr}")
        axes[0, j].axis("off")

        # --- PRED MASK ---
        pm_bgr = _read_frame_at(cap_pm, fr)
        if pm_bgr is not None:
            if pm_bgr.ndim == 3:
                pm_gray = cv2.cvtColor(pm_bgr, cv2.COLOR_BGR2GRAY)
            else:
                pm_gray = pm_bgr
            axes[1, j].imshow(pm_gray, cmap="gray", vmin=0, vmax=255)
        axes[1, j].axis("off")

        # --- FILTERED (overlay) + IDs ---
        f_bgr = _read_frame_at(cap_f, fr)
        if f_bgr is not None:
            f_rgb = cv2.cvtColor(f_bgr, cv2.COLOR_BGR2RGB)
            axes[2, j].imshow(f_rgb)

            # Draw IDs on the filtered frame using tracked CSV
            if ("frame" in dft.columns) and ("cx" in dft.columns) and ("cy" in dft.columns) and ("id" in dft.columns):
                d = dft[dft["frame"].astype("Int64") == int(fr)]
                for _, r in d.iterrows():
                    cx = float(r["cx"])
                    cy = float(r["cy"])
                    tid = str(r["id"])
                    axes[2, j].plot(cx, cy, marker="o", markersize=5)
                    axes[2, j].text(cx + 4, cy + 4, tid, fontsize=9, weight="bold",
                                    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
        axes[2, j].axis("off")

        # --- OPTIONAL GT MASK ROW ---
        if show_gt_row:
            gt_bgr = _read_frame_at(cap_gt, fr) if cap_gt is not None else None
            if gt_bgr is not None:
                if gt_bgr.ndim == 3:
                    gt_gray = cv2.cvtColor(gt_bgr, cv2.COLOR_BGR2GRAY)
                else:
                    gt_gray = gt_bgr
                axes[3, j].imshow(gt_gray, cmap="gray", vmin=0, vmax=255)
            axes[3, j].axis("off")

    # Row labels on the left
    row_names = ["Raw", "Pred Mask", "Filtered + IDs"] + (["GT Mask"] if show_gt_row else [])
    for i, name in enumerate(row_names):
        axes[i, 0].set_ylabel(name, fontsize=14, rotation=90, labelpad=18)

    cap_raw.release()
    cap_pm.release()
    cap_f.release()
    if cap_gt is not None:
        cap_gt.release()

    fig.suptitle("Qualitative Detection & Tracking (5 Consecutive Frames)", y=1.01, fontsize=18)
    savefig(out_png)

def plot_delta_r_histogram(dr_steps_px, out_png):
    """
    Histogram of frame-to-frame centroid displacement Δr (pixels).
    Used to define representative Δr for velocity uncertainty propagation.
    """
    set_thesis_style()

    fig, ax = plt.subplots(figsize=(7.2, 5.0))

    dr = np.asarray(dr_steps_px, float)
    dr = dr[np.isfinite(dr)]

    if dr.size == 0:
        ax.text(0.5, 0.5, "No Δr data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    # Robust clipping to avoid one bad ID jump dominating the axis
    dr_plot, lo, hi, n_total, n_kept = clip_for_hist(dr, 1, 99)
    n_clipped = n_total - n_kept

    ax.hist(dr_plot, bins=20, edgecolor="black", linewidth=0.6)
    ax.set_xlabel("Frame-to-frame displacement Δr (px)")
    ax.set_ylabel("Count")
    ax.set_title("Centroid Frame-to-Frame Displacement")

    ax.grid(True)

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(dr):.3g} px",
        f"Mean = {np.mean(dr):.3g} px",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}] px")

    add_statbox(ax, lines)
    savefig(out_png)

def plot_distribution(x, out_png, title, x_label, units, bins=20, legend_outside=False):
    set_thesis_style()
    x = _clean(x)

    # ---- NEW: clip plotted range
    x_plot, lo, hi, n_total, n_kept = clip_for_hist(x, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=(7.8, 5.2))
    if x_plot.size:
        ax.hist(x_plot, bins=bins, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"{x_label} ({units})")
    ax.set_ylabel("Count")

    st = stats_dict(x)  # stats on FULL data (not clipped)
    lines = [
        f"N = {st['N']}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Mean = {st['mean']:.4g} {units}" if st["N"] else "Mean = NA",
        f"Median = {st['median']:.4g} {units}" if st["N"] else "Median = NA",
        f"Std = {st['std']:.4g} {units}" if st["N"] else "Std = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}]")

    #add_statbox(ax, lines)
    savefig(out_png)


def plot_distribution_overlay(series_list, out_png, title, x_label, units, bins=20):
    set_thesis_style()
    fig, ax = plt.subplots(figsize=(10.5, 5.6))  # wider, like your Grace plot
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"{x_label} ({units})")
    ax.set_ylabel("Count")

    allv = np.concatenate([_clean(v) for _, v in series_list if _clean(v).size]) if series_list else np.array([])
    if allv.size == 0:
        ax.text(0.5, 0.5, "No data to overlay", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    lo, hi = robust_limits(allv, 1, 99, 0.10)
    bins_edges = np.linspace(lo, hi, bins + 1)

    # collect handles/labels like your Grace function does
    legend_handles = []
    legend_labels  = []

    for label, v in series_list:
        v = _clean(v)
        if v.size == 0:
            continue

        n, edges, patches = ax.hist(
            v,
            bins=bins_edges,
            histtype="step",
            linewidth=2.0
        )

        # histtype="step" returns a list of Polygon patches; use the first as legend handle
        if patches:
            legend_handles.append(patches[0])
            legend_labels.append(f"{label} (N={v.size})")

    ax.legend(
        legend_handles, legend_labels,
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),   # outside right
        borderaxespad=0.0,
        framealpha=0.98,
        title="Runs"
    )

    # reserve space on the right for legend (same pattern as Grace)
    fig.subplots_adjust(right=0.68)

    savefig(out_png)   # use the same saver as Grace for consistent layout
    plt.close(fig)


def plot_distribution_zoom_full(x, out_png, title, x_label, units, bins_zoom=30, bins_full=55):
    set_thesis_style()
    x = _clean(x)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.2))

    if x.size:
        z0, z1 = robust_limits(x, 1, 99, 0.10)
        ax1.hist(x, bins=bins_zoom, edgecolor="black", linewidth=0.6)
        ax1.set_xlim(z0, z1)
    ax1.grid(True)
    ax1.set_title("Zoomed (1–99th percentile)")
    ax1.set_xlabel(f"{x_label} ({units})")
    ax1.set_ylabel("Count")

    st = stats_dict(x)
    if st["N"]:
        ax1.axvline(st["mean"], linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax1.axvline(st["median"], linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax1.legend(loc="lower left", framealpha=1.0)

    add_statbox(ax1, [
        f"N = {st['N']}",
        f"MAE = {mae(x):.4g} {units}" if st["N"] else "MAE = NA",
        f"Bias = {bias(x):.4g} {units}" if st["N"] else "Bias = NA",
        f"Std = {st['std']:.4g} {units}" if st["N"] else "Std = NA",
    ])

    if x.size:
        ax2.hist(x, bins=bins_full, edgecolor="black", linewidth=0.6)
    ax2.grid(True)
    ax2.set_title("Full range")
    ax2.set_xlabel(f"{x_label} ({units})")
    ax2.set_ylabel("Count")
    if st["N"]:
        ax2.axvline(st["mean"], linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax2.axvline(st["median"], linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax2.legend(loc="lower left", framealpha=1.0)

    fig.suptitle(title, y=1.02, fontsize=18)
    savefig(out_png)

def plot_measured_vs_true(x_true, y_meas, out_png, title, units):
    set_thesis_style()
    x_true = np.asarray(x_true, float)
    y_meas = np.asarray(y_meas, float)
    m = np.isfinite(x_true) & np.isfinite(y_meas)
    x_true, y_meas = x_true[m], y_meas[m]

    fig, ax = plt.subplots(figsize=(7.4, 5.8))
    if x_true.size:
        ax.scatter(x_true, y_meas, s=30, alpha=0.8, edgecolor="black", linewidth=0.35, label="Matched samples")
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(f"True ({units})")
    ax.set_ylabel(f"Measured ({units})")

    if x_true.size:
        xmin = float(min(np.min(x_true), np.min(y_meas)))
        xmax = float(max(np.max(x_true), np.max(y_meas)))
        if xmin == xmax:
            xmin -= 1
            xmax += 1
        ax.plot([xmin, xmax], [xmin, xmax], linestyle="--", linewidth=2, label="1:1 line")

    err = y_meas - x_true
    add_statbox(ax, [
        f"N = {int(np.isfinite(err).sum())}",
        f"MAE = {mae(err):.4g} {units}",
        f"RMSE = {rmse(err):.4g} {units}",
        f"Bias = {bias(err):.4g} {units}",
    ])
    ax.legend(loc="lower left", framealpha=1.0)
    savefig(out_png)
    


def plot_velocity_vs_diameter(diam, vel, out_png, units_len, units_vel):
    set_thesis_style()
    diam = np.asarray(diam, float)
    vel  = np.asarray(vel, float)
    m = np.isfinite(diam) & np.isfinite(vel)
    diam, vel = diam[m], vel[m]

    fig, ax = plt.subplots(figsize=(7.8, 5.8))
    if diam.size:
        ax.scatter(diam, vel, s=30, alpha=0.75, edgecolor="black", linewidth=0.3, label="Samples")
    ax.grid(True)
    ax.set_title("Rise Velocity vs Diameter (Measured)")
    ax.set_xlabel(f"Diameter ({units_len})")
    ax.set_ylabel(f"Rise velocity ({units_vel})")

    if diam.size >= 8:
        nbins = min(10, max(5, int(np.sqrt(diam.size))))
        bins = np.linspace(np.min(diam), np.max(diam), nbins + 1)
        mids = 0.5 * (bins[:-1] + bins[1:])
        meds = []
        for i in range(nbins):
            sel = (diam >= bins[i]) & (diam < bins[i+1] if i < nbins-1 else diam <= bins[i+1])
            meds.append(np.median(vel[sel]) if np.sum(sel) else np.nan)
        meds = np.asarray(meds, float)
        ok = np.isfinite(meds)
        if np.any(ok):
            ax.plot(mids[ok], meds[ok], linestyle="--", linewidth=2.5, color="darkorange", label="Binned median")

    add_statbox(ax, [
        f"N = {diam.size}",
        f"Diam median = {np.median(diam):.4g} {units_len}" if diam.size else "Diam median = NA",
        f"Vel median = {np.median(vel):.4g} {units_vel}" if vel.size else "Vel median = NA",
    ])
    ax.legend(loc="upper left", framealpha=1.0)
    savefig(out_png)

def plot_track_lengths(track_len_frames, fps, out_png):
    set_thesis_style()
    L = np.asarray(track_len_frames, float)
    L = L[np.isfinite(L)]
    dur = (L / fps) if (fps and fps > 0) else None

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.2, 5.2))

    if L.size:
        ax1.hist(L, bins=20, edgecolor="black", linewidth=0.6)
    ax1.grid(True)
    ax1.set_title("Track length distribution")
    ax1.set_xlabel("Track length (frames)")
    ax1.set_ylabel("Count")
    st = stats_dict(L)
    add_statbox(ax1, [
        f"N tracks = {st['N']}",
        f"Mean = {st['mean']:.4g} frames" if st["N"] else "Mean = NA",
        f"Median = {st['median']:.4g} frames" if st["N"] else "Median = NA",
        f"Std = {st['std']:.4g} frames" if st["N"] else "Std = NA",
    ])

    if dur is not None:
        if dur.size:
            ax2.hist(dur, bins=20, edgecolor="black", linewidth=0.6)
        ax2.grid(True)
        ax2.set_title("Track duration distribution")
        ax2.set_xlabel("Track duration (s)")
        ax2.set_ylabel("Count")
        st2 = stats_dict(dur)
        add_statbox(ax2, [
            f"N tracks = {st2['N']}",
            f"Mean = {st2['mean']:.4g} s" if st2["N"] else "Mean = NA",
            f"Median = {st2['median']:.4g} s" if st2["N"] else "Median = NA",
            f"Std = {st2['std']:.4g} s" if st2["N"] else "Std = NA",
        ])
    else:
        ax2.axis("off")
        ax2.text(0.5, 0.5, "FPS not available\n(duration plot skipped)", ha="center", va="center")

    savefig(out_png)

def plot_error_vs_time_timecolored(x, err, out_png, title, xlab, units_y):
    set_thesis_style()
    x = np.asarray(x, float)
    e = np.asarray(err, float)
    m = np.isfinite(x) & np.isfinite(e)
    x, e = x[m], e[m]

    fig, ax = plt.subplots(figsize=(7.8, 5.4))
    if x.size:
        sc = ax.scatter(
            x, e,
            c=x, cmap="viridis",
            s=26, alpha=0.85,
            edgecolor="black", linewidth=0.25,
            label="Error samples"
        )
        cbar = plt.colorbar(sc, ax=ax, pad=0.02)
        cbar.set_label(xlab)

    ax.axhline(0, linestyle="--", linewidth=2, label="Zero error")
    ax.grid(True)
    ax.set_title(clean_plot_title(title))
    ax.set_xlabel(xlab)
    ax.set_ylabel(f"Error ({units_y})")

    st = stats_dict(e)
    add_statbox(
        ax,
        [
            f"N = {st['N']}",
            f"MAE = {mae(e):.4g} {units_y}",
            f"RMSE = {rmse(e):.4g} {units_y}",
            f"Bias = {bias(e):.4g} {units_y}",
        ],
        loc="lower right"
    )

    ax.legend(loc="lower left", framealpha=1.0)
    savefig(out_png)

def plot_trajectories_2d_timecolored(df, out_png, units_len, max_tracks=40, min_points=3):
    set_thesis_style()
    if df is None or df.empty:
        fig, ax = plt.subplots(figsize=(8.2, 6.0))
        ax.set_title("2D Trajectories Colored by Time")
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    xcol = "x_in" if "x_in" in df.columns else "cx"
    ycol = "y_in" if "y_in" in df.columns else "cy"

    has_time = ("t_s" in df.columns) and df["t_s"].notna().any()
    tcol = "t_s" if has_time else "frame"
    xlab = f"x ({units_len})" if xcol == "x_in" else "x (px)"
    ylab = f"y ({units_len})" if ycol == "y_in" else "y (px)"

    lens = df.groupby("id").size().sort_values(ascending=False)
    top_ids = list(lens.head(min(max_tracks, len(lens))).index)

    dsel = df[df["id"].isin(top_ids)].copy()
    dsel[tcol] = pd.to_numeric(dsel[tcol], errors="coerce")
    tvals = dsel[tcol].to_numpy(float)
    tvals = tvals[np.isfinite(tvals)]
    if tvals.size == 0:
        tmin, tmax = 0.0, 1.0
    else:
        tmin, tmax = float(np.min(tvals)), float(np.max(tvals))
        if tmin == tmax:
            tmax = tmin + 1.0

    norm = mpl.colors.Normalize(vmin=tmin, vmax=tmax)
    cmap = plt.get_cmap("viridis")

    fig, ax = plt.subplots(figsize=(8.6, 6.3))

    drawn = 0
    for tid in top_ids:
        d = df[df["id"] == tid].sort_values(tcol)
        x = pd.to_numeric(d[xcol], errors="coerce").to_numpy(float)
        y = pd.to_numeric(d[ycol], errors="coerce").to_numpy(float)
        t = pd.to_numeric(d[tcol], errors="coerce").to_numpy(float)

        m = np.isfinite(x) & np.isfinite(y) & np.isfinite(t)
        x, y, t = x[m], y[m], t[m]
        if len(x) < min_points:
            continue

        pts = np.column_stack([x, y])
        segs = np.stack([pts[:-1], pts[1:]], axis=1)

        lc = LineCollection(segs, cmap=cmap, norm=norm, linewidth=2.0, alpha=0.95)
        lc.set_array(t[:-1])
        ax.add_collection(lc)

        ax.scatter([x[0]], [y[0]], s=22, edgecolor="black", linewidth=0.25, marker="o")
        ax.scatter([x[-1]], [y[-1]], s=28, edgecolor="black", linewidth=0.25, marker="s")
        drawn += 1

    ax.autoscale()
    ax.grid(True)
    ax.set_title("Representative Bubble Trajectories (2D, Colored by Time)")
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.invert_yaxis()

    if drawn == 0:
        ax.text(0.5, 0.5, "No valid tracks (too short / missing time)", ha="center", va="center",
                transform=ax.transAxes, bbox=dict(boxstyle="round", fc="white", ec="black"))

    sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("Time (s)" if has_time else "Frame")

    savefig(out_png)

def plot_velocity_vs_time(df, out_png, units_vel):
    set_thesis_style()

    if df is None or df.empty or (("vel_px_s" not in df.columns) and ("vel_in_s" not in df.columns)):
        fig, ax = plt.subplots(figsize=(7.8, 5.4))
        ax.text(0.5, 0.5, "No velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    has_time = "t_s" in df.columns and df["t_s"].notna().any()
    tcol = "t_s" if has_time else "frame"
    xlabel = "Time (s)" if has_time else "Frame"

    fig, ax = plt.subplots(figsize=(7.8, 5.4))

    lens = df.groupby("id").size().sort_values(ascending=False)
    ids = lens.head(6).index

    for tid in ids:
        d = df[df["id"] == tid]
        ax.plot(
            d[tcol],
            d["vel_px_s"] if units_vel == "px/s" else d["vel_in_s"],
            linewidth=2,
            alpha=0.9
        )

    ax.grid(True)
    ax.set_title("Bubble Rise Velocity vs Time (Representative Tracks)")
    ax.set_xlabel(xlabel)
    ax.set_ylabel(f"Rise velocity ({units_vel})")

    savefig(out_png)

def plot_velocity_smoothness(df, out_png, units_vel):
    set_thesis_style()

    if df is None or df.empty or "vel_px_s" not in df.columns:
        fig, ax = plt.subplots(figsize=(7.2, 5.0))
        ax.text(0.5, 0.5, "No velocity data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    dv_all = []
    for _, d in df.groupby("id"):
        v = d["vel_px_s"].to_numpy(float)
        v = v[np.isfinite(v)]
        if len(v) >= 2:
            dv_all.extend(np.abs(np.diff(v)))

    dv_all = np.asarray(dv_all, float)

    # ---- clip plotting range to avoid one giant outlier ruining the axis
    dv_plot, lo, hi, n_total, n_kept = clip_for_hist(dv_all, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    if dv_plot.size:
        ax.hist(dv_plot, bins=20, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title("Velocity Change Between Frames (Trajectory Smoothness)")
    ax.set_xlabel(f"|Δv| ({units_vel})")
    ax.set_ylabel("Count")

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(dv_all):.4g} {units_vel}" if n_total else "Median = NA",
        f"Mean = {np.mean(dv_all):.4g} {units_vel}" if n_total else "Mean = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.3g}, {hi:.3g}]")

    add_statbox(ax, lines)
    savefig(out_png)


def plot_relative_error(err, true, out_png):
    set_thesis_style()

    err = np.asarray(err, float)
    true = np.asarray(true, float)

    m = np.isfinite(err) & np.isfinite(true) & (true != 0)
    rel_pct = (err[m] / true[m]) * 100.0  # percent

    # ---- NEW: clip plotted percent range
    rel_plot, lo, hi, n_total, n_kept = clip_for_hist(rel_pct, 1, 99)
    n_clipped = n_total - n_kept

    fig, ax = plt.subplots(figsize=(7.2, 5.0))
    if rel_plot.size:
        ax.hist(rel_plot, bins=20, edgecolor="black", linewidth=0.6)

    ax.grid(True)
    ax.set_title("Relative Diameter Error Distribution")
    ax.set_xlabel("Relative error (%)")
    ax.set_ylabel("Count")
    ax.axvline(0, linestyle="--", linewidth=2)

    lines = [
        f"N = {n_kept}" + (f" (clipped {n_clipped})" if n_clipped > 0 else ""),
        f"Median = {np.median(rel_pct):.2f} %" if n_total else "Median = NA",
        f"Mean = {np.mean(rel_pct):.2f} %" if n_total else "Mean = NA",
    ]
    if n_clipped > 0 and np.isfinite(lo) and np.isfinite(hi):
        lines.append(f"Shown range = [{lo:.2f}%, {hi:.2f}%]")

    add_statbox(ax, lines)
    savefig(out_png)



# ============================================================
# Mask IoU (semantic)
# ============================================================
def compute_iou(mask_pred, mask_true):
    inter = np.logical_and(mask_pred, mask_true).sum()
    union = np.logical_or(mask_pred, mask_true).sum()
    return inter / union if union > 0 else np.nan

def _read_mask_frame(cap):
    ok, frame = cap.read()
    if not ok:
        return None
    if frame.ndim == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return frame

def _binarize_mask(img_gray, thresh=128):
    if img_gray is None:
        return None
    return (img_gray >= thresh)

def compute_iou_video_semantic(gt_mask_mp4, pred_mask_mp4, out_csv,
                               thresh=128, max_frames=None,
                               crop_top=0, crop_bottom=0):
    """
    Semantic IoU per frame between GT mask video and Pred mask video.
    - GT video: white bubble(s) on black background
    - Pred video: binary/semantic mask video (NOT overlay video)
    """
    if not _HAS_CV2:
        raise RuntimeError("cv2 not available; cannot read mask videos.")

    if not (gt_mask_mp4 and os.path.exists(gt_mask_mp4)):
        return (np.nan, np.nan, 0)
    if not (pred_mask_mp4 and os.path.exists(pred_mask_mp4)):
        return (np.nan, np.nan, 0)

    cap_gt = cv2.VideoCapture(gt_mask_mp4)
    cap_pr = cv2.VideoCapture(pred_mask_mp4)

    rows = []
    k = 0
    while True:
        gt_gray = _read_mask_frame(cap_gt)
        pr_gray = _read_mask_frame(cap_pr)
        if gt_gray is None or pr_gray is None:
            break

        if pr_gray.shape != gt_gray.shape:
            pr_gray = cv2.resize(pr_gray, (gt_gray.shape[1], gt_gray.shape[0]), interpolation=cv2.INTER_NEAREST)

        if crop_top or crop_bottom:
            H = gt_gray.shape[0]
            y0 = int(crop_top)
            y1 = int(H - crop_bottom) if crop_bottom else H
            gt_gray = gt_gray[y0:y1, :]
            pr_gray = pr_gray[y0:y1, :]

        gt_bin = _binarize_mask(gt_gray, thresh=thresh)
        pr_bin = _binarize_mask(pr_gray, thresh=thresh)

        iou = compute_iou(pr_bin, gt_bin)
        rows.append({"frame_idx": k, "iou": float(iou) if np.isfinite(iou) else np.nan})

        k += 1
        if max_frames is not None and k >= int(max_frames):
            break

    cap_gt.release()
    cap_pr.release()

    df_iou = pd.DataFrame(rows)
    ensure_dir(os.path.dirname(out_csv))
    df_iou.to_csv(out_csv, index=False)

    v = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    v = v[np.isfinite(v)]
    return (float(np.mean(v)) if v.size else np.nan,
            float(np.median(v)) if v.size else np.nan,
            int(v.size))

def plot_iou_vs_time(df_iou, out_png, fps=None):
    set_thesis_style()
    if df_iou is None or df_iou.empty or "iou" not in df_iou.columns:
        fig, ax = plt.subplots(figsize=(7.8, 5.2))
        ax.text(0.5, 0.5, "No IoU data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    x = pd.to_numeric(df_iou.get("frame_idx", df_iou.index), errors="coerce").to_numpy(float)
    y = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    m = np.isfinite(x) & np.isfinite(y)
    x, y = x[m], y[m]

    if fps and fps > 0:
        x_plot = x / fps
        xlab = "Time (s)"
    else:
        x_plot = x
        xlab = "Frame"

    fig, ax = plt.subplots(figsize=(8.4, 5.4))
    if len(x_plot):
        ax.plot(x_plot, y, linewidth=1.8)
    ax.grid(True)
    ax.set_title("Segmentation IoU vs Time/Frame")
    ax.set_xlabel(xlab)
    ax.set_ylabel("IoU")

    yy = y[np.isfinite(y)]
    if yy.size: 
        q1, q3 = np.percentile(yy, [25, 75])
        add_statbox(ax, [
            f"N = {yy.size}",
            f"Mean = {np.mean(yy):.3f}",
            f"Median = {np.median(yy):.3f}",
            f"IQR = {q3-q1:.3f}",
            f"Min/Max = {np.min(yy):.3f} / {np.max(yy):.3f}",
        ], loc="lower right")

    savefig(out_png)

def plot_iou_histogram(df_iou, out_png, bins=20):
    set_thesis_style()
    if df_iou is None or df_iou.empty or "iou" not in df_iou.columns:
        fig, ax = plt.subplots(figsize=(7.8, 5.2))
        ax.text(0.5, 0.5, "No IoU data", ha="center", va="center", transform=ax.transAxes)
        savefig(out_png)
        return

    y = pd.to_numeric(df_iou["iou"], errors="coerce").to_numpy(float)
    y = y[np.isfinite(y)]

    fig, ax = plt.subplots(figsize=(7.8, 5.2))
    if y.size:
        ax.hist(y, bins=bins, edgecolor="black", linewidth=0.6)
        ax.axvline(np.mean(y), linestyle="--", linewidth=2, color="crimson", label="Mean")
        ax.axvline(np.median(y), linestyle=":", linewidth=2, color="darkorange", label="Median")
        ax.legend(loc="lower left", framealpha=1.0)

    ax.grid(True)
    ax.set_title("Segmentation IoU Distribution")
    ax.set_xlabel("IoU")
    ax.set_ylabel("Count")

    # ---- NEW: force x-range
    ax.set_xlim(0.0, 1.0)

    savefig(out_png)


# ============================================================
# MOT metrics (UNCHANGED)
# ============================================================
def compute_mot_metrics(df_gt, df_m, max_dist_px):
    df_gt = df_gt.copy()
    df_m  = df_m.copy()
    df_gt["frame"] = pd.to_numeric(df_gt["frame"], errors="coerce").astype("Int64")
    df_m["frame"]  = pd.to_numeric(df_m["frame"], errors="coerce").astype("Int64")
    df_gt["gt_id"] = df_gt["gt_id"].astype(str)
    df_m["id"]     = df_m["id"].astype(str)

    frames = sorted(set(df_gt["frame"].dropna().astype(int)) | set(df_m["frame"].dropna().astype(int)))

    FP = 0
    FN = 0
    IDSW = 0
    dist_sum = 0.0
    match_count = 0
    gt_total = 0

    last_match_for_gt = {}
    pair_hits = {}
    gt_hits = {}
    trk_hits = {}

    for fr in frames:
        gtf = df_gt[df_gt["frame"].astype(int) == fr].dropna(subset=["gt_cx_px","gt_cy_px","gt_id"])
        mf  = df_m[df_m["frame"].astype(int) == fr].dropna(subset=["cx","cy","id"])

        gt_total += len(gtf)

        if len(gtf) == 0 and len(mf) == 0:
            continue
        if len(gtf) == 0:
            FP += len(mf)
            continue
        if len(mf) == 0:
            FN += len(gtf)
            continue

        gtf2 = pd.DataFrame({
            "gt_cx_px": pd.to_numeric(gtf["gt_cx_px"], errors="coerce"),
            "gt_cy_px": pd.to_numeric(gtf["gt_cy_px"], errors="coerce"),
        })
        mf2 = pd.DataFrame({
            "cx": pd.to_numeric(mf["cx"], errors="coerce"),
            "cy": pd.to_numeric(mf["cy"], errors="coerce"),
        })

        pairs = match_frame(gtf2, mf2, max_dist_px=max_dist_px)

        matched_gt_idx = set()
        matched_m_idx  = set()

        for gi, mi, dist in pairs:
            matched_gt_idx.add(gi)
            matched_m_idx.add(mi)

            dist_sum += dist
            match_count += 1

            gt_id = str(gtf.iloc[gi]["gt_id"])
            trk_id = str(mf.iloc[mi]["id"])

            if gt_id in last_match_for_gt and last_match_for_gt[gt_id] != trk_id:
                IDSW += 1
            last_match_for_gt[gt_id] = trk_id

            pair_hits[(gt_id, trk_id)] = pair_hits.get((gt_id, trk_id), 0) + 1
            gt_hits[gt_id] = gt_hits.get(gt_id, 0) + 1
            trk_hits[trk_id] = trk_hits.get(trk_id, 0) + 1

        FN += (len(gtf) - len(matched_gt_idx))
        FP += (len(mf)  - len(matched_m_idx))

    MOTP = (dist_sum / match_count) if match_count > 0 else np.nan
    MOTA = 1.0 - (FN + FP + IDSW) / gt_total if gt_total > 0 else np.nan

    gt_ids = list(gt_hits.keys())
    trk_ids = list(trk_hits.keys())

    if len(gt_ids) == 0 or len(trk_ids) == 0:
        IDF1 = np.nan
        IDTP = 0
        IDFP = FP
        IDFN = FN
    else:
        H = np.zeros((len(gt_ids), len(trk_ids)), dtype=float)
        gt_index = {g:i for i,g in enumerate(gt_ids)}
        tr_index = {t:j for j,t in enumerate(trk_ids)}
        for (g, t), cnt in pair_hits.items():
            if g in gt_index and t in tr_index:
                H[gt_index[g], tr_index[t]] = cnt

        r, c = linear_sum_assignment(-H)

        IDTP = float(H[r, c].sum())
        total_trk_matched = float(sum(trk_hits.values()))
        total_gt_matched  = float(sum(gt_hits.values()))

        IDFP = total_trk_matched - IDTP
        IDFN = total_gt_matched  - IDTP

        IDF1 = (2*IDTP) / (2*IDTP + IDFP + IDFN) if (2*IDTP + IDFP + IDFN) > 0 else np.nan

    return {
        "MOTA": float(MOTA) if np.isfinite(MOTA) else np.nan,
        "MOTP_px": float(MOTP) if np.isfinite(MOTP) else np.nan,
        "IDF1": float(IDF1) if np.isfinite(IDF1) else np.nan,
        "FP": int(FP),
        "FN": int(FN),
        "IDSW": int(IDSW),
        "GT_total": int(gt_total),
        "matches": int(match_count),
    }

# ============================================================
# Optional manual experimental cross-check (runs ONLY if *_manual.csv exists)
# ============================================================
def compare_manual_auto(df_manual, df_auto):
    err_d = df_auto["diam"] - df_manual["diam"]
    err_v = df_auto["vel"] - df_manual["vel"]

    return {
        "diam_mean_err": np.mean(err_d),
        "diam_std_err": np.std(err_d, ddof=1),
        "diam_rmse": rmse(err_d),
        "vel_mean_err": np.mean(err_v),
        "vel_std_err": np.std(err_v, ddof=1),
        "vel_rmse": rmse(err_v),
        "N": len(err_d)
    }

def _try_manual_crosscheck(manual_csv, df_auto, out_dir, units_len, units_vel):
    if not manual_csv or (not os.path.exists(manual_csv)):
        return

    try:
        df_m = pd.read_csv(manual_csv)
    except Exception as e:
        log_error(out_dir, f"[MANUAL READ FAIL] {type(e).__name__}: {e}")
        return

    fcol = pick_col(df_m, ["frame", "Frame"])
    dcol = pick_col(df_m, ["diam", "diam_in", "diam_px", "D_manual", "D"])
    vcol = pick_col(df_m, ["vel", "vel_in_s", "vel_px_s", "v_manual", "v"])

    if fcol is None or dcol is None or vcol is None:
        log_error(out_dir, f"[MANUAL SKIP] manual csv missing required cols. Found: {list(df_m.columns)}")
        return

    df_m = df_m.copy()
    df_m["frame"] = pd.to_numeric(df_m[fcol], errors="coerce").astype("Int64")
    df_m["diam"]  = pd.to_numeric(df_m[dcol], errors="coerce")
    df_m["vel"]   = pd.to_numeric(df_m[vcol], errors="coerce")

    if "frame" not in df_auto.columns or "diam_cmp" not in df_auto.columns or "vel_cmp" not in df_auto.columns:
        log_error(out_dir, "[MANUAL SKIP] auto dataframe missing diam_cmp/vel_cmp.")
        return

    ma = df_m.groupby("frame")[["diam", "vel"]].mean().reset_index()
    au = df_auto.groupby("frame")[["diam_cmp", "vel_cmp"]].mean().reset_index()

    df_join = pd.merge(ma, au, on="frame", how="inner")
    if df_join.empty:
        log_error(out_dir, "[MANUAL SKIP] no overlapping frames between manual and auto.")
        return

    df_manual2 = pd.DataFrame({
        "diam": df_join["diam"].to_numpy(float),
        "vel":  df_join["vel"].to_numpy(float),
    })
    df_auto2 = pd.DataFrame({
        "diam": df_join["diam_cmp"].to_numpy(float),
        "vel":  df_join["vel_cmp"].to_numpy(float),
    })

    summary = compare_manual_auto(df_manual2, df_auto2)
    tabs = ensure_dir(os.path.join(out_dir, "tables"))
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "manual_vs_auto_summary.csv"), index=False)
    df_join.to_csv(os.path.join(tabs, "manual_vs_auto_joined_by_frame.csv"), index=False)

# ============================================================
# Core evaluation: NO GT 
# ============================================================
def evaluate_no_gt(tracked_csv, out_dir, avg_csv=None, frames_dir=None, manual_csv=None):
    ensure_dir(out_dir)
    figs = ensure_dir(os.path.join(out_dir, "figs"))
    tabs = ensure_dir(os.path.join(out_dir, "tables"))

    df = pd.read_csv(tracked_csv)
    m = map_tracked(df)

    df["frame"] = pd.to_numeric(df[m["frame"]], errors="coerce").astype("Int64")
    df["id"]    = df[m["id"]].astype(str)
    df["cx"]    = pd.to_numeric(df[m["cx"]], errors="coerce")
    df["cy"]    = pd.to_numeric(df[m["cy"]], errors="coerce")
    df["diam_px"] = pd.to_numeric(df[m["diam"]], errors="coerce")

    if m["spd"]:
        df["vel_px_s"] = pd.to_numeric(df[m["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df[m["vx"]], errors="coerce") if m["vx"] else np.nan
        vy = pd.to_numeric(df[m["vy"]], errors="coerce") if m["vy"] else np.nan
        df["vel_px_s"] = np.hypot(vx, vy)

    settings, df_avg = (None, None)
    if avg_csv and os.path.exists(avg_csv):
        try:
            settings, df_avg = read_averages_with_settings(avg_csv, header_line_1_indexed=25)
        except Exception as e:
            log_error(out_dir, f"[AVG READ FAIL] {type(e).__name__}: {e}")
            settings, df_avg = None, None

    scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
    fps = infer_fps(settings=settings)

    units_len = "in" if scale_in_per_px else "px"
    units_vel = "in/s" if scale_in_per_px else "px/s"
    units_len_fname = fname_units(units_len)
    units_vel_fname = fname_units(units_vel)

    if fps and fps > 0 and df["frame"].notna().any():
        df["t_s"] = df["frame"].astype(float) / fps
    else:
        df["t_s"] = np.nan

    if scale_in_per_px:
        df["diam_in"]  = df["diam_px"] * scale_in_per_px
        df["vel_in_s"] = df["vel_px_s"] * scale_in_per_px
        df["x_in"]     = df["cx"] * scale_in_per_px
        df["y_in"]     = df["cy"] * scale_in_per_px

    track_len = df.groupby("id").size()
    pd.DataFrame({"id": track_len.index, "track_len_frames": track_len.values}) \
        .to_csv(os.path.join(tabs, "track_stats.csv"), index=False)

    diam = df["diam_in"].values if scale_in_per_px else df["diam_px"].values
    vel  = df["vel_in_s"].values if scale_in_per_px else df["vel_px_s"].values

    # centroid uncertainty
    centroid_u_xy_px, centroid_u_xy_in = run_centroid_uncertainty(
        df, tabs, out_dir, fps, scale_in_per_px
    )

    # diameter (segmentation) uncertainty (NO GT, precision-based)
    diam_u_D_px, diam_u_D_in = run_diameter_uncertainty(
        df, tabs, out_dir, fps, scale_in_per_px
    )
    
    # (Optional) keep Δr histogram for “motion statistics” only, not uncertainty
    delta_r_med_px, delta_r_mean_px, n_steps, dr_steps = compute_delta_r_stats(
        df[["id", "frame", "cx", "cy"]].copy(),
        require_consecutive=True,
        max_step_px=100
    )
    plot_delta_r_histogram(dr_steps, os.path.join(figs, "fig_delta_r_histogram.png"))
    pd.DataFrame([{
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "require_consecutive": True,
        "max_step_px": 100,
        "note": "Δr is reported as motion statistic; not used in uncertainty propagation"
    }]).to_csv(os.path.join(tabs, "delta_r_summary.csv"), index=False)

    # --- Video-based qualitative panel (no frames_dir needed) ---
    stem = os.path.basename(tracked_csv).replace("_tracked.csv", "")
    base = os.path.dirname(tracked_csv)
    
    raw_mp4       = _find_raw_video(base, stem)
    pred_mask_mp4 = os.path.join(base, stem + "_pred_mask.mp4")
    filtered_mp4  = os.path.join(base, stem + "_filtered.mp4")
    
    # ---- choose start frame near the middle of tracked data ----
    if df["frame"].notna().any():
        fmin = int(df["frame"].dropna().min())
        fmax = int(df["frame"].dropna().max())
        start_frame = max(fmin, (fmin + fmax)//2 - 2)  # centers a 5-frame window
    else:
        start_frame = 0
    
    safe_plot(
        out_dir,
        "fig_qualitative_sequence",
        os.path.join(figs, "fig_qualitative_sequence.png"),
        lambda: make_qualitative_panel_from_videos(
            raw_mp4=raw_mp4,
            pred_mask_mp4=pred_mask_mp4,
            filtered_mp4=filtered_mp4,
            df_tracked=df,
            out_png=os.path.join(figs, "fig_qualitative_sequence.png"),
            start_frame=start_frame,
            n_frames=5,
            show_gt_row=False
        )
    )

    safe_plot(out_dir, "fig_diameter_hist",
              os.path.join(figs, f"fig_diameter_hist_{units_len_fname}.png"),
              lambda: plot_distribution(
                  diam, os.path.join(figs, f"fig_diameter_hist_{units_len_fname}.png"),
                  "Bubble Diameter Distribution (Measured)", "Equivalent diameter", units_len))

    safe_plot(out_dir, "fig_velocity_hist",
              os.path.join(figs, f"fig_velocity_hist_{units_vel_fname}.png"),
              lambda: plot_distribution(
                  vel, os.path.join(figs, f"fig_velocity_hist_{units_vel_fname}.png"),
                  "Bubble Rise Velocity Distribution (Measured)", "Rise velocity", units_vel))

    safe_plot(out_dir, "fig_track_lengths",
              os.path.join(figs, "fig_track_lengths.png"),
              lambda: plot_track_lengths(track_len.values, fps, os.path.join(figs, "fig_track_lengths.png")))

    safe_plot(out_dir, "fig_trajectories_2d_timecolored",
              os.path.join(figs, "fig_trajectories_2d_timecolored.png"),
              lambda: plot_trajectories_2d_timecolored(
                  df, os.path.join(figs, "fig_trajectories_2d_timecolored.png"), units_len))

    safe_plot(out_dir, "fig_velocity_vs_diameter",
              os.path.join(figs, f"fig_velocity_vs_diameter_{units_len_fname}.png"),
              lambda: plot_velocity_vs_diameter(
                  diam, vel, os.path.join(figs, f"fig_velocity_vs_diameter_{units_len_fname}.png"),
                  units_len, units_vel))

    safe_plot(out_dir, "fig_velocity_vs_time",
              os.path.join(figs, f"fig_velocity_vs_time_{units_vel_fname}.png"),
              lambda: plot_velocity_vs_time(
                  df, os.path.join(figs, f"fig_velocity_vs_time_{units_vel_fname}.png"),
                  units_vel))

    safe_plot(out_dir, "fig_velocity_smoothness",
              os.path.join(figs, f"fig_velocity_smoothness_{units_vel_fname}.png"),
              lambda: plot_velocity_smoothness(
                  df, os.path.join(figs, f"fig_velocity_smoothness_{units_vel_fname}.png"),
                  units_vel))
   
    safe_plot(out_dir, "fig_grace_avg_point",
          os.path.join(figs, "fig_grace_avg_point.png"),
          lambda: plot_grace_avg_point(df, os.path.join(figs, "fig_grace_avg_point.png"),
                                       df_avg=df_avg, label="Auto (no GT)", annotate=True))


    sd = stats_dict(diam)
    sv = stats_dict(vel)
    summary = {
        "mode": "NO_GT",
        "N_tracks": int(track_len.size),
        "N_samples": int(np.isfinite(diam).sum()),
        "scale_in_per_px": scale_in_per_px if scale_in_per_px else "",
        "fps": fps if fps else "",
        f"diam_mean_{units_len}": sd["mean"],
        f"diam_median_{units_len}": sd["median"],
        f"diam_std_{units_len}": sd["std"],
        f"vel_mean_{units_vel}": sv["mean"],
        f"vel_median_{units_vel}": sv["median"],
        f"vel_std_{units_vel}": sv["std"],
        "mean_track_len_frames": float(track_len.mean()) if track_len.size else np.nan,
        "centroid_u_xy_px": centroid_u_xy_px,
        "centroid_u_xy_in": centroid_u_xy_in if scale_in_per_px else "",
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "units_len": units_len,
        "units_vel": units_vel,
        "diameter_u_D_px": diam_u_D_px,
        "diameter_u_D_in": diam_u_D_in if scale_in_per_px else "",
    }
    
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "run_summary.csv"), index=False)
    return summary


# ============================================================
# Core evaluation: WITH GT 
# ============================================================
def evaluate_with_gt(gt_csv, tracked_csv, out_dir, max_dist_px, avg_csv=None, frames_dir=None, manual_csv=None):
    ensure_dir(out_dir)
    figs = ensure_dir(os.path.join(out_dir, "figs"))
    tabs = ensure_dir(os.path.join(out_dir, "tables"))

    df_gt = pd.read_csv(gt_csv)
    df_m  = pd.read_csv(tracked_csv)

    gtm = map_gt(df_gt)
    trm = map_tracked(df_m)

    settings, df_avg = (None, None)
    if avg_csv and os.path.exists(avg_csv):
        try:
            settings, df_avg = read_averages_with_settings(avg_csv, header_line_1_indexed=25)
        except Exception as e:
            log_error(out_dir, f"[AVG READ FAIL] {type(e).__name__}: {e}")
            settings, df_avg = None, None

    ignore_top = infer_ignore_top_px(settings)
    bottom_bar = BOTTOM_BAR_PX

    scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
    fps = infer_fps(settings=settings)

    units_len = "in" if scale_in_per_px else "px"
    units_vel = "in/s" if scale_in_per_px else "px/s"
    units_len_fname = fname_units(units_len)
    units_vel_fname = fname_units(units_vel)

    # --- Build mask video paths early (used for frame_h + IoU) ---
    stem = os.path.basename(tracked_csv).replace("_tracked.csv", "")
    base = os.path.dirname(tracked_csv)

    gt_mask_mp4   = os.path.join(base, stem + "_gt_mask.mp4")
    pred_mask_mp4 = os.path.join(base, stem + "_pred_mask.mp4")  # binary masks (correct for IoU)

    # IMPORTANT: overlay video exists but is NOT used for IoU
    filtered_mp4  = os.path.join(base, stem + "_filtered.mp4")

    # --- Determine frame height BEFORE ROI filtering ---
    frame_h = None
    if _HAS_CV2 and os.path.exists(gt_mask_mp4):
        cap0 = cv2.VideoCapture(gt_mask_mp4)
        ok0, fr0 = cap0.read()
        cap0.release()
        if ok0 and fr0 is not None:
            frame_h = fr0.shape[0]

    # --- Standardize GT dataframe ---
    df_gt["frame"] = pd.to_numeric(df_gt[gtm["frame"]], errors="coerce").astype("Int64")
    if "gt_id" not in df_gt.columns:
        raise ValueError(f"GT csv must include gt_id column. Found: {list(df_gt.columns)}")
    df_gt["gt_id"] = df_gt["gt_id"].astype(str)

    df_gt["gt_cx_px"] = pd.to_numeric(df_gt[gtm["cx"]], errors="coerce")
    df_gt["gt_cy_px"] = pd.to_numeric(df_gt[gtm["cy"]], errors="coerce")
    df_gt["gt_diam_px"] = pd.to_numeric(df_gt[gtm["diam"]], errors="coerce")
    if gtm["spd"]:
        df_gt["gt_speed_px_s"] = pd.to_numeric(df_gt[gtm["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df_gt[gtm["vx"]], errors="coerce") if gtm["vx"] else np.nan
        vy = pd.to_numeric(df_gt[gtm["vy"]], errors="coerce") if gtm["vy"] else np.nan
        df_gt["gt_speed_px_s"] = np.hypot(vx, vy)

    # --- Standardize tracked dataframe ---
    df_m["frame"] = pd.to_numeric(df_m[trm["frame"]], errors="coerce").astype("Int64")
    df_m["id"]    = df_m[trm["id"]].astype(str)
    df_m["cx"]    = pd.to_numeric(df_m[trm["cx"]], errors="coerce")
    df_m["cy"]    = pd.to_numeric(df_m[trm["cy"]], errors="coerce")
    df_m["diam_px"] = pd.to_numeric(df_m[trm["diam"]], errors="coerce")
    if trm["spd"]:
        df_m["meas_speed_px_s"] = pd.to_numeric(df_m[trm["spd"]], errors="coerce")
    else:
        vx = pd.to_numeric(df_m[trm["vx"]], errors="coerce") if trm["vx"] else np.nan
        vy = pd.to_numeric(df_m[trm["vy"]], errors="coerce") if trm["vy"] else np.nan
        df_m["meas_speed_px_s"] = np.hypot(vx, vy)

    if fps and fps > 0 and df_m["frame"].notna().any():
        df_m["t_s"] = df_m["frame"].astype(float) / fps
    else:
        df_m["t_s"] = np.nan

    # --- Apply ROI to BOTH GT and tracked ---
    df_gt = apply_roi_filter(df_gt, "gt_cy_px", ignore_top_px=ignore_top, bottom_bar_px=bottom_bar, frame_h=frame_h)
    df_m  = apply_roi_filter(df_m,  "cy",      ignore_top_px=ignore_top, bottom_bar_px=bottom_bar, frame_h=frame_h)

    # --- Per-frame matching ---
    rows = []
    common_frames = sorted(set(df_gt["frame"].dropna().astype(int)) & set(df_m["frame"].dropna().astype(int)))
    for fr in common_frames:
        gtf = df_gt[df_gt["frame"].astype(int) == fr].dropna(subset=["gt_cx_px", "gt_cy_px", "gt_diam_px"])
        mf  = df_m[df_m["frame"].astype(int) == fr].dropna(subset=["cx", "cy", "diam_px"])
        if len(gtf) == 0 or len(mf) == 0:
            continue

        pairs = match_frame(gtf, mf, max_dist_px=max_dist_px)
        for gi, mi, dist in pairs:
            g = gtf.iloc[gi]
            mrow = mf.iloc[mi]
            rows.append({
                "frame": fr,
                "match_dist_px": dist,
                "gt_diam_px": g["gt_diam_px"],
                "meas_diam_px": mrow["diam_px"],
                "gt_speed_px_s": g["gt_speed_px_s"],
                "meas_speed_px_s": mrow["meas_speed_px_s"],
                "err_diam_px": mrow["diam_px"] - g["gt_diam_px"],
                "err_speed_px_s": mrow["meas_speed_px_s"] - g["gt_speed_px_s"],
            })

    df_match = pd.DataFrame(rows)
    df_match.to_csv(os.path.join(tabs, "matched_pairs.csv"), index=False)

    # --- Tracking metrics (MOTA/MOTP/IDF1) ---
    mot = compute_mot_metrics(
        df_gt[["frame","gt_id","gt_cx_px","gt_cy_px"]],
        df_m[["frame","id","cx","cy"]],
        max_dist_px=max_dist_px
    )
    pd.DataFrame([mot]).to_csv(os.path.join(tabs, "tracking_metrics.csv"), index=False)

    # centroid uncertainty (Eq. 3.17) on the tracked data
    centroid_u_xy_px, centroid_u_xy_in = run_centroid_uncertainty(
        df_m, tabs, out_dir, fps, scale_in_per_px
    )
        
    # --- Centroid contribution to velocity uncertainty (raw, no smoothing) ---
    df_v = df_m.copy()
    
    # Force the velocity column we want to use (measured speed) into a single clean Series
    if "meas_speed_px_s" in df_v.columns:
        df_v["vel_px_s"] = pd.to_numeric(df_v["meas_speed_px_s"], errors="coerce")
    elif "vel_px_s" in df_v.columns:
        df_v["vel_px_s"] = pd.to_numeric(df_v["vel_px_s"], errors="coerce")
    else:
        df_v["vel_px_s"] = np.nan
    
    # Drop duplicate columns by name (keeps the first occurrence)
    df_v = df_v.loc[:, ~df_v.columns.duplicated()]
    
    delta_r_med_px, delta_r_mean_px, n_steps, dr_steps = compute_delta_r_stats(
    df_m[["id", "frame", "cx", "cy"]].copy(),
    require_consecutive=True,
    max_step_px=100
    )

    plot_delta_r_histogram(
        dr_steps,
        os.path.join(figs, "fig_delta_r_histogram.png")
    )

    pd.DataFrame([{
        "delta_r_median_px": delta_r_med_px,
        "delta_r_mean_px": delta_r_mean_px,
        "N_steps": n_steps,
        "require_consecutive": True,
        "max_step_px": 100
    }]).to_csv(os.path.join(tabs, "delta_r_summary.csv"), index=False)
    
    # --- IoU validation (semantic masks) ---
    if not os.path.exists(gt_mask_mp4):
        log_error(out_dir, f"[IOU SKIP] Missing GT mask video: {gt_mask_mp4}")
    if not os.path.exists(pred_mask_mp4):
        log_error(out_dir, f"[IOU SKIP] Missing PRED mask video: {pred_mask_mp4}")
        if os.path.exists(filtered_mp4):
            log_error(out_dir, f"[NOTE] Found overlay video (not valid for IoU): {filtered_mp4}")

    mean_iou, median_iou, n_iou = compute_iou_video_semantic(
        gt_mask_mp4,
        pred_mask_mp4,
        out_csv=os.path.join(tabs, "iou_per_frame.csv"),
        thresh=128,
        crop_top=ignore_top,
        crop_bottom=bottom_bar
    )

    df_iou = None
    iou_csv = os.path.join(tabs, "iou_per_frame.csv")
    if os.path.exists(iou_csv):
        try:
            df_iou = pd.read_csv(iou_csv)
        except Exception as e:
            log_error(out_dir, f"[IOU CSV READ FAIL] {type(e).__name__}: {e}")

    safe_plot(out_dir, "fig_iou_vs_time",
              os.path.join(figs, "fig_iou_vs_time.png"),
              lambda: plot_iou_vs_time(df_iou, os.path.join(figs, "fig_iou_vs_time.png"), fps=fps))

    safe_plot(out_dir, "fig_iou_histogram",
              os.path.join(figs, "fig_iou_histogram.png"),
              lambda: plot_iou_histogram(df_iou, os.path.join(figs, "fig_iou_histogram.png"), bins=20))

    # --- Convert match units if scale exists ---
    if scale_in_per_px and (not df_match.empty):
        df_match["gt_diam_in"] = df_match["gt_diam_px"] * scale_in_per_px
        df_match["meas_diam_in"] = df_match["meas_diam_px"] * scale_in_per_px
        df_match["err_diam_in"] = df_match["err_diam_px"] * scale_in_per_px

        df_match["gt_speed_in_s"] = df_match["gt_speed_px_s"] * scale_in_per_px
        df_match["meas_speed_in_s"] = df_match["meas_speed_px_s"] * scale_in_per_px
        df_match["err_speed_in_s"] = df_match["err_speed_px_s"] * scale_in_per_px

    if scale_in_per_px and (not df_match.empty):
        true_d = df_match["gt_diam_in"].values
        meas_d = df_match["meas_diam_in"].values
        err_d  = df_match["err_diam_in"].values
        true_v = df_match["gt_speed_in_s"].values
        meas_v = df_match["meas_speed_in_s"].values
        err_v  = df_match["err_speed_in_s"].values
    else:
        true_d = df_match["gt_diam_px"].values if not df_match.empty else np.array([])
        meas_d = df_match["meas_diam_px"].values if not df_match.empty else np.array([])
        err_d  = df_match["err_diam_px"].values if not df_match.empty else np.array([])
        true_v = df_match["gt_speed_px_s"].values if not df_match.empty else np.array([])
        meas_v = df_match["meas_speed_px_s"].values if not df_match.empty else np.array([])
        err_v  = df_match["err_speed_px_s"].values if not df_match.empty else np.array([])

    safe_plot(out_dir, "fig_measured_vs_true_diameter",
              os.path.join(figs, f"fig_measured_vs_true_diameter_{units_len_fname}.png"),
              lambda: plot_measured_vs_true(
                  true_d, meas_d, os.path.join(figs, f"fig_measured_vs_true_diameter_{units_len_fname}.png"),
                  "Measured vs True Diameter (Synthetic Validation)", units_len))

    safe_plot(out_dir, "fig_measured_vs_true_speed",
              os.path.join(figs, f"fig_measured_vs_true_speed_{units_vel_fname}.png"),
              lambda: plot_measured_vs_true(
                  true_v, meas_v, os.path.join(figs, f"fig_measured_vs_true_speed_{units_vel_fname}.png"),
                  "Measured vs True Speed (Synthetic Validation)", units_vel))

    safe_plot(out_dir, "fig_error_hist_diameter",
              os.path.join(figs, f"fig_error_hist_diameter_{units_len_fname}.png"),
              lambda: plot_distribution_zoom_full(
                  err_d, os.path.join(figs, f"fig_error_hist_diameter_{units_len_fname}.png"),
                  "Diameter Error Distribution (Synthetic Validation)", "Error", units_len))

    safe_plot(out_dir, "fig_error_hist_speed",
              os.path.join(figs, f"fig_error_hist_speed_{units_vel_fname}.png"),
              lambda: plot_distribution_zoom_full(
                  err_v, os.path.join(figs, f"fig_error_hist_speed_{units_vel_fname}.png"),
                  "Speed Error Distribution (Synthetic Validation)", "Error", units_vel))

    safe_plot(out_dir, "fig_relative_error_diameter",
              os.path.join(figs, f"fig_relative_error_diameter.png"),
              lambda: plot_relative_error(
                  err_d, true_d,
                  os.path.join(figs, f"fig_relative_error_diameter.png")))

    safe_plot(out_dir, "fig_grace_avg_point",
          os.path.join(figs, "fig_grace_avg_point.png"),
          lambda: plot_grace_avg_point(df_m, os.path.join(figs, "fig_grace_avg_point.png"),
                                       df_avg=df_avg, label="Auto (GT runs)", annotate=True))


    # Time-colored error vs time/frame
    if not df_match.empty:
        if fps and fps > 0:
            x = df_match["frame"].values / fps
            xlab = "Time (s)"
        else:
            x = df_match["frame"].values
            xlab = "Frame"
    else:
        x = np.array([])
        xlab = "Frame"

    safe_plot(out_dir, "fig_error_vs_time_diameter",
              os.path.join(figs, f"fig_error_vs_time_diameter_{units_len_fname}.png"),
              lambda: plot_error_vs_time_timecolored(
                  x, err_d, os.path.join(figs, f"fig_error_vs_time_diameter_{units_len_fname}.png"),
                  "Diameter Error vs Time/Frame (Synthetic)", xlab, units_len))

    safe_plot(out_dir, "fig_error_vs_time_speed",
              os.path.join(figs, f"fig_error_vs_time_speed_{units_vel_fname}.png"),
              lambda: plot_error_vs_time_timecolored(
                  x, err_v, os.path.join(figs, f"fig_error_vs_time_speed_{units_vel_fname}.png"),
                  "Speed Error vs Time/Frame (Synthetic)", xlab, units_vel))

    safe_plot(out_dir, "fig_trajectories_2d_timecolored",
              os.path.join(figs, "fig_trajectories_2d_timecolored.png"),
              lambda: plot_trajectories_2d_timecolored(
                  df_m, os.path.join(figs, "fig_trajectories_2d_timecolored.png"), units_len))

    # --- Video-based qualitative panel (raw + pred mask + filtered + IDs; optionally GT mask) ---
    raw_mp4 = _find_raw_video(base, stem)
     # ---- choose start frame near the middle of tracked data ----
    if df_m["frame"].notna().any():
        fmin = int(df_m["frame"].dropna().min())
        fmax = int(df_m["frame"].dropna().max())
        start_frame = max(fmin, (fmin + fmax)//2 - 2)  # centers a 5-frame window
    else:
        start_frame = 0
    
    safe_plot(out_dir, "fig_qualitative_sequence",
              os.path.join(figs, "fig_qualitative_sequence.png"),
              lambda: make_qualitative_panel_from_videos(
                  raw_mp4=raw_mp4,
                  pred_mask_mp4=pred_mask_mp4,
                  filtered_mp4=filtered_mp4,
                  gt_mask_mp4=gt_mask_mp4,
                  show_gt_row=True,  # set False if you don’t want the GT row
                  df_tracked=df_m,
                  out_png=os.path.join(figs, "fig_qualitative_sequence.png"),
                  start_frame=start_frame,#int(df_m["frame"].dropna().min()) if df_m["frame"].notna().any() else 0,
                  n_frames=5
              ))

    # --- GT vs Pred mask difference panel (white=GT, red=over, blue=under) ---
    safe_plot(out_dir, "fig_mask_diff_sequence",
              os.path.join(figs, "fig_mask_diff_sequence.png"),
              lambda: make_mask_diff_panel_from_videos(
                  gt_mask_mp4=gt_mask_mp4,
                  pred_mask_mp4=pred_mask_mp4,
                  out_png=os.path.join(figs, "fig_mask_diff_sequence.png"),
                  start_frame=start_frame,
                  n_frames=5,
                  thresh=128,
                  crop_top=ignore_top,
                  crop_bottom=bottom_bar
              ))


    def stdev(x):
        x = _clean(x)
        return float(np.std(x, ddof=1)) if x.size > 1 else np.nan

    summary = {
        "mode": "GT",
        "N_matches": int(np.isfinite(err_d).sum()) if err_d.size else 0,
        "scale_in_per_px": scale_in_per_px if scale_in_per_px else "",
        "fps": fps if fps else "",
        f"diam_MAE_{units_len}": mae(err_d),
        f"diam_RMSE_{units_len}": rmse(err_d),
        f"diam_Bias_{units_len}": bias(err_d),
        f"diam_Std_{units_len}": stdev(err_d),
        f"speed_MAE_{units_vel}": mae(err_v),
        f"speed_RMSE_{units_vel}": rmse(err_v),
        f"speed_Bias_{units_vel}": bias(err_v),
        f"speed_Std_{units_vel}": stdev(err_v),
        "IoU_mean": mean_iou,
        "IoU_median": median_iou,
        "IoU_N_frames": n_iou,
        "MOTA": mot["MOTA"],
        "MOTP_px": mot["MOTP_px"],
        "IDF1": mot["IDF1"],
        "FP": mot["FP"],
        "FN": mot["FN"],
        "IDSW": mot["IDSW"],
        "centroid_u_xy_px": centroid_u_xy_px,
        "centroid_u_xy_in": centroid_u_xy_in if scale_in_per_px else "",
        "delta_r_median_px": delta_r_med_px,
    }
    pd.DataFrame([summary]).to_csv(os.path.join(tabs, "run_summary.csv"), index=False)

    # manual cross-check (optional)
    df_for_manual = df_m.copy()
    if scale_in_per_px:
        df_for_manual["diam_cmp"] = df_for_manual["diam_px"] * scale_in_per_px
        df_for_manual["vel_cmp"]  = df_for_manual["meas_speed_px_s"] * scale_in_per_px
    else:
        df_for_manual["diam_cmp"] = df_for_manual["diam_px"]
        df_for_manual["vel_cmp"]  = df_for_manual["meas_speed_px_s"]
    _try_manual_crosscheck(manual_csv, df_for_manual.rename(columns={"meas_speed_px_s":"vel_px_s"}), out_dir, units_len, units_vel)

    return summary


# ============================================================
# Batch overlay plots 
# ============================================================
def overlay_from_batch(tracked_files, out_dir_overlay, max_runs=10):
    ensure_dir(out_dir_overlay)

    diam_series = []
    vel_series = []
    units_len = "px"
    units_vel = "px/s"

    for f in tracked_files[:max_runs]:
        stem = os.path.basename(f).replace("_tracked.csv", "")
        base = os.path.dirname(f)
        avg = os.path.join(base, stem + "_averages.csv")

        df = pd.read_csv(f)
        m = map_tracked(df)

        df["frame"] = pd.to_numeric(df[m["frame"]], errors="coerce").astype("Int64")
        df["id"] = df[m["id"]].astype(str)
        df["diam_px"] = pd.to_numeric(df[m["diam"]], errors="coerce")

        if m["spd"]:
            df["vel_px_s"] = pd.to_numeric(df[m["spd"]], errors="coerce")
        else:
            vx = pd.to_numeric(df[m["vx"]], errors="coerce") if m["vx"] else np.nan
            vy = pd.to_numeric(df[m["vy"]], errors="coerce") if m["vy"] else np.nan
            df["vel_px_s"] = np.hypot(vx, vy)

        settings, df_avg = (None, None)
        if os.path.exists(avg):
            try:
                settings, df_avg = read_averages_with_settings(avg, header_line_1_indexed=25)
            except Exception:
                settings, df_avg = None, None

        scale_in_per_px = infer_scale_in_per_px(settings=settings, df_avg=df_avg)
        if scale_in_per_px:
            d = df["diam_px"].to_numpy(float) * scale_in_per_px
            v = df["vel_px_s"].to_numpy(float) * scale_in_per_px
            units_len = "in"
            units_vel = "in/s"
        else:
            d = df["diam_px"].to_numpy(float)
            v = df["vel_px_s"].to_numpy(float)

        diam_series.append((stem, d))
        vel_series.append((stem, v))

    plot_distribution_overlay(
        diam_series,
        os.path.join(out_dir_overlay, "fig_batch_overlay_diameter.png"),
        "Diameter Distribution Overlay (Batch)",
        "Equivalent diameter",
        units_len,
        bins=20
    )

    plot_distribution_overlay(
        vel_series,
        os.path.join(out_dir_overlay, "fig_batch_overlay_velocity.png"),
        "Velocity Distribution Overlay (Batch)",
        "Rise velocity",
        units_vel,
        bins=20
    )

def plot_batch_accuracy_from_master(master_csv, out_dir):
    ensure_dir(out_dir)
    set_thesis_style()

    df = pd.read_csv(master_csv)
    df = df[df["mode"].astype(str).str.upper().eq("GT")].copy()

    if df.empty:
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.grid(True)
        ax.set_title("Batch Accuracy (GT) - No GT rows found")
        ax.text(0.5, 0.5, "No GT cases in master_summary.csv", ha="center", va="center", transform=ax.transAxes)
        savefig(os.path.join(out_dir, "fig_batch_accuracy_NONE.png"))
        return

    diam_mae_cols = [c for c in df.columns if c.startswith("diam_MAE_")]
    spd_mae_cols  = [c for c in df.columns if c.startswith("speed_MAE_")]
    if not diam_mae_cols or not spd_mae_cols:
        raise ValueError("master_summary.csv missing diam_MAE_* or speed_MAE_* columns.")

    units_len = diam_mae_cols[0].split("diam_MAE_", 1)[1]
    units_vel = spd_mae_cols[0].split("speed_MAE_", 1)[1]

    d_mae  = f"diam_MAE_{units_len}"
    d_rmse = f"diam_RMSE_{units_len}"
    d_bias = f"diam_Bias_{units_len}"
    d_std  = f"diam_Std_{units_len}"

    v_mae  = f"speed_MAE_{units_vel}"
    v_rmse = f"speed_RMSE_{units_vel}"
    v_bias = f"speed_Bias_{units_vel}"
    v_std  = f"speed_Std_{units_vel}"

    def sort_key(s):
        s = str(s).lower()
        for i, k in enumerate(["circle", "ellipse", "wobble", "multi"]):
            if k in s:
                return i
        return 99

    if "case" in df.columns:
        df = df.sort_values(by="case", key=lambda s: s.map(sort_key))
        cases = df["case"].astype(str).tolist()
    else:
        cases = [str(i) for i in range(len(df))]

    x = np.arange(len(cases))

    def bar_with_err(ycol, ecol, title, ylabel, fname):
        y = pd.to_numeric(df[ycol], errors="coerce").to_numpy(float)
        e = pd.to_numeric(df[ecol], errors="coerce").to_numpy(float) if ecol in df.columns else None
        n = pd.to_numeric(df.get("N_matches", np.nan), errors="coerce").to_numpy(float)

        fig, ax = plt.subplots(figsize=(10.8, 5.2))
        ax.grid(True)
        ax.bar(x, y)
        ax.set_title(clean_plot_title(title))
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(cases, rotation=20, ha="right")

        for xi, yi, ni in zip(x, y, n):
            if np.isfinite(yi):
                ax.text(xi, yi, f"N={int(ni) if np.isfinite(ni) else ''}", ha="center", va="bottom", fontsize=10)

        savefig(os.path.join(out_dir, fname))

    def bias_plot(ycol, title, ylabel, fname):
        y = pd.to_numeric(df[ycol], errors="coerce").to_numpy(float)

        fig, ax = plt.subplots(figsize=(10.8, 5.2))
        ax.grid(True)
        ax.axhline(0, linestyle="--", linewidth=2)
        ax.bar(x, y)
        ax.set_title(clean_plot_title(title))
        ax.set_ylabel(ylabel)
        ax.set_xticks(x)
        ax.set_xticklabels(cases, rotation=20, ha="right")
        savefig(os.path.join(out_dir, fname))

    bar_with_err(d_mae,  d_std,  "Diameter MAE by Video (GT)",  f"MAE ({units_len})",  "fig_batch_diam_MAE.png")
    bar_with_err(d_rmse, d_std,  "Diameter RMSE by Video (GT)", f"RMSE ({units_len})", "fig_batch_diam_RMSE.png")
    bias_plot(d_bias,    "Diameter Bias by Video (GT)",         f"Bias ({units_len})", "fig_batch_diam_Bias.png")

    bar_with_err(v_mae,  v_std,  "Speed MAE by Video (GT)",     f"MAE ({units_vel})",  "fig_batch_speed_MAE.png")
    bar_with_err(v_rmse, v_std,  "Speed RMSE by Video (GT)",    f"RMSE ({units_vel})", "fig_batch_speed_RMSE.png")
    bias_plot(v_bias,    "Speed Bias by Video (GT)",            f"Bias ({units_vel})", "fig_batch_speed_Bias.png")


# ============================================================
# GUI (UNCHANGED)
# ============================================================
class BubbleEvalGUI(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Bubble Evaluation Tool (Thesis-ready v3)")
        self.geometry("900x460")
        self.resizable(False, False)

        self.tracked = tk.StringVar()
        self.folder  = tk.StringVar()
        self.outroot = tk.StringVar()
        self.frames_dir = tk.StringVar()
        self.max_dist = tk.StringVar(value="65")

        self._build()

    def _build(self):
        pad = {"padx": 8, "pady": 6}

        ttk.Label(self, text="Single Run: select *_tracked.csv").grid(row=0, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.tracked, width=78).grid(row=0, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_tracked).grid(row=0, column=2, **pad)

        ttk.Label(self, text="Batch Folder: contains *_tracked.csv").grid(row=1, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.folder, width=78).grid(row=1, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_folder).grid(row=1, column=2, **pad)

        ttk.Label(self, text="Output Root Folder").grid(row=2, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.outroot, width=78).grid(row=2, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_outroot).grid(row=2, column=2, **pad)

        ttk.Label(self, text="(Optional) Frames folder for qualitative panel").grid(row=3, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.frames_dir, width=78).grid(row=3, column=1, **pad)
        ttk.Button(self, text="Browse", command=self.pick_frames).grid(row=3, column=2, **pad)

        ttk.Label(self, text="Max match distance (px) [GT only]").grid(row=4, column=0, sticky="e", **pad)
        ttk.Entry(self, textvariable=self.max_dist, width=12).grid(row=4, column=1, sticky="w", **pad)

        ttk.Button(self, text="Run Single", command=self.run_single).grid(row=5, column=0, columnspan=3, pady=(12, 6))
        ttk.Button(self, text="Run Batch Folder (+ overlay plots)", command=self.run_batch).grid(row=6, column=0, columnspan=3)

        self.status = ttk.Label(self, text="Ready.", foreground="gray")
        self.status.grid(row=7, column=0, columnspan=3, pady=(10, 0))

        note = "Tip: set Output Root to something like C:\\Users\\Lab_Staff\\Desktop\\BubbleEvalOutputs"
        ttk.Label(self, text=note, foreground="gray").grid(row=8, column=0, columnspan=3, pady=(6, 0))

    def pick_tracked(self):
        self.tracked.set(filedialog.askopenfilename(filetypes=[("CSV", "*.csv")]))

    def pick_folder(self):
        self.folder.set(filedialog.askdirectory())

    def pick_outroot(self):
        self.outroot.set(filedialog.askdirectory())

    def pick_frames(self):
        self.frames_dir.set(filedialog.askdirectory())

    def _run_one(self, tracked_path, outroot):
        if not tracked_path or not os.path.isfile(tracked_path):
            raise ValueError("Invalid tracked.csv path.")
        if not outroot or not os.path.isdir(outroot):
            raise ValueError("Invalid output root folder.")

        try:
            maxd = float(self.max_dist.get())
        except Exception:
            maxd = 65.0

        stem = os.path.basename(tracked_path).replace("_tracked.csv", "")
        base = os.path.dirname(tracked_path)

        gt     = os.path.join(base, stem + "_gt.csv")
        avg    = os.path.join(base, stem + "_averages.csv")
        manual = os.path.join(base, stem + "_manual.csv")  # optional

        out_dir = ensure_dir(os.path.join(outroot, stem))

        if os.path.exists(gt):
            return out_dir, evaluate_with_gt(
                gt, tracked_path, out_dir, maxd, avg_csv=avg,
                frames_dir=self.frames_dir.get().strip() or None,
                manual_csv=manual if os.path.exists(manual) else None
            )
        else:
            return out_dir, evaluate_no_gt(
                tracked_path, out_dir, avg_csv=avg,
                frames_dir=self.frames_dir.get().strip() or None,
                manual_csv=manual if os.path.exists(manual) else None
            )

    def run_single(self):
        if not self.tracked.get() or not self.outroot.get():
            messagebox.showerror("Missing input", "Select *_tracked.csv and an Output Root folder.")
            return
        self.status.config(text="Running single...")
        self.update_idletasks()

        try:
            out_dir, summary = self._run_one(self.tracked.get(), self.outroot.get())
        except Exception as e:
            messagebox.showerror("Run failed", f"{type(e).__name__}: {e}")
            self.status.config(text="Failed.")
            return

        self.status.config(text=f"Done. Outputs: {out_dir}")
        messagebox.showinfo("Done", f"Completed.\nMode: {summary.get('mode')}\nOutputs in:\n{out_dir}")

    def run_batch(self):
        if not self.folder.get() or not self.outroot.get():
            messagebox.showerror("Missing input", "Select a Batch Folder and an Output Root folder.")
            return

        self.status.config(text="Batch running...")
        self.update_idletasks()

        tracked_files = sorted(glob.glob(os.path.join(self.folder.get(), "*_tracked.csv")))
        if not tracked_files:
            messagebox.showerror("No runs found", "No *_tracked.csv files found in batch folder.")
            self.status.config(text="Ready.")
            return

        rows = []
        for f in tracked_files:
            stem = os.path.basename(f).replace("_tracked.csv", "")
            try:
                out_dir, summary = self._run_one(f, self.outroot.get())
                summary["case"] = stem
                rows.append(summary)
            except Exception as e:
                rows.append({"case": stem, "mode": "ERROR", "error": f"{type(e).__name__}: {e}"})

        # Create overlay output folder FIRST (so everything can use it)
        overlay_dir = ensure_dir(os.path.join(self.outroot.get(), "batch_overlays"))

        # --- Grace diagram: all runs on one map (avg Eo/Re per run) ---
        try:
            run_points = []
            for f in tracked_files:
                stem = os.path.basename(f).replace("_tracked.csv", "")
                base = os.path.dirname(f)
                avg = os.path.join(base, stem + "_averages.csv")

                df_tr = pd.read_csv(f)
                df_av = None
                if os.path.exists(avg):
                    try:
                        _, df_av = read_averages_with_settings(avg, header_line_1_indexed=25)
                    except Exception:
                        df_av = None

                Eo_avg, Re_avg = _extract_avg_eo_re(df_avg=df_av, df_tracked=df_tr)
                run_points.append({"label": stem, "Eo": Eo_avg, "Re": Re_avg})

            grace_all_png = os.path.join(overlay_dir, "fig_grace_all_runs_avg.png")
            plot_grace_avg_points_all_runs(run_points, grace_all_png)

        except Exception as e:
            log_error(overlay_dir, f"[GRACE ALL RUNS FAIL] {type(e).__name__}: {e}")

        master_df = pd.DataFrame(rows)

        master_csv = os.path.join(self.outroot.get(), "master_summary.csv")
        try:
            master_df.to_csv(master_csv, index=False)
        except PermissionError:
            # Most common cause: file open in Excel or folder not writable.
            # Fall back to overlay_dir with a unique name.
            alt_csv = os.path.join(overlay_dir, f"master_summary_{pd.Timestamp.now():%Y%m%d_%H%M%S}.csv")
            master_df.to_csv(alt_csv, index=False)
            log_error(overlay_dir, f"[MASTER CSV PERMISSION] Could not write {master_csv}; wrote {alt_csv} instead.")


        acc_dir = ensure_dir(os.path.join(overlay_dir, "accuracy"))
        try:
            plot_batch_accuracy_from_master(master_csv, acc_dir)
        except Exception as e:
            log_error(overlay_dir, f"[BATCH ACCURACY PLOTS FAIL] {type(e).__name__}: {e}")

        try:
            overlay_from_batch(tracked_files, overlay_dir, max_runs=12)
        except Exception as e:
            log_error(overlay_dir, f"[BATCH OVERLAY FAIL] {type(e).__name__}: {e}")

        self.status.config(text="Batch complete.")
        messagebox.showinfo("Done", f"Batch complete.\nSaved master_summary.csv\nOverlay plots in:\n{overlay_dir}")

        

if __name__ == "__main__":
    BubbleEvalGUI().mainloop()
